In [ ]:

import os
import datetime
import re
import math
from collections import OrderedDict
import multiprocessing
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import tensorflow.keras.backend as K
import tensorflow.keras.layers as KL
import tensorflow.keras.utils as KU
from tensorflow.python.eager import context
import tensorflow.keras.models as KM

from mrcnn import utils
import sys
from mrcnn.parallel_model import ParallelModel

from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
config = ConfigProto() 
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)

# Requires TensorFlow 2.0+
from distutils.version import LooseVersion
assert LooseVersion(tf.__version__) >= LooseVersion("2.0")

tf.compat.v1.disable_eager_execution()

NotFoundError: dlopen(/Users/vanshikaarora/Desktop/Mask-rcnn-from-scratch-personal/venv/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib, 0x0006): Symbol not found: __ZN10tensorflow16TensorShapeProtoC1ERKS0_
  Referenced from: <10B7FC95-0B10-3E4E-84D0-79A2D52E4D78> /Users/vanshikaarora/Desktop/Mask-rcnn-from-scratch-personal/venv/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib
  Expected in:     <A5DBF409-59EB-37C4-ABE0-CF000BC95F69> /Users/vanshikaarora/Desktop/Mask-rcnn-from-scratch-personal/venv/lib/python3.11/site-packages/tensorflow/python/_pywrap_tensorflow_internal.so

In [ ]:
############################################################
#  Utility Functions
############################################################


def log(text, array=None):
    """Prints a text message. And, optionally, if a Numpy array is provided it
    prints it's shape, min, and max values.
     """
    """It is basically
    DEBUGGING UTILITY - Saves 45+ lines of manual print code.
    
    Instead of writing 5-6 print statements per tensor (shape, min, max, dtype),
    this does it all in 1 line. Critical for monitoring 10+ tensors:
    - Input images, 5 FPN feature maps, thousands of anchors, proposal boxes,
    - Class predictions, bbox refinements, masks, and more.
    
    Example: log("Feature map", tensor)  # 1 line vs 6 lines manually
    """
    if array is not None:
        text = text.ljust(25)
        text += ("shape: {:20}  ".format(str(array.shape)))
        if array.size:
            text += ("min: {:10.5f}  max: {:10.5f}".format(array.min(),array.max()))
        else:
            text += ("min: {:10}  max: {:10}".format("",""))
        text += "  {}".format(array.dtype)
    print(text)

class BatchNorm(KL.BatchNormalization):
    """Extends the Keras BatchNormalization class to allow a central place
    to make changes if needed.

    Batch normalization has a negative effect on training if batches are small
    so this layer is often frozen (via setting in Config class) and functions
    as linear layer.
    """
    """
    CALCULATES FEATURE MAP SIZES at each backbone stage (C2 through C6).
    
    WHY NEEDED: Different FPN levels (P2-P6) have different resolutions.
    Each stride (4,8,16,32,64) tells us how much the image shrinks.
    
    EXAMPLE: 1024x1024 image with stride 4 -> 256x256 feature map.
    These sizes determine where to place anchors and which ROIs go to which pyramid level.
    
    RETURNS: Array of [height, width] for each stage: [[256,256], [128,128], [64,64], [32,32], [16,16]]
    """
    def call(self, inputs, training=None):
        """
        Note about training values:
            None: Train BN layers. This is the normal mode
            False: Freeze BN layers. Good when batch size is small
            True: (don't use). Set layer in training mode even when making inferences
        """
        return super(self.__class__, self).call(inputs, training=training)
    

def compute_backbone_shapes(config, image_shape):
    """Computes the width and height of each stage of the backbone network.

    Returns:
        [N, (height, width)]. Where N is the number of stages
    """
    """
    PURPOSE: Calculate output dimensions of each ResNet stage (C2-C6).
    
    HOW IT WORKS: For each stride (4,8,16,32,64), compute: ceil(image_size / stride)
    - Stride 4  -> Stage C2 (256x256 for 1024 input)
    - Stride 8  -> Stage C3 (128x128)
    - Stride 16 -> Stage C4 (64x64)
    - Stride 32 -> Stage C5 (32x32)
    - Stride 64 -> Stage C6 (16x16) - RPN only
    
    WHY NEEDED: 
    - Determines anchor grid positions
    - Assigns ROIs to correct FPN levels
    - Validates image size compatibility
    
    RETURNS: np.array([[H1,W1], [H2,W2], [H3,W3], [H4,W4], [H5,W5]])
    """
    if callable(config.BACKBONE):
        return config.COMPUTE_BACKBONE_SHAPE(image_shape)

    # Currently supports ResNet only
    assert config.BACKBONE in ["resnet50", "resnet101"]
    return np.array(
        [[int(math.ceil(image_shape[0] / stride)),
            int(math.ceil(image_shape[1] / stride))]
            for stride in config.BACKBONE_STRIDES])

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    HOW CONV + IDENTITY BLOCKS WORK TOGETHER                        │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  INPUT IMAGE (224x224x3)                                                           │
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 1: Conv7x7 + MaxPool → 56x56x64                                         ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 2: (NO SHAPE CHANGE - stride=1)                                         ││
│  │                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  CONV BLOCK (block='a')                                                  │  ││
│  │  │  Input: 56x56x64                                                         │  ││
│  │  │  Output: 56x56x256  ← Changes channels only (64→256)                    │  ││
│  │  │  ★ This is the "DIMENSION CHANGER"                                      │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='b')                                              │  ││
│  │  │  Input: 56x56x256                                                        │  ││
│  │  │  Output: 56x56x256  ← NO SHAPE CHANGE!                                  │  ││
│  │  │  ★ This is the "FEATURE ENHANCER"                                       │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='c')                                              │  ││
│  │  │  Input: 56x56x256                                                        │  ││
│  │  │  Output: 56x56x256  ← NO SHAPE CHANGE!                                  │  ││
│  │  │  ★ More feature refinement                                              │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │                                                                                 ││
│  │  RESULT: C2 = 56x56x256 (same shape after 3 blocks)                           ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 3: (SHAPE CHANGE - stride=2)                                            ││
│  │                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  CONV BLOCK (block='a')                                                  │  ││
│  │  │  Input: 56x56x256                                                        │  ││
│  │  │  Output: 28x28x512  ← DOWNSAMPLES + INCREASES CHANNELS!                 │  ││
│  │  │  ★ Conv Block changes shape again!                                      │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='b')                                              │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 28x28x512  ← NO CHANGE!                                        │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='c')                                              │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 28x28x512  ← NO CHANGE!                                        │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK (block='d')                                              │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 28x28x512  ← NO CHANGE!                                        │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │                                                                                 ││
│  │  RESULT: C3 = 28x28x512 (changed shape once, refined 3 times)                 ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 4: (SHAPE CHANGE - stride=2)                                            ││
│  │                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  CONV BLOCK (block='a')                                                  │  ││
│  │  │  Input: 28x28x512                                                        │  ││
│  │  │  Output: 14x14x1024 ← DOWNSAMPLES + INCREASES CHANNELS!                 │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │      │                                                                         ││
│  │      ▼                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────┐  ││
│  │  │  IDENTITY BLOCK × (5 for ResNet50, 22 for ResNet101)                    │  ││
│  │  │  Each: 14x14x1024 → 14x14x1024 (NO CHANGE)                             │  ││
│  │  │  ★ MANY identity blocks = DEEPER network!                               │  ││
│  │  └─────────────────────────────────────────────────────────────────────────┘  ││
│  │                                                                                 ││
│  │  RESULT: C4 = 14x14x1024                                                       ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STAGE 5: (SHAPE CHANGE - stride=2)                                            ││
│  │                                                                                 ││
│  │  CONV BLOCK (block='a') + 2x IDENTITY BLOCK                                    ││
│  │  Input: 14x14x1024 → Output: 7x7x2048                                          ││
│  │                                                                                 ││
│  │  RESULT: C5 = 7x7x2048                                                          ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│                                                                                     │
│  FINAL OUTPUT: [C1, C2, C3, C4, C5]                                               │
│                                                                                     │
│  ★ EACH STAGE: 1 Conv Block (changes shape) + N Identity Blocks (refines)         │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
Why This Pattern Works
1. Conv Block = "Gateway to New Resolution"
Purpose: Change the scale

What it does: Downsamples image, increases channels

Why: To detect larger objects (more semantic features)

When: Only at the start of each stage

2. Identity Blocks = "Feature Learners at That Resolution"
Purpose: Learn complex features at this scale

What it does: Adds more layers, more capacity

Why: To learn increasingly complex patterns

When: Multiple times after the Conv Block

3. Together = "Hierarchical Feature Learning"
text
Stage 2: Learn fine details at 56x56
Stage 3: Learn medium features at 28x28
Stage 4: Learn large features at 14x14
Stage 5: Learn semantic concepts at 7x7


The number of identity blocks in each stage is fixed and known from the ResNet paper:

Stage	ResNet50	ResNet101	ResNet152
Stage 2	2 identity blocks	2 identity blocks	3 identity blocks
Stage 3	3 identity blocks	3 identity blocks	8 identity blocks
Stage 4	5 identity blocks	22 identity blocks	36 identity blocks
Stage 5	2 identity blocks	2 identity blocks	3 identity blocks
Total Identity Blocks:

ResNet50: 2 + 3 + 5 + 2 = 12 identity blocks

ResNet101: 2 + 3 + 22 + 2 = 29 identity blocks

ResNet152: 3 + 8 + 36 + 3 = 50 identity blocks



You don't "use ResNet separately" - you use the complete trained Mask R-CNN model!

text
Trained Mask R-CNN Model:
┌─────────────────────────────────────────────────────────────┐
│  ┌─────────────────────────────────────────────────────┐   │
│  │  RESNET LAYERS (Trained as part of Mask R-CNN)     │   │
│  └─────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  FPN LAYERS (Trained as part of Mask R-CNN)        │   │
│  └─────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  RPN LAYERS (Trained as part of Mask R-CNN)        │   │
│  └─────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  HEADS (Trained as part of Mask R-CNN)             │   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
It's ONE model, trained ONCE, with ALL layers updated together! 🎯



In [ ]:
############################################################
#  Resnet Graph
############################################################
def identity_block(input_tensor, kernel_size, filters, stage, block,
                   use_bias=True, train_bn=True):
    """The identity_block is the block that has no conv layer at shortcut
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of middle conv layer at main path
        filters: list of integers, the nb_filters of 3 conv layer at main path
        stage: integer, current stage label, used for generating layer names
        block: 'a','b'..., current block label, used for generating layer names
        use_bias: Boolean. To use or not use a bias in conv layers.
        train_bn: Boolean. Train or freeze Batch Norm layers
    """
    nb_filter1, nb_filter2, nb_filter3 = filters
    conv_name_base='res'+str(stage)+block+'_branch'
    bn_name_base='bn'+str(stage)+block+'_branch'
    x=KL.Conv2D(nb_filter1,(1,1),name=conv_name_base+'2a',use_bias=use_bias)(input_tensor)
    x=BatchNorm(name=bn_name_base+'2a')(x,training=train_bn)
    x=KL.Activation('relu')(x)
    x = KL.Conv2D(nb_filter2, (kernel_size, kernel_size), padding='same',
                  name=conv_name_base + '2b', use_bias=use_bias)(x)
    x=BatchNorm(name=conv_name_base+'2b')(x,training=train_bn)
    x=KL.Activation('relu')(x)
    x = KL.Conv2D(nb_filter3, (1, 1), name=conv_name_base + '2c',
                  use_bias=use_bias)(x)
    x = BatchNorm(name=bn_name_base + '2c')(x, training=train_bn)
    x=KL.add()([x,input_tensor])#This single line does the skip connection!
## Element-wise addition:
# output[i] = x[i] + input_tensor[i]
# output = F(input) + input
      # = processed_features + original_input

    x = KL.Activation('relu', name='res' + str(stage) + block + '_out')(x)#Apply ReLU to the sum: output = ReLU(F(input) + input) its final activation
    #Why This is Called "Identity" Block
#The "Identity" refers to the shortcut path:
    return x
    """stage: 2, 3, 4, 5        # Which ResNet stage (C2, C3, C4, C5)
block: 'a', 'b', 'c'...   # Which block within that stage (first, second, third...) Why They Matter - Layer Naming!
Every layer needs a UNIQUE name so TensorFlow/Keras can track them.

python
# Example: Stage 2, Block 'b'
stage = 2
block = 'b'

conv_name_base = 'res' + str(stage) + block + '_branch'
# = 'res2b_branch'

bn_name_base = 'bn' + str(stage) + block + '_branch'
# = 'bn2b_branch'

# Now each layer gets a unique name:
# conv_name_base + '2a' = 'res2b_branch2a'
# conv_name_base + '2b' = 'res2b_branch2b'
# conv_name_base + '2c' = 'res2b_branch2c'
# bn_name_base + '2a' = 'bn2b_branch2a'
# etc. """
    
"""                  WHY x DOESN'T LOSE VALUE                                        │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  Think of it like making a sandwich:                                              │
│                                                                                     │
│  input_tensor = "BREAD"  (Original ingredient, NEVER CHANGES)                     │
│                                                                                     │
│  Step 1: x = "BREAD" + "BUTTER"  (x becomes buttered bread)                       │
│  Step 2: x = "BUTTERED BREAD" + "CHEESE"  (x becomes cheesy bread)               │
│  Step 3: x = "CHEESY BREAD" + "HAM"  (x becomes ham sandwich)                    │
│  Step 4: x = "HAM SANDWICH" + "TOMATO"  (x becomes complete sandwich)            │
│                                                                                     │
│  ★ input_tensor is still "BREAD" (unchanged!)                                     │
│  ★ x has been overwritten 4 times, but that's okay!                               │
│                                                                                     │
│  At the end:                                                                       │
│  x = complete sandwich                                                             │
│  input_tensor = bread (still there!)"""



"""Block Type	Main Path (Layers)	Shortcut Path
Identity Block	✅ Has layers (1x1 → 3x3 → 1x1)	❌ No layers (identity, direct pass)
Conv Block	✅ Has layers (1x1 → 3x3 → 1x1)	✅ Has layers (1x1 convolution)
What Each Block Actually Contributes
Conv Block Contribution: "Dimension Transformer"
python
Input:  [batch, 56, 56, 256]   # High resolution, 256 channels
        ↓
Conv Block
        ↓
Output: [batch, 28, 28, 512]   # Half resolution, double channels
What it contributes:

Downsamples spatial size (56×56 → 28×28)

Increases channels (256 → 512)

Reduces computation for deeper layers

Creates more semantic features (larger receptive field)

Identity Block Contribution: "Feature Enhancer"
python
Input:  [batch, 28, 28, 512]   # Fixed size, fixed channels
        ↓
Identity Block (3x in a row)
        ↓
Output: [batch, 28, 28, 512]   # SAME size, SAME channels
What it contributes:

Learns richer features at the same resolution

No shape change - just improves feature quality

Skip connections help gradients flow

Each block refines the features slightly more
so basically resnet architecture is lik it has two blocks identifty and conv block both do exact
 same thing like same layers goes through both but identituy has skip connectiuon to give output in a diff 
way and conv block doe siut the normal way that all other cn architecture 
# Both blocks have THESE EXACT SAME layers:
Conv1x1 → BN → ReLU → Conv3x3 → BN → ReLU → Conv1x1 → BN
in identity x = KL.Add()([x, input_tensor])  # Direct addition
Conv block does it the normal way"
YES! Conv block adds via a learned shortcut:

python
shortcut = KL.Conv2D(...)(input_tensor)  # Learned transformation
x = KL.Add()([x, shortcut])


Now todether how they are used
he Only Difference Summarized
Aspect	Identity Block	Conv Block
Main Path	✅ SAME	✅ SAME
Layers	✅ SAME	✅ SAME
Shortcut	Direct (no layers)	Learned (has Conv1x1)
When used	When shapes match	When shapes change

 Both blocks together create the feature hierarchy!


 
 Conv Block (changes shape) → Identity Blocks (refine features) → Conv Block (changes shape again) → Identity Blocks (refine again) → ...
 The Pattern: "One Conv Block + Many Identity Blocks"
In Each Stage:
text
STAGE 2: [CONV BLOCK] + [IDENTITY] + [IDENTITY]
STAGE 3: [CONV BLOCK] + [IDENTITY] + [IDENTITY] + [IDENTITY]
STAGE 4: [CONV BLOCK] + [IDENTITY] × (5 or 22)
STAGE 5: [CONV BLOCK] + [IDENTITY] + [IDENTITY]
Why This Pattern?
Block Type	When It's Used	How Many Times
Conv Block	ONCE at the start of each stage	1 per stage
Identity Block	MANY TIMES after the Conv Block	2-22 per stage
"""

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    BOTH BLOCKS HAVE SHORTCUTS!                                     │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  Identity Block:                                                                    │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  shortcut = input_tensor (DIRECT, NO LAYERS!)                                  ││
│  │  x = Add()([x, shortcut])                                                      ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  Conv Block:                                                                       │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  shortcut = Conv2D(input_tensor) (WITH LAYERS!)                                ││
│  │  x = Add()([x, shortcut])                                                      ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  ★ BOTH have shortcuts! The DIFFERENCE is what's IN the shortcut                  │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

# Identity Block shortcut:
# shortcut = input_tensor  (DIRECT, no processing!)
x = KL.Add()([x, input_tensor])
#            ↑
#            This is the shortcut - just passes input through

# Visual:
input_tensor ──────────────────────────────────────┐
    │                                               │
    ▼                                               │
Conv1x1 → BN → ReLU → Conv3x3 → BN → ReLU → Conv1x1 → BN
    │                                               │
    └───────────────┬───────────────────────────────┘
                    ▼
                Add() ←──┐
                    │    │  shortcut = input_tensor
                    ▼    │  (NO LAYERS, just identity)
                ReLU     │
                    │    │
                    ▼    │
                output ──┘

# Conv Block shortcut:
shortcut = KL.Conv2D(nb_filter3, (1, 1), strides=strides)(input_tensor)
#          ↑
#          This IS the shortcut - has a Conv2D layer!
shortcut = BatchNorm()(shortcut)
x = KL.Add()([x, shortcut])

# Visual:
input_tensor ──────────────────────────────────────┐
    │                                               │
    ▼                                               │
Conv1x1(stride) → BN → ReLU → Conv3x3 → BN → ReLU → Conv1x1 → BN
    │                                               │
    │                                               │
    └───────────────┬───────────▼───────────────────┘
                    │       Conv1x1(stride) → BN
                    │       ↑
                    │       shortcut = Conv2D(input_tensor)
                    │       (HAS LAYERS!)
                    │
                    └───────┬───────────────┘
                            ▼
                        Add()
                            │
                            ▼
                        ReLU
                            │
                            ▼
                        output
                        okay so just the diff is the identitu block does not have  has conv layer in shirtcut and whle conv block does have it

In [ ]:
def conv_block(input_tensor, kernel_size, filters, stage, block,
               strides=(2, 2), use_bias=True, train_bn=True):
    """conv_block is the block that has a conv layer at shortcut
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of middle conv layer at main path
        filters: list of integers, the nb_filters of 3 conv layer at main path
        stage: integer, current stage label, used for generating layer names
        block: 'a','b'..., current block label, used for generating layer names
        use_bias: Boolean. To use or not use a bias in conv layers.
        train_bn: Boolean. Train or freeze Batch Norm layers
    Note that from stage 3, the first conv layer at main path is with subsample=(2,2)
    And the shortcut should have subsample=(2,2) as well
    """
    nb_filter1, nb_filter2, nb_filter3 = filters
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    x = KL.Conv2D(nb_filter1, (1, 1), strides=strides,
                  name=conv_name_base + '2a', use_bias=use_bias)(input_tensor)
    x = BatchNorm(name=bn_name_base + '2a')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter2, (kernel_size, kernel_size), padding='same',
                  name=conv_name_base + '2b', use_bias=use_bias)(x)
    x = BatchNorm(name=bn_name_base + '2b')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter3, (1, 1), name=conv_name_base +
                  '2c', use_bias=use_bias)(x)
    x = BatchNorm(name=bn_name_base + '2c')(x, training=train_bn)
    shortcut=KL.Conv2D(nb_filter3, (1, 1), strides=strides,
                         name=conv_name_base + '1', use_bias=use_bias)(input_tensor)
    shortcut = BatchNorm(name=bn_name_base + '1')(shortcut, training=train_bn)
    """The +1 in the shortcut name ('branch1') identifies it as the SHORTCUT PATH, while 2a, 2b, 2c are the MAIN PATH layers.

1 = Branch 1 (shortcut) - the skip connection path

2a, 2b, 2c = Branch 2 (main path) - the conv layers

Why? To give each layer a UNIQUE name so TensorFlow can track them separately. The shortcut needs its own conv layer in Conv Block to match dimensions, so it gets labeled branch1.

"""
    x = KL.Add()([x, shortcut])
    x = KL.Activation('relu', name='res' + str(stage) + block + '_out')(x)
    return x

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    WHY THIS STRUCTURE                                              │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  RESNET50:                                                                         │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Stage 1: Conv7x7 + MaxPool                                                    ││
│  │  Stage 2: Conv + 2x Identity → C2                                              ││
│  │  Stage 3: Conv + 3x Identity → C3                                              ││
│  │  Stage 4: Conv + 5x Identity → C4  ← block_count = 5                          ││
│  │  Stage 5: Conv + 2x Identity → C5                                              ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  RESNET101:                                                                        │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Stage 1: Conv7x7 + MaxPool                                                    ││
│  │  Stage 2: Conv + 2x Identity → C2                                              ││
│  │  Stage 3: Conv + 3x Identity → C3                                              ││
│  │  Stage 4: Conv + 22x Identity → C4  ← block_count = 22 (DEEPER!)              ││
│  │  Stage 5: Conv + 2x Identity → C5                                              ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  ★ SAME BLOCKS, JUST MORE IDENTITY BLOCKS IN STAGE 4!                              │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

In [ ]:
def resnet_graph(input_image, architecture, stage5=False, train_bn=True):
    """Build a ResNet graph.
        architecture: Can be resnet50 or resnet101
        stage5: Boolean. If False, stage5 of the network is not created
        train_bn: Boolean. Train or freeze Batch Norm layers
    """

    """ResNet50	50 layers	Shallower (faster, less memory)
ResNet101	101 layers	Deeper (more accurate, slower)
They use the SAME conv and identity blocks - just different NUMBER of identity blocks in Stage 4!
block_count = {"resnet50": 5, "resnet101": 22}[architecture]
This decides how many identity blocks to put in Stage 4!

C4 is the OUTPUT of Stage 4. The loop keeps updating x, and after ALL identity blocks, C4 gets the final valu

stage5 value	What happens	Why
True	Create Stage 5 → C5	For deeper networks (full ResNet)
False	Skip Stage 5 → C5 = None	For smaller networks (save memory)

Stage 1 is just the "entry layer" - it's NOT a ResNet stage with blocks. It's the initial convolution + pooling that happens BEFORE the actual ResNet stages begin
"""
    assert architecture in ["resnet50", "resnet101"]
    # Stage 1
    x = KL.ZeroPadding2D((3, 3))(input_image)
    x = KL.Conv2D(64, (7, 7), strides=(2, 2), name='conv1', use_bias=True)(x)
    x = BatchNorm(name='bn_conv1')(x, training=train_bn)
    x = KL.Activation('relu')(x)
    C1 = x = KL.MaxPooling2D((3, 3), strides=(2, 2), padding="same")(x)
    # Stage 2
    x = conv_block(x, 3, [64, 64, 256], stage=2, block='a', strides=(1, 1), train_bn=train_bn)
    x = identity_block(x, 3, [64, 64, 256], stage=2, block='b', train_bn=train_bn)
    C2 = x = identity_block(x, 3, [64, 64, 256], stage=2, block='c', train_bn=train_bn)
     # Stage 3
    x = conv_block(x, 3, [128, 128, 512], stage=3, block='a', train_bn=train_bn)
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='b', train_bn=train_bn)
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='c', train_bn=train_bn)
    C3 = x = identity_block(x, 3, [128, 128, 512], stage=3, block='d', train_bn=train_bn)
    # Stage 4
    x = conv_block(x, 3, [256, 256, 1024], stage=4, block='a', train_bn=train_bn)
    block_count = {"resnet50": 5, "resnet101": 22}[architecture]
    for i in range(block_count):
        x = identity_block(x, 3, [256, 256, 1024], stage=4, block=chr(98 + i), train_bn=train_bn)
    C4 = x
    # Stage 5
    if stage5:
        x = conv_block(x, 3, [512, 512, 2048], stage=5, block='a', train_bn=train_bn)
        x = identity_block(x, 3, [512, 512, 2048], stage=5, block='b', train_bn=train_bn)
        C5 = x = identity_block(x, 3, [512, 512, 2048], stage=5, block='c', train_bn=train_bn)
    else:
        C5 = None
    return [C1, C2, C3, C4, C5]


┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    FILE ORDER vs EXECUTION ORDER                                    │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  FILE ORDER (Top to Bottom):                                                       │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  1. Utility functions (log, BatchNorm, compute_backbone_shapes)                ││
│  │  2. ResNet blocks (identity_block, conv_block, resnet_graph)  ← Defined here   ││
│  │  3. Proposal Layer                                                             ││
│  │  4. ROI Align Layer                                                           ││
│  │  5. Detection Target Layer                                                    ││
│  │  6. Detection Layer                                                           ││
│  │  7. RPN (rpn_graph, build_rpn_model)                                         ││
│  │  8. FPN HEADS (fpn_classifier_graph, build_fpn_mask_graph)  ← Defined here   ││
│  │  9. Loss Functions                                                            ││
│  │  10. Data Generator                                                           ││
│  │  11. MaskRCNN Class                                                           ││
│  │      └─── build() method  ← FPN IS BUILT HERE!                              ││
│  │      └─── train() method                                                      ││
│  │      └─── detect() method                                                     ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  EXECUTION ORDER (When image flows through):                                       │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 1: resnet_graph()  ← Called from build()                                ││
│  │  STEP 2: FPN (P2-P6)     ← Called from build()                                ││
│  │  STEP 3: RPN             ← Called from build()                                ││
│  │  STEP 4: ProposalLayer   ← Called from build()                                ││
│  │  STEP 5: ROI Align       ← Called from build()                                ││
│  │  STEP 6: fpn_classifier_graph()  ← Called from build()                       ││
│  │  STEP 7: build_fpn_mask_graph()  ← Called from build()                       ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

FPN is a network that creates feature maps at MULTIPLE SCALES so the model can detect objects of ALL sizes.

Without FPN: Only one feature map (C5) → good for large objects, bad for small ones
With FPN: Multiple feature maps (P2-P6) → good for ALL object sizes!
C5 (32x32) ──► P5 (32x32)  ← For LARGE objects
    ↓ Upsample
C4 (64x64) ──► P4 (64x64)  ← For MEDIUM-LARGE objects
    ↓ Upsample
C3 (128x128) ──► P3 (128x128) ← For MEDIUM objects
    ↓ Upsample
C2 (256x256) ──► P2 (256x256) ← For SMALL objects
. FPN Construction (The Pyramid Itself)
python
# This builds the actual pyramid - NOT a function, just code in build()
P5 = KL.Conv2D(256, (1, 1), name='fpn_c5p5')(C5)
P4 = KL.Add(...)([...])
P3 = KL.Add(...)([...])
P2 = KL.Add(...)([...])
P6 = KL.MaxPooling2D(...)(P5)
2. FPN Heads (The Functions You Posted)
python
# These USE the pyramid features - these ARE functions
fpn_classifier_graph()   # Uses P2-P5 to classify objects
build_fpn_mask_graph()   # Uses P2-P5 to make masks

def build(self, mode, config):
    # ============================================================
    # STEP 1: RESNET BACKBONE
    # ============================================================
    _, C2, C3, C4, C5 = resnet_graph(input_image, config.BACKBONE,
                                     stage5=True, train_bn=config.TRAIN_BN)
    # ↑ Output: C2, C3, C4, C5
    
    # ============================================================
    # STEP 2: FPN CONSTRUCTION (THE PYRAMID!)
    # ============================================================
    # THIS IS THE FPN ITSELF!
    P5 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c5p5')(C5)
    P4 = KL.Add(name="fpn_p4add")([
        KL.UpSampling2D(size=(2, 2), name="fpn_p5upsampled")(P5),
        KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c4p4')(C4)
    ])
    P3 = KL.Add(name="fpn_p3add")([
        KL.UpSampling2D(size=(2, 2), name="fpn_p4upsampled")(P4),
        KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c3p3')(C3)
    ])
    P2 = KL.Add(name="fpn_p2add")([
        KL.UpSampling2D(size=(2, 2), name="fpn_p3upsampled")(P3),
        KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c2p2')(C2)
    ])
    
    # Final 3x3 conv on each P layer
    P2 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p2")(P2)
    P3 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p3")(P3)
    P4 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p4")(P4)
    P5 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p5")(P5)
    
    # P6 for RPN
    P6 = KL.MaxPooling2D(pool_size=(1, 1), strides=2, name="fpn_p6")(P5)
    
    # ↑↑↑ FPN CONSTRUCTION COMPLETE: P2, P3, P4, P5, P6
    
    # ============================================================
    # STEP 3: RPN (Uses P2-P6)
    # ============================================================
    rpn_feature_maps = [P2, P3, P4, P5, P6]  # ← FPN features go to RPN!
    rpn = build_rpn_model(...)
    for p in rpn_feature_maps:
        layer_outputs.append(rpn([p]))
    rpn_class_logits, rpn_class, rpn_bbox = outputs
    
    # ============================================================
    # STEP 4: PROPOSAL LAYER
    # ============================================================
    rpn_rois = ProposalLayer(...)([rpn_class, rpn_bbox, anchors])
    
    # ============================================================
    # STEP 5: DETECTION TARGET LAYER (Training only)
    # ============================================================
    rois, target_class_ids, target_bbox, target_mask = DetectionTargetLayer(...)(
        [target_rois, input_gt_class_ids, gt_boxes, input_gt_masks])
    
    # ============================================================
    # STEP 6: FPN CLASSIFIER HEAD ← YOUR FIRST FUNCTION!
    # ============================================================
    mrcnn_feature_maps = [P2, P3, P4, P5]  # ← FPN features for heads!
    
    mrcnn_class_logits, mrcnn_class, mrcnn_bbox = fpn_classifier_graph(
        rois,                    # ← Proposals
        mrcnn_feature_maps,      # ← P2, P3, P4, P5 (FROM FPN!)
        input_image_meta,
        config.POOL_SIZE,
        config.NUM_CLASSES,
        train_bn=config.TRAIN_BN,
        fc_layers_size=config.FPN_CLASSIF_FC_LAYERS_SIZE
    )
    # ↑↑↑ fpn_classifier_graph() EXECUTED HERE!
    
    # ============================================================
    # STEP 7: FPN MASK HEAD ← YOUR SECOND FUNCTION!
    # ============================================================
    mrcnn_mask = build_fpn_mask_graph(
        rois,                    # ← Proposals
        mrcnn_feature_maps,      # ← P2, P3, P4, P5 (FROM FPN!)
        input_image_meta,
        config.MASK_POOL_SIZE,
        config.NUM_CLASSES,
        train_bn=config.TRAIN_BN
    )
    # ↑↑↑ build_fpn_mask_graph() EXECUTED HERE!

Final Summary: What FPN Does and Where It Fits
FPN (Feature Pyramid Network) is the bridge between ResNet's single-scale features and Mask R-CNN's multi-scale detection needs. It takes the four feature maps from ResNet—C2 (256×256, fine details), C3 (128×128, object parts), C4 (64×64, whole objects), and C5 (32×32, semantic context)—and transforms them into a five-level pyramid: P2 (256×256), P3 (128×128), P4 (64×64), P5 (32×32), and P6 (16×16). It does this by first reducing all channels to 256 via 1×1 convolutions (lateral connections), then repeatedly upsampling the deeper, more semantic features and adding them to the shallower, more detailed features (top-down pathway), and finally smoothing each level with a 3×3 convolution. The result is a set of feature maps that all have the same channel depth (256) but different spatial resolutions, allowing the network to detect objects at every scale—P2 for tiny objects, P3 for medium, P4 for large, P5 for very large, and P6 (created by max-pooling P5) for the largest. These five maps are then passed directly to the RPN, which applies the same shared network to each level to generate region proposals (candidate boxes) across all scales, ensuring that no object—whether small or large—is missed. In essence, FPN gives RPN the ability to "see" objects of all sizes by providing a multi-scale feature pyramid built from ResNet's hierarchical features.

In [ ]:
############################################################
#  Region Proposal Network (RPN)
############################################################
def rpn_graph(feature_map, anchors_per_location, anchor_stride):
    """Builds the computation graph of Region Proposal Network.

    feature_map: backbone features [batch, height, width, depth]
    anchors_per_location: number of anchors per pixel in the feature map
    anchor_stride: Controls the density of anchors. Typically 1 (anchors for
                   every pixel in the feature map), or 2 (every other pixel).

    Returns:
        rpn_class_logits: [batch, H * W * anchors_per_location, 2] Anchor classifier logits (before softmax)
        rpn_probs: [batch, H * W * anchors_per_location, 2] Anchor classifier probabilities.
        rpn_bbox: [batch, H * W * anchors_per_location, (dy, dx, log(dh), log(dw))] Deltas to be
                  applied to anchors.
    """
    # TODO: check if stride of 2 causes alignment issues if the feature map
    # is not even.
    # Shared convolutional base of the RPN
    #Applies 3×3 convolution with 512 filters
    #Why "shared"? Both heads (classification and bbox) use these features.

    shared = KL.Conv2D(512, (3, 3), padding='same', activation='relu',
                       strides=anchor_stride,
                       name='rpn_conv_shared')(feature_map)
    # Anchor Score. [batch, height, width, anchors per location * 2].
    #Step 2: Classification Head (Is it FG or BG?) For each anchor: predicts 2 values (FG score, BG score)
    x = KL.Conv2D(2 * anchors_per_location, (1, 1), padding='valid',
                  activation='linear', name='rpn_class_raw')(shared)
    # Reshape to [batch, anchors, 2] Reshapes from [batch, H, W, 2*A] to [batch, H*W*A, 2]
    rpn_class_logits = KL.Lambda(
        lambda t: tf.reshape(t, [tf.shape(input=t)[0], -1, 2]))(x)
    # Softmax on last dimension of BG/FG. Applies softmax to get probabilities probability of background, probability of foreground (object)
    rpn_probs = KL.Activation(
        "softmax", name="rpn_class_xxx")(rpn_class_logits)
    #BBox Regression Head (How to adjust the box)
    # Bounding box refinement. [batch, H, W, anchors per location * depth]
    # where depth is [x, y, log(w), log(h)]
    x = KL.Conv2D(anchors_per_location * 4, (1, 1), padding="valid",
                  activation='linear', name='rpn_bbox_pred')(shared)
#     dy = how much to shift center_y (normalized by height)
# dx = how much to shift center_x (normalized by width)
# log(dh) = how much to change height (log scale)
# log(dw) = how much to change width (log scale)
    # Reshape to [batch, anchors, 4]
    """Example:Anchor: [100, 50, 200, 150]  (y1, x1, y2, x2)
Deltas: [0.1, -0.05, 0.2, 0.15]

Apply deltas:
center_y = 150 + 0.1*100 = 160
center_x = 100 + (-0.05)*100 = 95
height = 100 * exp(0.2) = 122
width = 100 * exp(0.15) = 116

Refined box: [160-61, 95-58, 160+61, 95+58] = [99, 37, 221, 153]"""
    rpn_bbox = KL.Lambda(lambda t: tf.reshape(t, [tf.shape(input=t)[0], -1, 4]))(x)
#Reshapes to [batch, H*W*A, 4]Each anchor has 4 deltas
    return [rpn_class_logits, rpn_probs, rpn_bbox]

""" What rpn_graph() Returns (Raw Outputs)
rpn_class_logits: [batch, total_anchors, 2] - Classification scores (before softmax)

rpn_probs: [batch, total_anchors, 2] - Classification probabilities (after softmax)

rpn_bbox: [batch, total_anchors, 4] - BBox deltas for each anchor

build_rpn_model() wraps rpn_graph() into a reusable Keras Model so the SAME RPN can be applied to multiple FPN levels with SHARED weights.
# Would need to manually create RPN for each level:
rpn_class_logits_p2, rpn_probs_p2, rpn_bbox_p2 = rpn_graph(P2, ...)
rpn_class_logits_p3, rpn_probs_p3, rpn_bbox_p3 = rpn_graph(P3, ...)
rpn_class_logits_p4, rpn_probs_p4, rpn_bbox_p4 = rpn_graph(P4, ...)
rpn_class_logits_p5, rpn_probs_p5, rpn_bbox_p5 = rpn_graph(P5, ...)
rpn_class_logits_p6, rpn_probs_p6, rpn_bbox_p6 = rpn_graph(P6, ...)

# Problem: Each would have SEPARATE weights! (not shared)"""
def build_rpn_model(anchor_stride, anchors_per_location, depth):
    """Builds a Keras model of the Region Proposal Network.
    It wraps the RPN graph so it can be used multiple times with shared
    weights.

    anchors_per_location: number of anchors per pixel in the feature map
    anchor_stride: Controls the density of anchors. Typically 1 (anchors for
                   every pixel in the feature map), or 2 (every other pixel).
    depth: Depth of the backbone feature map.

    Returns a Keras Model object. The model outputs, when called, are:
    rpn_class_logits: [batch, H * W * anchors_per_location, 2] Anchor classifier logits (before softmax)
    rpn_probs: [batch, H * W * anchors_per_location, 2] Anchor classifier probabilities.
    rpn_bbox: [batch, H * W * anchors_per_location, (dy, dx, log(dh), log(dw))] Deltas to be
                applied to anchors.
    """
    input_feature_map = KL.Input(shape=[None, None, depth],
                                 name="input_rpn_feature_map")
    outputs = rpn_graph(input_feature_map, anchors_per_location, anchor_stride)
    return KM.Model([input_feature_map], outputs, name="rpn_model")

    

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    RPN: CLASSIFICATION + BBOX HEADS                                │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  INPUT: Feature Map [batch, H, W, depth]                                           │
│      │                                                                              │
│      ▼                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  SHARED CONV: KL.Conv2D(512, (3,3), activation='relu')                         ││
│  │  Output: [batch, H, W, 512]                                                   ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                               │
│                    ┌───────────────┴───────────────┐                              │
│                    │                               │                              │
│                    ▼                               ▼                              │
│  ┌─────────────────────────────────┐  ┌──────────────────────────────────────────┐│
│  │  CLASSIFICATION HEAD            │  │  BBOX REGRESSION HEAD                    ││
│  │  ┌───────────────────────────┐  │  │  ┌──────────────────────────────────────┐││
│  │  │  KL.Conv2D(2*A, (1,1))    │  │  │  │  KL.Conv2D(4*A, (1,1))              │││
│  │  │  activation='linear'      │  │  │  │  activation='linear'                │││
│  │  │  Output: [batch,H,W,2A]   │  │  │  │  Output: [batch,H,W,4A]             │││
│  │  └───────────────────────────┘  │  │  └──────────────────────────────────────┘││
│  │              │                   │  │              │                          ││
│  │              ▼                   │  │              ▼                          ││
│  │  ┌───────────────────────────┐  │  │  ┌──────────────────────────────────────┐││
│  │  │  Reshape:                  │  │  │  │  Reshape:                           │││
│  │  │  [batch, H*W*A, 2]        │  │  │  │  [batch, H*W*A, 4]                 │││
│  │  │  rpn_class_logits          │  │  │  │  rpn_bbox                          │││
│  │  └───────────────────────────┘  │  │  └──────────────────────────────────────┘││
│  │              │                   │  │                                          ││
│  │              ▼                   │  │                                          ││
│  │  ┌───────────────────────────┐  │  │                                          ││
│  │  │  Softmax                   │  │  │                                          ││
│  │  │  rpn_probs                 │  │  │                                          ││
│  │  └───────────────────────────┘  │  │                                          ││
│  └─────────────────────────────────┘  └──────────────────────────────────────────┘│
│                    │                               │                              │
│                    └───────────────┬───────────────┘                              │
│                                    ▼                                               │
│  OUTPUT:                                                                          │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  rpn_class_logits: [batch, H*W*A, 2]  ← Classification (before softmax)       ││
│  │  rpn_probs:        [batch, H*W*A, 2]  ← Classification (after softmax)        ││
│  │  rpn_bbox:         [batch, H*W*A, 4]  ← BBox regression                       ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

How rpn being used in main mask block
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    ONE RPN MODEL, MANY FEATURE MAPS                                │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  rpn = Keras Model (created once)                                                 │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Input: feature_map [batch, H, W, 256]                                        ││
│  │  ├── Conv2D(512, 3x3)  ← weights w1, w2, w3 (SHARED!)                        ││
│  │  ├── Conv2D(2A, 1x1)   ← weights w4, w5 (SHARED!)                            ││
│  │  └── Conv2D(4A, 1x1)   ← weights w6, w7 (SHARED!)                            ││
│  │  Output: [class_logits, probs, bbox]                                          ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                               │
│        ┌───────────────────────────┼───────────────────────────┐                  │
│        │                           │                           │                  │
│        ▼                           ▼                           ▼                  │
│  ┌───────────┐              ┌───────────┐              ┌───────────┐             │
│  │    P2     │              │    P3     │              │    P4     │             │
│  │ 256x256   │              │ 128x128   │              │ 64x64     │             │
│  └─────┬─────┘              └─────┬─────┘              └─────┬─────┘             │
│        │                           │                           │                  │
│        ▼                           ▼                           ▼                  │
│  ┌───────────┐              ┌───────────┐              ┌───────────┐             │
│  │ rpn(P2)   │              │ rpn(P3)   │              │ rpn(P4)   │             │
│  │ Uses w1-w7│              │ Uses w1-w7│              │ Uses w1-w7│             │
│  │ (SHARED!) │              │ (SHARED!) │              │ (SHARED!) │             │
│  └─────┬─────┘              └─────┬─────┘              └─────┬─────┘             │
│        │                           │                           │                  │
│        ▼                           ▼                           ▼                  │
│  ┌───────────┐              ┌───────────┐              ┌───────────┐             │
│  │ [class,   │              │ [class,   │              │ [class,   │             │
│  │  probs,   │              │  probs,   │              │  probs,   │             │
│  │  bbox]    │              │  bbox]    │              │  bbox]    │             │
│  └───────────┘              └───────────┘              └───────────┘             │
│        │                           │                           │                  │
│        └───────────────────────────┼───────────────────────────┘                  │
│                                    ▼                                               │
│                    ┌─────────────────────────────────────────────────────────────────┐│
│                    │  Concatenate all levels:                                       ││
│                    │  rpn_class_logits: [batch, total_anchors, 2]                 ││
│                    │  rpn_probs:        [batch, total_anchors, 2]                 ││
│                    │  rpn_bbox:         [batch, total_anchors, 4]                 ││
│                    └─────────────────────────────────────────────────────────────────┘│
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

okay so u mena when rpn is applied in build it takes all fpn outputs inside its keras model layer and creates outputs to all then those are feeded to porposal layer

What Goes INTO ProposalLayer
python
rpn_rois = ProposalLayer(...)([rpn_class, rpn_bbox, anchors])
#                              ↑           ↑          ↑
#                          Input 1    Input 2    Input 3
Input 1: rpn_class - Classification Scores
text
Shape: [batch, total_anchors, 2]
Contains: For each anchor → [BG_prob, FG_prob]
- FG_prob = probability this anchor contains an object
- BG_prob = probability this anchor is background
From ALL FPN levels combined! (P2, P3, P4, P5, P6)

Input 2: rpn_bbox - BBox Deltas
text
Shape: [batch, total_anchors, 4]
Contains: For each anchor → [dy, dx, log(dh), log(dw)]
- These tell how to refine the anchor box
From ALL FPN levels combined!

Input 3: anchors - Pre-defined Boxes
text
Shape: [batch, total_anchors, 4]
Contains: For each anchor → [y1, x1, y2, x2]
- The actual anchor boxes (in normalized coordinates)
Pre-generated for all FPN levels!



PROPOSAL LAYER - Simple Explanation

What It Does (In Simple Terms)

The Proposal Layer takes 785,000 rough anchor boxes and picks the best 2,000/1,000 boxes that actually contain objects. It's like having 785,000 guesses about where objects might be, keeping only the best 2,000 guesses, refining those guesses to fit objects better, and removing overlapping guesses so you don't have duplicates.

What Goes IN (Inputs)

Input: rpn_class, Shape: [batch, 785k, 2], What It Contains: FG/BG scores for every anchor from ALL FPN levels (P2-P6). Input: rpn_bbox, Shape: [batch, 785k, 4], What It Contains: 4 deltas [dy, dx, log(dh), log(dw)] for every anchor. Input: anchors, Shape: [batch, 785k, 4], What It Contains: Pre-defined anchor boxes [y1, x1, y2, x2].

What Happens INSIDE

785,000 anchors with scores and deltas go through the following steps: First, keep top 6,000 by FG score (best guesses). Second, refine each box using deltas to adjust position and size. Third, clip to image boundaries to keep boxes inside image. Fourth, apply NMS to remove overlapping boxes with IoU greater than 0.7. Fifth, keep top 2,000 for training or 1,000 for inference. Finally, output rpn_rois with shape [batch, 2000/1000, (y1, x1, y2, x2)].

What Comes OUT (Output)

The output is rpn_rois with shape [batch, proposal_count, (y1, x1, y2, x2)], where proposal_count is 2000 for training or 1000 for inference. Each box is in normalized coordinates ranging from 0 to 1.

Where It Goes (Output Destination)

The rpn_rois output goes to ROI ALIGN (PyramidROIAlign), which then extracts features for each proposal, passes them to Classification and Mask Heads, and finally produces the output with Classes, Refined Boxes, and Masks.

What Exactly Are Anchors?
Anchors are pre-defined boxes placed at every position on the image.
 Anchors are PRE-GENERATED (NOT predicted!)

Anchor Properties:
Fixed positions: Placed at regular intervals

Fixed sizes: From RPN_ANCHOR_SCALES (32, 64, 128, 256, 512)

Fixed ratios: From RPN_ANCHOR_RATIOS (0.5, 1:2, 2:1)

Total anchors: ~785,000 for a 1024x1024 image

NEXT


Question 1: What Are Deltas?
Deltas are the "instructions" on how to adjust an anchor box to better fit an object.

Simple Explanation:
Instead of predicting boxes directly, RPN predicts how much to change each anchor box. These changes are called "deltas."

The 4 Deltas:
python
deltas = [dy, dx, log(dh), log(dw)]
Delta	What It Does	How It Works
dy	Shift box up/down	center_y += dy * height
dx	Shift box left/right	center_x += dx * width
log(dh)	Change height	height *= exp(log(dh))
log(dw)	Change width	width *= exp(log(dw))
Example:
text
Anchor: [100, 50, 200, 150]  (100x100 box)
Deltas: [0.1, -0.05, 0.2, 0.15]

Meaning:
- Move center down by 10% of height
- Move center left by 5% of width
- Increase height by 22% (exp(0.2))
- Increase width by 16% (exp(0.15))

Result: [99, 37, 221, 153]  (refined box)
Question 2: How Are Deltas Predicted?
Deltas are the OUTPUT of RPN's BBox Regression Head!


What RPN Predicts:
text
For each anchor (785k anchors):
  - 2 scores: [BG_prob, FG_prob]  (Classification)
  - 4 deltas: [dy, dx, log(dh), log(dw)]  (BBox Regression)


   Anchors are PRE-GENERATED (NOT predicted!)

In [ ]:
############################################################
#  Proposal Layer
############################################################

def apply_box_deltas_graph(boxes, deltas):
    """Applies the given deltas to the given boxes.
    boxes: [N, (y1, x1, y2, x2)] boxes to update
    deltas: [N, (dy, dx, log(dh), log(dw))] refinements to apply
    """
    # Convert to y, x, h, w
    #It takes boxes defined by their corners [y1, x1, y2, x2] and converts them to [center_y, center_x, height, width].
#Step 1: Calculate Height and Width What this does: Subtracts top from bottom to get height, left from right to get width.
#Step 2: Calculate Center What this does: Adds half the height/width to the top/left corner to find the middle.
    height = boxes[:, 2] - boxes[:, 0]
    width = boxes[:, 3] - boxes[:, 1]
    center_y = boxes[:, 0] + 0.5 * height
    center_x = boxes[:, 1] + 0.5 * width
    # Apply deltas
    center_y += deltas[:, 0] * height
    center_x += deltas[:, 1] * width
    height *= tf.exp(deltas[:, 2])
    width *= tf.exp(deltas[:, 3])
    """What Each Delta Does:
Delta	Operation	Effect
deltas[:, 0] (dy)	center_y += dy * height	Move box vertically (up/down)
deltas[:, 1] (dx)	center_x += dx * width	Move box horizontally (left/right)
deltas[:, 2] (log(dh))	height *= exp(log(dh))	Change height (taller/shorter)
deltas[:, 3] (log(dw))	width *= exp(log(dw))	Change width (wider/narrower)

exp() ensures height and width are ALWAYS positive numbers!

STEP 1: CONVERT TO CENTER FORMAT
    [y1, x1, y2, x2] → [center_y, center_x, height, width]
    
STEP 2: APPLY DELTAS
    [center_y, center_x, height, width] + [dy, dx, log(dh), log(dw)]
    → [center_y', center_x', height', width']
    
STEP 3: CONVERT BACK TO CORNER FORMAT
    [center_y', center_x', height', width'] → [y1', x1', y2', x2']"""
    # Convert back to y1, x1, y2, x2
    y1 = center_y - 0.5 * height
    x1 = center_x - 0.5 * width
    y2 = y1 + height
    x2 = x1 + width
    result = tf.stack([y1, x1, y2, x2], axis=1, name="apply_box_deltas_out")
    return result


def clip_boxes_graph(boxes, window):
    """
    boxes: [N, (y1, x1, y2, x2)]
    window: [4] in the form y1, x1, y2, x2
    """
    # Split
    wy1, wx1, wy2, wx2 = tf.split(window, 4)
    y1, x1, y2, x2 = tf.split(boxes, 4, axis=1)
    # Clip
    y1 = tf.maximum(tf.minimum(y1, wy2), wy1)
    x1 = tf.maximum(tf.minimum(x1, wx2), wx1)
    y2 = tf.maximum(tf.minimum(y2, wy2), wy1)
    x2 = tf.maximum(tf.minimum(x2, wx2), wx1)
    clipped = tf.concat([y1, x1, y2, x2], axis=1, name="clipped_boxes")
    clipped.set_shape((clipped.shape[0], 4))
    return clipped
#Simply put: clip_boxes_graph() ensures that no box goes outside the image. It trims any part of a box that extends beyond the image boundaries, keeping all boxes valid for later processing.
#proposal_count is the number of region proposals the layer should output.Training	2000	More proposals = better training diversity
#Inference	1000	Fewer proposals = faster inference
"""What is nms_threshold?
nms_threshold is the IoU threshold for Non-Maximum Suppression.

IoU = Intersection over Union (how much two boxes overlap)

If IoU > nms_threshold, boxes are considered duplicates and removed

Threshold	Effect
0.7 (default)	Remove boxes that overlap >70% (balanced)
0.5 (lower)	Remove more boxes (fewer proposals)
0.9 (higher)	Keep more boxes (more proposals)"""
#get_config() saves the layer's configuration so it can be serialized (saved to disk) and reloaded later.

class ProposalLayer(KL.Layer):
    """Receives anchor scores and selects a subset to pass as proposals
    to the second stage. Filtering is done based on anchor scores and
    non-max suppression to remove overlaps. It also applies bounding
    box refinement deltas to anchors.

    Inputs:
        rpn_probs: [batch, num_anchors, (bg prob, fg prob)]
        rpn_bbox: [batch, num_anchors, (dy, dx, log(dh), log(dw))]
        anchors: [batch, num_anchors, (y1, x1, y2, x2)] anchors in normalized coordinates

    Returns:
        Proposals in normalized coordinates [batch, rois, (y1, x1, y2, x2)]
    """

    def __init__(self, proposal_count, nms_threshold, config=None, **kwargs):
        super(ProposalLayer, self).__init__(**kwargs)
        self.config = config
        self.proposal_count = proposal_count
        self.nms_threshold = nms_threshold

    def get_config(self):
        config = super(ProposalLayer, self).get_config()
        config["config"] = self.config.to_dict()
        config["proposal_count"] = self.proposal_count
        config["nms_threshold"] = self.nms_threshold
        return config

    def call(self, inputs):
        # Box Scores. Use the foreground class confidence. [Batch, num_rois, 1]Extract Foreground Scores
        #Why only FG? We only care about anchors that CONTAIN objects (foreground). Background anchors are useless to us.
        scores = inputs[0][:, :, 1]
        # Box deltas [batch, num_rois, 4]
        deltas = inputs[1]
        #What it does: Scales the deltas by a standard deviation factor.
        deltas = deltas * np.reshape(self.config.RPN_BBOX_STD_DEV, [1, 1, 4])
        # AnchorsWhat it does: Just stores the anchor boxes for later use.
        anchors = inputs[2]
  #This code is the Pre-NMS Selection step - it reduces ~785,000 anchors to the top 6,000 most promising ones.
        # Improve performance by trimming to top anchors by score
        # and doing the rest on the smaller subset.
        #Why scale? During training, RPN learns normalized deltas. Scaling brings them to the right magnitude.
        #Keep at most 6,000 anchors, or fewer if there aren't enough anchors."below line
        pre_nms_limit = tf.minimum(self.config.PRE_NMS_LIMIT, tf.shape(input=anchors)[1])
        #What it does: Sets how many anchors to keep before NMS.
        """Why 6000?

785k anchors is too many for NMS (slow!)

Keeping top 6,000 makes NMS much faster

The top 6,000 anchors are already the best candidates

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    TOP-K SELECTION                                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  All Anchors (785k):                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  Index:  0     1     2     3     4     5     6     7     8     9    ...      ││
│  │  Score:  0.95  0.12  0.88  0.45  0.92  0.33  0.78  0.21  0.65  0.09  ...      ││
│  │           ↑                 ↑     ↑           ↑           ↑                     ││
│  │       highest             high   high        high        high                   ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  Top 6,000 Scores:                                                                  │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  [0.95, 0.92, 0.88, 0.78, 0.65, 0.45, ...]  (6,000 highest scores)           ││
│  │   ↑     ↑     ↑     ↑     ↑     ↑                                              ││
│  │  [0,    4,    2,    6,    8,    3, ...]     ← These are the INDICES!         ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  Result: ix = [0, 4, 2, 6, 8, 3, ...]  (indices of top 6,000 anchors)            │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘"""
#Top-K Selection (The Core!)
        ix = tf.nn.top_k(scores, pre_nms_limit, sorted=True,
                         name="top_anchors").indices
        #What it does: Finds the indices of the top pre_nms_limit anchors with highest FG scores.
        #What it does: Uses the indices (ix) to select only the top anchors from scores, deltas, and anchors.
        scores = utils.batch_slice([scores, ix], lambda x, y: tf.gather(x, y),
                                   self.config.IMAGES_PER_GPU)
        deltas = utils.batch_slice([deltas, ix], lambda x, y: tf.gather(x, y),
                                   self.config.IMAGES_PER_GPU)
        pre_nms_anchors = utils.batch_slice([anchors, ix], lambda a, x: tf.gather(a, x),
                                    self.config.IMAGES_PER_GPU,
                                    names=["pre_nms_anchors"])
        # Apply deltas to anchors to get refined anchors.
        # [batch, N, (y1, x1, y2, x2)]
        boxes = utils.batch_slice([pre_nms_anchors, deltas],
                                  lambda x, y: apply_box_deltas_graph(x, y),
                                  self.config.IMAGES_PER_GPU,
                                  names=["refined_anchors"])

        # Clip to image boundaries. Since we're in normalized coordinates,
        # clip to 0..1 range. [batch, N, (y1, x1, y2, x2)] Ensures all refined boxes stay within the image boundaries [0, 1].
        window = np.array([0, 0, 1, 1], dtype=np.float32)
        boxes = utils.batch_slice(boxes,
                                  lambda x: clip_boxes_graph(x, window),
                                  self.config.IMAGES_PER_GPU,
                                  names=["refined_anchors_clipped"])

        # Filter out small boxes
        # According to Xinlei Chen's paper, this reduces detection accuracy
        # for small objects, so we're skipping it.

        # Non-max suppression
        """What this does:

Takes 6,000 boxes and their scores

Removes boxes that overlap too much (IoU > nms_threshold)

Keeps at most proposal_count boxes
1. Sort boxes by score (highest first)
2. Pick highest score box → KEEP
3. Remove all boxes with IoU > 0.7 → DUPLICATES
4. Repeat until proposal_count boxes selected
Step 3: Pad to Exact Count
Part 4: Batch Processing
Part 5: Set Output Shape (Graph Mode)"""
        def nms(boxes, scores):
            indices = tf.image.non_max_suppression(
                boxes, scores, self.proposal_count,
                self.nms_threshold, name="rpn_non_max_suppression")
            proposals = tf.gather(boxes, indices)
            # Pad if needed
            padding = tf.maximum(self.proposal_count - tf.shape(input=proposals)[0], 0)
            proposals = tf.pad(tensor=proposals, paddings=[(0, padding), (0, 0)])
            return proposals
        proposals = utils.batch_slice([boxes, scores], nms,
                                      self.config.IMAGES_PER_GPU)

        if not context.executing_eagerly():
            # Infer the static output shape:
            out_shape = self.compute_output_shape(None)
            proposals.set_shape(out_shape)
        return proposals

    def compute_output_shape(self, input_shape):
        return None, self.proposal_count, 4
        


mportant Clarification: 0.3 and 0.7 are NOT in ProposalLayer!
These thresholds belong to RPN TRAINING (when anchors are labeled as positive/negative), NOT to ProposalLayer (which is inference).

OVERALL FULL PROPOSAL


┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    6,000 → 2,000/1,000 → ROI Align                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  STEP 1: Start with 785,000 anchors                                                │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 2: Keep top 6,000 by FG score                                                │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 3: Apply deltas → 6,000 refined boxes                                        │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 4: Clip to image → 6,000 clipped boxes                                       │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 5: NMS removes overlapping boxes!                                            │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  NMS (IoU > 0.7):                                                              ││
│  │  ┌───────────────────────────────────────────────────────────────────────────┐ ││
│  │  │  6,000 boxes → Remove overlaps → ~2,000 boxes (training)                 │ ││
│  │  │  6,000 boxes → Remove overlaps → ~1,000 boxes (inference)                │ ││
│  │  └───────────────────────────────────────────────────────────────────────────┘ ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│      │                                                                              │
│      ▼                                                                              │
│  STEP 6: Pad to exact count                                                        │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 7: OUTPUT: rpn_rois [batch, 2000/1000, 4]                                   │
│      │                                                                              │
│      ▼                                                                              │
│  STEP 8: ROI Align uses these 2,000/1,000 proposals!                               │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

NEXT ROI ALIGN 

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    CROPPING FROM P2 vs P5 (SAME OBJECT)                            │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  OBJECT: 20x20 pixels in original image                                            │
│                                                                                     │
│  CROP FROM P2 (256x256):                                                           │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  ┌─────────┐                                                                   ││
│  │  │ ████████ │                                                                   ││
│  │  │ ████████ │  ← 5x5 pixels in P2                                              ││
│  │  │ ████████ │  ← CLEAR, DETAILED features                                      ││
│  │  │ ████████ │  ← Can see edges, textures, patterns                             ││
│  │  └─────────┘                                                                   ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                               │
│                                    │ Resize to 7x7                                │
│                                    ▼                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  ┌─────────┐                                                                   ││
│  │  │ ████████ │                                                                   ││
│  │  │ ████████ │  ← 7x7 pixels (upsampled from 5x5)                              ││
│  │  │ ████████ │  ← STILL CLEAR and DETAILED!                                    ││
│  │  │ ████████ │  ← GOOD for detection!                                          ││
│  │  └─────────┘                                                                   ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  VS                                                                                 │
│                                                                                     │
│  CROP FROM P5 (32x32):                                                             │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  ┌─┐                                                                           ││
│  │  │█│  ← 0.625x0.625 pixels in P5 (sub-pixel!)                                ││
│  │  └─┘  ← BLURRY, NO DETAIL                                                     ││
│  │      ← Object is just a blurry blob                                          ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                               │
│                                    │ Resize to 7x7                                │
│                                    ▼                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  ┌─────────┐                                                                   ││
│  │  │ ░░░░░░░░ │                                                                   ││
│  │  │ ░░░██░░░ │  ← 7x7 pixels (upsampled from 0.625x0.625)                      ││
│  │  │ ░░░██░░░ │  ← BLURRY, NO DETAIL!                                           ││
│  │  │ ░░░░░░░░ │  ← BAD for detection!                                           ││
│  │  └─────────┘                                                                   ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

Feature Maps Lose Resolution as You Go Deeper:
text
Original Image: 1024x1024
    ↓ Conv + Pooling
P2: 256x256 (1/4 resolution)  ← HIGH DETAIL
    ↓ Conv + Pooling
P3: 128x128 (1/8 resolution)  ← MEDIUM DETAIL
    ↓ Conv + Pooling
P4: 64x64 (1/16 resolution)   ← LOW DETAIL
    ↓ Conv + Pooling
P5: 32x32 (1/32 resolution)   ← VERY LOW DETAIL

Object Size	Best Level	Why
Small (20x20)	P2 (high res)	Has DETAILED features, object is visible
Medium (100x100)	P3	Balanced detail and context
Large (300x300)	P5 (low res)	Captures WHOLE object, semantic features

What is PyramidROIAlign?
PyramidROIAlign is the layer that takes 2,000 proposals (different sizes) and extracts fixed-size feature maps (7x7 or 14x14) from the right FPN level (P2-P5).



┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                   PYRAMIDROIALIGN - COMPLETE ARCHITECTURE                                        │
├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                                                     │
│  INPUTS:                                                                                                            │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │                                                                                                                 ││
│  │  boxes: [batch, 2000, (y1,x1,y2,x2)]    ← rpn_rois from ProposalLayer                                         ││
│  │  image_meta: [batch, meta_size]         ← Image metadata                                                       ││
│  │  P2: [batch, 256, 256, 256]             ← FPN feature map (highest resolution)                                 ││
│  │  P3: [batch, 128, 128, 256]             ← FPN feature map (medium resolution)                                  ││
│  │  P4: [batch, 64, 64, 256]               ← FPN feature map (low resolution)                                     ││
│  │  P5: [batch, 32, 32, 256]               ← FPN feature map (lowest resolution)                                  ││
│  │                                                                                                                 ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                                                     │
│  STEP 1: CALCULATE AREA FOR EACH PROPOSAL                                                                          │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │                                                                                                                 ││
│  │  For each box:                                                                                                  ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  h = y2 - y1                                                                                               │││
│  │  │  w = x2 - x1                                                                                               │││
│  │  │  area = h * w                                                                                              │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  │  Example:                                                                                                       ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  Proposal 1: 0.05 * 0.05 = 0.0025 (small)                                                                  │││
│  │  │  Proposal 2: 0.20 * 0.20 = 0.04   (medium)                                                                 │││
│  │  │  Proposal 3: 0.40 * 0.30 = 0.12   (large)                                                                  │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                                                     │
│  STEP 2: ASSIGN TO FPN LEVEL                                                                                       │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │                                                                                                                 ││
│  │  roi_level = 4 + log2(sqrt(area) / (224 / sqrt(image_area)))                                                  ││
│  │                                                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  Proposal 1 (small):  roi_level = 2  → ASSIGN TO P2                                                         │││
│  │  │  Proposal 2 (medium): roi_level = 3  → ASSIGN TO P3                                                         │││
│  │  │  Proposal 3 (large):  roi_level = 4  → ASSIGN TO P4                                                         │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                                                     │
│  STEP 3: LOOP THROUGH LEVELS (P2-P5)                                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │                                                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  LEVEL 1: P2 (256x256)                                                                                     │││
│  │  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────┐│││
│  │  │  │  │ Boxes assigned to P2 (small objects)                                                                ││││
│  │  │  │  │ tf.image.crop_and_resize(P2, boxes, batch_indices, [7,7], bilinear)                                ││││
│  │  │  │  │ → [num_boxes_p2, 7, 7, 256]                                                                        ││││
│  │  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────┘│││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  LEVEL 2: P3 (128x128)                                                                                     │││
│  │  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────┐│││
│  │  │  │  │ Boxes assigned to P3 (medium objects)                                                               ││││
│  │  │  │  │ tf.image.crop_and_resize(P3, boxes, batch_indices, [7,7], bilinear)                                ││││
│  │  │  │  │ → [num_boxes_p3, 7, 7, 256]                                                                        ││││
│  │  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────┘│││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  LEVEL 3: P4 (64x64)                                                                                       │││
│  │  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────┐│││
│  │  │  │  │ Boxes assigned to P4 (large objects)                                                                ││││
│  │  │  │  │ tf.image.crop_and_resize(P4, boxes, batch_indices, [7,7], bilinear)                                ││││
│  │  │  │  │ → [num_boxes_p4, 7, 7, 256]                                                                        ││││
│  │  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────┘│││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  LEVEL 4: P5 (32x32)                                                                                       │││
│  │  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────┐│││
│  │  │  │  │ Boxes assigned to P5 (very large objects)                                                           ││││
│  │  │  │  │ tf.image.crop_and_resize(P5, boxes, batch_indices, [7,7], bilinear)                                ││││
│  │  │  │  │ → [num_boxes_p5, 7, 7, 256]                                                                        ││││
│  │  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────┘│││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                                                     │
│  STEP 4: CONCATENATE AND REORDER                                                                                   │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │                                                                                                                 ││
│  │  pooled = concat([pooled_p2, pooled_p3, pooled_p4, pooled_p5])                                                ││
│  │  → [2000, 7, 7, 256]                                                                                          ││
│  │                                                                                                                 ││
│  │  Reorder to match original box order:                                                                         ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  Before reorder: [box_p5, box_p2, box_p3, box_p4, ...]                                                   │││
│  │  │  After reorder:  [box_1, box_2, box_3, box_4, ...] (original order)                                      │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                                 ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                                                     │
│  STEP 5: RE-ADD BATCH DIMENSION                                                                                    │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │                                                                                                                 ││
│  │  OUTPUT: [batch, 2000, 7, 7, 256]                                                                              ││
│  │                                                                                                                 ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘

The shift from RoI Pooling in Faster R-CNN to RoI Align in Mask R-CNN was one of the key innovations that made pixel-perfect instance segmentation possible.

1. What is ROI Pooling? (Faster R-CNN)
ROI Pooling is the original method used in Faster R-CNN to convert proposals of different sizes into fixed-size feature maps.

The Problem It Solves:
Proposals come in different sizes (50x50, 100x200, etc.)

The classifier needs fixed-size inputs (7x7)

ROI Pooling solves this by cropping and resizing

The Two Quantization Problems:
Problem 1: Mapping to Feature Map

python
# Original proposal: [100, 50, 300, 250]
# Stride = 16

# Without quantization (ideal):
y1 = 100/16 = 6.25
x1 = 50/16 = 3.125
y2 = 300/16 = 18.75
x2 = 250/16 = 15.625

# With quantization (ROI Pooling):
y1 = 6 (rounded down)  ❌ Lost 0.25 = 4 pixels!
x1 = 3 (rounded down)  ❌ Lost 0.125 = 2 pixels!
y2 = 19 (rounded up)   ❌ Added 0.25 = 4 pixels!
x2 = 16 (rounded up)   ❌ Added 0.375 = 6 pixels!
Problem 2: Dividing into Bins

python
# Feature map region: 13x13 (after rounding)
# Desired output: 7x7

# Without quantization:
bin_size = 13/7 = 1.857

# With quantization (ROI Pooling):
bin_size = 2 (rounded up)  ❌ Each bin is too big!
# Some bins get 2 pixels, some get 1 pixel
# Creates uneven sampling!
2. What is ROI Align? (Mask R-CNN)
ROI Align is the improved method used in Mask R-CNN that avoids quantization and uses bilinear interpolation.

How ROI Align Works:
Bilinear Interpolation Example:
python
# Point: (6.25, 3.125) on feature map
# Surrounding integer pixels: (6,3), (7,3), (6,4), (7,4)

# Values at integer pixels:
v00 = feature_map[6, 3] = 0.8
v10 = feature_map[7, 3] = 0.9
v01 = feature_map[6, 4] = 0.7
v11 = feature_map[7, 4] = 0.6

# Weights (based on distance to integer pixels):
dx = 0.25  # Distance from x=6
dy = 0.125 # Distance from y=3

# Bilinear interpolation:
v = (1-dy) * ((1-dx)*v00 + dx*v10) + dy * ((1-dx)*v01 + dx*v11)
v = 0.875 * (0.75*0.8 + 0.25*0.9) + 0.125 * (0.75*0.7 + 0.25*0.6)
v = 0.875 * (0.6 + 0.225) + 0.125 * (0.525 + 0.15)
v = 0.875 * 0.825 + 0.125 * 0.675
v = 0.722 + 0.084
v = 0.806

# Perfect value at fractional location! ✅
3. ROI Pooling vs ROI Align - Key Differences
Aspect	ROI Pooling (Faster R-CNN)	ROI Align (Mask R-CNN)
Quantization	❌ Yes (rounds coordinates)	✅ No (keeps floats)
Interpolation	❌ No (uses nearest neighbor)	✅ Yes (bilinear)
Accuracy	❌ Misaligned by 1-2 pixels	✅ Perfect alignment
Mask Quality	❌ Poor (can't segment)	✅ Excellent (pixel-perfect)
Speed	✅ Faster (simpler)	⚠️ Slightly slower
Memory	✅ Less	⚠️ More


ROI Align replaced ROI Pooling's harsh quantization with precise bilinear interpolation, eliminating the 1-2 pixel misalignment that made accurate instance segmentation impossible, and transforming object detection into pixel-perfect instance segmentation. 🎯
In FPN-based Faster/Mask R-CNN, each proposal is assigned to a single pyramid level and converted to a fixed-size feature map (e.g., 7×7) before the classification head. ROI Pooling performs coordinate quantization by rounding ROI boundaries and sampling locations, causing misalignment. ROI Align removes this coordinate quantization and uses bilinear interpolation to obtain feature values at exact floating-point locations, producing more accurate features for detection and segmentation.



A common misunderstanding is to think that ROI Align uses interpolation because it is converting a feature map such as 14×14 into a fixed 7×7 output, similar to how OpenCV resizes an image from 5×5 to 7×7. However, that is not what ROI Align is doing. Instead, ROI Align first divides the ROI into a fixed number of bins (for example, 7×7 bins) and then samples points within each bin. These sampling locations are often fractional coordinates such as (3.7, 5.2). Since feature maps only contain values at integer locations like (3,5), (3,6), (4,5), and (4,6), ROI Align uses bilinear interpolation to estimate the feature value at the exact fractional location (3.7, 5.2). Therefore, interpolation is not being used to resize 14×14 into 7×7; it is being used to obtain feature values at fractional coordinates. In contrast, ROI Pooling would round the coordinates, for example converting 3.7→4 and 5.2→5, before pooling. This rounding is called coordinate quantization and introduces localization errors. ROI Align avoids this error by keeping the fractional coordinates unchanged and using bilinear interpolation to compute the corresponding feature values, resulting in better alignment between the ROI and the extracted features.


Interpolation is NOT converting the coordinate into a whole number.

It is converting a fractional position into a feature value.

For example:

Coordinate = (3.7, 5.2)

ROI Align keeps this coordinate exactly as it is.

The problem is that the feature map only stores:

(3,5)=10
(3,6)=20
(4,5)=30
(4,6)=40

There is no stored value at (3.7, 5.2).

So interpolation asks:

"Given these four nearby values, what should the value be at position (3.7, 5.2)?"

Maybe it computes:

Value at (3.7, 5.2) ≈ 29

Notice:

Coordinate stays:
(3.7, 5.2)

Feature value becomes:
29

So ROI Align is not turning 3.7 into 4.

ROI Pooling does:

3.7 → 4

ROI Align does:

3.7 stays 3.7

and estimates the feature value at that location using interpolation.

That's the subtle but very important distinction. The interpolation is happening on the feature values, not on the coordinates

In [ ]:
#Takes input from ProposalLayer as 2000 boxes and FPN features, then resizes them by first assigning them to their FPN level, then resizes to 7x7, and that is the output for every 2000 box coming from ROI layer.

############################################################
#  ROIAlign Layer
############################################################
def compose_image_meta(image_id, original_image_shape, image_shape,
                       window, scale, active_class_ids):
    """Takes attributes of an image and puts them in one 1D array.

    image_id: An int ID of the image. Useful for debugging.
    original_image_shape: [H, W, C] before resizing or padding.
    image_shape: [H, W, C] after resizing and padding
    window: (y1, x1, y2, x2) in pixels. The area of the image where the real
            image is (excluding the padding)
    scale: The scaling factor applied to the original image (float32)
    active_class_ids: List of class_ids available in the dataset from which
        the image came. Useful if training on images from multiple datasets
        where not all classes are present in all datasets.
    """
    meta = np.array(
        [image_id] +                  # size=1
        list(original_image_shape) +  # size=3
        list(image_shape) +           # size=3
        list(window) +                # size=4 (y1, x1, y2, x2) in image cooredinates
        [scale] +                     # size=1
        list(active_class_ids)        # size=num_classes
    )
    return meta

def parse_image_meta_graph(meta):
    """Parses a tensor that contains image attributes to its components.
    See compose_image_meta() for more details.

    meta: [batch, meta length] where meta length depends on NUM_CLASSES

    Returns a dict of the parsed tensors.
    """
    image_id = meta[:, 0]
    original_image_shape = meta[:, 1:4]
    image_shape = meta[:, 4:7]
    window = meta[:, 7:11]  # (y1, x1, y2, x2) window of image in in pixels
    scale = meta[:, 11]
    active_class_ids = meta[:, 12:]
    return {
        "image_id": image_id,
        "original_image_shape": original_image_shape,
        "image_shape": image_shape,
        "window": window,
        "scale": scale,
        "active_class_ids": active_class_ids,
    }

def log2_graph(x):
    """Implementation of Log2. TF doesn't have a native implementation."""
    return tf.math.log(x) / tf.math.log(2.0)


class PyramidROIAlign(KL.Layer):
    """Implements ROI Pooling on multiple levels of the feature pyramid.

    Params:
    - pool_shape: [pool_height, pool_width] of the output pooled regions. Usually [7, 7]

    Inputs:
    - boxes: [batch, num_boxes, (y1, x1, y2, x2)] in normalized
             coordinates. Possibly padded with zeros if not enough
             boxes to fill the array.
    - image_meta: [batch, (meta data)] Image details. See compose_image_meta()
    - feature_maps: List of feature maps from different levels of the pyramid.
                    Each is [batch, height, width, channels]

    Output:
    Pooled regions in the shape: [batch, num_boxes, pool_height, pool_width, channels].
    The width and height are those specific in the pool_shape in the layer
    constructor.
    """

    def __init__(self, pool_shape, **kwargs):
        super(PyramidROIAlign, self).__init__(**kwargs)
        self.pool_shape = tuple(pool_shape)

    def get_config(self):
        config = super(PyramidROIAlign, self).get_config()
        config['pool_shape'] = self.pool_shape
        return config

    def call(self, inputs):
        # Crop boxes [batch, num_boxes, (y1, x1, y2, x2)] in normalized coords
        boxes = inputs[0]

        # Image meta
        # Holds details about the image. See compose_image_meta()
        image_meta = inputs[1]

        # Feature Maps. List of feature maps from different level of the
        # feature pyramid. Each is [batch, height, width, channels]
        feature_maps = inputs[2:]

        # Assign each ROI to a level in the pyramid based on the ROI area.
        y1, x1, y2, x2 = tf.split(boxes, 4, axis=2)
        h = y2 - y1
        w = x2 - x1
        # Use shape of first image. Images in a batch must have the same size.
        image_shape = parse_image_meta_graph(image_meta)['image_shape'][0]
        # Equation 1 in the Feature Pyramid Networks paper. Account for
        # the fact that our coordinates are normalized here.
        # e.g. a 224x224 ROI (in pixels) maps to P4
        image_area = tf.cast(image_shape[0] * image_shape[1], tf.float32)
        roi_level = log2_graph(tf.sqrt(h * w) / (224.0 / tf.sqrt(image_area)))
        roi_level = tf.minimum(5, tf.maximum(
            2, 4 + tf.cast(tf.round(roi_level), tf.int32)))
        roi_level = tf.squeeze(roi_level, 2)

        # Loop through levels and apply ROI pooling to each. P2 to P5.
        pooled = []
        box_to_level = []
        for i, level in enumerate(range(2, 6)):
            ix = tf.compat.v1.where(tf.equal(roi_level, level))
            level_boxes = tf.gather_nd(boxes, ix)

            # Box indices for crop_and_resize.
            box_indices = tf.cast(ix[:, 0], tf.int32)

            # Keep track of which box is mapped to which level
            box_to_level.append(ix)

            # Stop gradient propogation to ROI proposals
            level_boxes = tf.stop_gradient(level_boxes)
            box_indices = tf.stop_gradient(box_indices)

            # Crop and Resize
            # From Mask R-CNN paper: "We sample four regular locations, so
            # that we can evaluate either max or average pooling. In fact,
            # interpolating only a single value at each bin center (without
            # pooling) is nearly as effective."
            #
            # Here we use the simplified approach of a single value per bin,
            # which is how it's done in tf.crop_and_resize()
            # Result: [batch * num_boxes, pool_height, pool_width, channels]
            pooled.append(tf.image.crop_and_resize(
                feature_maps[i], level_boxes, box_indices, self.pool_shape,
                method="bilinear"))

        # Pack pooled features into one tensor
        pooled = tf.concat(pooled, axis=0)

        # Pack box_to_level mapping into one array and add another
        # column representing the order of pooled boxes
        box_to_level = tf.concat(box_to_level, axis=0)
        box_range = tf.expand_dims(tf.range(tf.shape(input=box_to_level)[0]), 1)
        box_to_level = tf.concat([tf.cast(box_to_level, tf.int32), box_range],
                                 axis=1)

        # Rearrange pooled features to match the order of the original boxes
        # Sort box_to_level by batch then box index
        # TF doesn't have a way to sort by two columns, so merge them and sort.
        sorting_tensor = box_to_level[:, 0] * 100000 + box_to_level[:, 1]
        ix = tf.nn.top_k(sorting_tensor, k=tf.shape(
            input=box_to_level)[0]).indices[::-1]
        ix = tf.gather(box_to_level[:, 2], ix)
        pooled = tf.gather(pooled, ix)

        # Re-add the batch dimension
        shape = tf.concat([tf.shape(input=boxes)[:2], tf.shape(input=pooled)[1:]], axis=0)
        pooled = tf.reshape(pooled, shape)
        return pooled

    def compute_output_shape(self, input_shape):
        return input_shape[0][:2] + self.pool_shape + (input_shape[2][-1], )

What Comes Out of ROI Align
ALL 2000 proposals come out as the SAME SIZE!

python
ROI Align Output: [batch, 2000, 7, 7, 256]
Dimension	Meaning
batch	Number of images
2000	Number of proposals (ALL the same size!)
7	Height of each feature map
7	Width of each feature map
256	Number of channels

┌─────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                    PHASE 3: TRAINING - HOW GROUND TRUTH IS USED                                    │
├─────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                                     │
│  INPUTS TO MODEL:                                                                                   │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  Image: [batch, H, W, 3]                                                                       ││
│  │  Ground Truth: gt_boxes, gt_class_ids, gt_masks (from dataset)                                 ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 1: RESNET BACKBONE                                                                       ││
│  │  Image → ResNet → C2, C3, C4, C5                                                               ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 2: FPN                                                                                    ││
│  │  C2-C5 → P2, P3, P4, P5, P6                                                                    ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 3: RPN (Region Proposal Network)                                                          ││
│  │                                                                                                 ││
│  │  P2-P6 → RPN                                                                                    ││
│  │  │                                                                                              ││
│  │  ├── rpn_class: [batch, 785k, 2]  (FG/BG scores)                                              ││
│  │  └── rpn_bbox: [batch, 785k, 4]   (Box deltas)                                                ││
│  │                                                                                                 ││
│  │  ★ RPN doesn't use Ground Truth - it just predicts based on image features!                    ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 4: PROPOSAL LAYER                                                                         ││
│  │                                                                                                 ││
│  │  rpn_class + rpn_bbox + anchors → ProposalLayer → rpn_rois: [batch, 2000, 4]                  ││
│  │                                                                                                 ││
│  │  ★ These are the model's "guesses" of where objects might be                                   ││
│  │  ★ No Ground Truth used here!                                                                  ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 5: DETECTION TARGET LAYER ★ GROUND TRUTH IS USED HERE! ★                                 ││
│  │                                                                                                 ││
│  │  Inputs:                                                                                        ││
│  │  ├── rpn_rois (model's guesses)                                                                 ││
│  │  ├── gt_class_ids (ground truth)  ← FROM DATASET!                                              ││
│  │  ├── gt_boxes (ground truth)      ← FROM DATASET!                                              ││
│  │  └── gt_masks (ground truth)      ← FROM DATASET!                                              ││
│  │                                                                                                 ││
│  │  Process:                                                                                       ││
│  │  1. Compute IoU between rpn_rois and gt_boxes                                                  ││
│  │  2. Match proposals to ground truth                                                             ││
│  │  3. Select best 200 proposals                                                                  ││
│  │  4. Create targets:                                                                             ││
│  │     - target_class_ids: "This proposal should predict class = 1"                               ││
│  │     - target_bbox: "This proposal should predict these bbox deltas"                            ││
│  │     - target_mask: "This proposal should predict this mask"                                    ││
│  │                                                                                                 ││
│  │  Outputs:                                                                                       ││
│  │  ├── rois: [batch, 200, 4]          (selected proposals)                                       ││
│  │  ├── target_class_ids: [batch, 200] (answers for learning)                                     ││
│  │  ├── target_bbox: [batch, 200, 4]   (answers for learning)                                     ││
│  │  └── target_mask: [batch, 200, 28, 28] (answers for learning)                                  ││
│  │                                                                                                 ││
│  │  ★ This is where Ground Truth is used to create training targets!                              ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 6: ROI ALIGN                                                                              ││
│  │                                                                                                 ││
│  │  rois + P2-P5 → PyramidROIAlign → ROI features: [batch, 200, 7, 7, 256]                       ││
│  │                                                                                                 ││
│  │  ★ Features extracted from selected proposals (200)                                            ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 7: HEADS (Classifier + Mask)                                                              ││
│  │                                                                                                 ││
│  │  ROI features → Heads → Predictions:                                                            ││
│  │  ├── mrcnn_class_logits: [batch, 200, num_classes] (predicted classes)                         ││
│  │  ├── mrcnn_bbox: [batch, 200, num_classes, 4] (predicted bbox deltas)                         ││
│  │  └── mrcnn_mask: [batch, 200, 28, 28, num_classes] (predicted masks)                           ││
│  │                                                                                                 ││
│  │  ★ Model's predictions based on image features!                                                ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 8: LOSS COMPUTATION ★ COMPARE PREDICTIONS TO GROUND TRUTH ★                              ││
│  │                                                                                                 ││
│  │  Compare predictions to targets (from Step 5):                                                 ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  class_loss = compare(mrcnn_class_logits, target_class_ids)                                 │││
│  │  │  bbox_loss = compare(mrcnn_bbox, target_bbox)                                               │││
│  │  │  mask_loss = compare(mrcnn_mask, target_mask)                                               │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────┘││
│  │                                                                                                 ││
│  │  ★ If predictions are wrong → LOSS is HIGH                                                    ││
│  │  ★ If predictions are correct → LOSS is LOW                                                   ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 9: BACKPROPAGATION                                                                        ││
│  │                                                                                                 ││
│  │  Loss → Backpropagate → Update ALL weights (ResNet, FPN, RPN, Heads)                           ││
│  │                                                                                                 ││
│  │  ★ Model learns to reduce loss → Better predictions!                                           ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────────────────────┘

VERY IMPORTANT 

During training in Mask R-CNN, we start with input images along with their ground-truth (GT) annotations, which include bounding boxes, class labels, and segmentation masks. The image is passed through the ResNet backbone, which extracts feature maps from the entire image; the GT boxes themselves do not go through the network but are kept separately as labels for supervision. These backbone features are then sent to the Feature Pyramid Network (FPN), which generates multi-scale feature maps such as P2, P3, P4, and P5. The Region Proposal Network (RPN) operates on these FPN feature maps and predicts object proposals, which are new candidate bounding boxes and are different from the original GT boxes. During training, these proposals are compared with the GT boxes using IoU to determine which proposals are positive, negative, or ignored samples. After filtering and ranking the proposals, each proposal is assigned to the most appropriate FPN level based on its size and is processed by ROI Align. ROI Align extracts a fixed-size feature representation (such as 7×7 for the detection head) by sampling features at precise floating-point locations using bilinear interpolation, avoiding the coordinate quantization errors of ROI Pooling. These aligned features are then passed to the classification head, bounding-box regression head, and mask head to predict object classes, refined bounding boxes, and segmentation masks. Finally, the predicted outputs are compared with the ground-truth classes, boxes, and masks to compute the losses, and the network weights are updated through backpropagation. During inference, no GT annotations are needed; only the image is provided, and the trained model predicts the boxes, classes, and masks on its own.



Ground-truth (GT) boxes, classes, and masks are essential during training because they act as the correct answers that the model learns from. Although GT annotations do not pass through the ResNet backbone, FPN, RPN, or ROI Align to generate features, they are still used throughout the training process as supervision signals. The image is passed through the network to produce predicted bounding boxes, class labels, and masks, while the GT annotations remain separate. These predictions are then compared against the corresponding GT boxes, classes, and masks to calculate classification, localization, and mask losses. For example, if the model predicts a bounding box that does not correctly overlap the actual object, the GT box tells the model how far off it is. Similarly, GT class labels indicate whether the predicted class is correct, and GT masks show the exact object shape that the predicted mask should match. The computed losses are then backpropagated through the network to update the model weights. Without GT annotations, the model would still produce predictions, but it would have no way of knowing whether those predictions were correct or incorrect, making it impossible to compute a loss and therefore impossible to learn in a supervised training setting. In short, GT annotations do not generate features, but they serve as the answer key that guides the entire learning process.


During training, the model generates predicted boxes, classes, and masks. These predictions are compared against the ground-truth (GT) boxes, classes, and masks using loss functions. The difference between the predictions and the GT annotations produces an error (loss), and backpropagation updates the model weights to reduce that error. Over many training iterations, the predicted boxes gradually become closer to the GT boxes, the predicted classes become more accurate, and the predicted masks better match the GT masks.

What DetectionTargetLayer Does (One Sentence)
DetectionTargetLayer takes the 2,000 raw proposals from RPN, compares them to Ground Truth boxes using IoU, selects the best 200 proposals, and creates "training targets" (class IDs, bbox deltas, masks) that tell the heads what to learn.

IoU = Area of Intersection / Area of Union

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    IoU EXAMPLE                                                      │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  ┌──────────────┐                                                                  │
│  │  Proposal    │                                                                  │
│  │  ┌──────────┐│                                                                  │
│  │  │  GT Box   ││  Intersection = Overlap area                                   │
│  │  └──────────┘│  Union = Total area covered by both boxes                       │
│  └──────────────┘  IoU = Intersection / Union                                     │
│                                                                                     │
│  IoU = 0.85 → Very good match                                                     │
│  IoU = 0.45 → Partial match                                                       │
│  IoU = 0.05 → Poor match                                                          │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

In [ ]:
############################################################
#  Detection Target Layer
############################################################
def norm_boxes_graph(boxes, shape):
    """Converts boxes from pixel coordinates to normalized coordinates.
    boxes: [..., (y1, x1, y2, x2)] in pixel coordinates
    shape: [..., (height, width)] in pixels

    Note: In pixel coordinates (y2, x2) is outside the box. But in normalized
    coordinates it's inside the box.

    Returns:
        [..., (y1, x1, y2, x2)] in normalized coordinates
    """
    h, w = tf.split(tf.cast(shape, tf.float32), 2)
    scale = tf.concat([h, w, h, w], axis=-1) - tf.constant(1.0)
    shift = tf.constant([0., 0., 1., 1.])
    return tf.divide(boxes - shift, scale)

def overlaps_graph(boxes1, boxes2):
    """Computes IoU overlaps between two sets of boxes.
    boxes1, boxes2: [N, (y1, x1, y2, x2)].
    """
    # 1. Tile boxes2 and repeat boxes1. This allows us to compare
    # every boxes1 against every boxes2 without loops.
    # TF doesn't have an equivalent to np.repeat() so simulate it
    # using tf.tile() and tf.reshape.
    #Purpose: Create all possible pairs of boxes1 and boxes2 without using loops.
    b1 = tf.reshape(tf.tile(tf.expand_dims(boxes1, 1),
                            [1, 1, tf.shape(input=boxes2)[0]]), [-1, 4])
    b2 = tf.tile(boxes2, [tf.shape(input=boxes1)[0], 1])
    """boxes1 = [A, B, C]  (3 boxes)
boxes2 = [1, 2]     (2 boxes)
We want: A-1, A-2, B-1, B-2, C-1, C-2  (6 pairs)\
Step-by-step:
1. tf.expand_dims(boxes1, 1):
   boxes1 = [[A], [B], [C]]  (shape: [3, 1, 4])
2. tf.tile(..., [1, 1, 2]):
   boxes1 = [[A, A], [B, B], [C, C]]  (shape: [3, 2, 4])
3. tf.reshape(..., [-1, 4]):
   b1 = [A, A, B, B, C, C]  (shape: [6, 4])
4. tf.tile(boxes2, [3, 1]):
   b2 = [1, 2, 1, 2, 1, 2]  (shape: [6, 4])
Result: b1 and b2 are paired!
b1 = [A, A, B, B, C, C]
b2 = [1, 2, 1, 2, 1, 2]"""
    # 2. Compute intersections
    b1_y1, b1_x1, b1_y2, b1_x2 = tf.split(b1, 4, axis=1)
    b2_y1, b2_x1, b2_y2, b2_x2 = tf.split(b2, 4, axis=1)
    y1 = tf.maximum(b1_y1, b2_y1)
    x1 = tf.maximum(b1_x1, b2_x1)
    y2 = tf.minimum(b1_y2, b2_y2)
    x2 = tf.minimum(b1_x2, b2_x2)
    intersection = tf.maximum(x2 - x1, 0) * tf.maximum(y2 - y1, 0)
    # 3. Compute unions
    b1_area = (b1_y2 - b1_y1) * (b1_x2 - b1_x1)
    b2_area = (b2_y2 - b2_y1) * (b2_x2 - b2_x1)
    union = b1_area + b2_area - intersection
    # 4. Compute IoU and reshape to [boxes1, boxes2]
    iou = intersection / union
    overlaps = tf.reshape(iou, [tf.shape(input=boxes1)[0], tf.shape(input=boxes2)[0]])
    return overlaps

"""┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    INTERSECTION COMPUTATION                                        │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  Box 1: [y1_1, x1_1, y2_1, x2_1]                                                 │
│  Box 2: [y1_2, x1_2, y2_2, x2_2]                                                 │
│                                                                                     │
│  Intersection corners:                                                             │
│  ┌─────────────────────────────────────────────────────────────────────────────────┐│
│  │  y1 = max(y1_1, y1_2)  ← Bottom of the topmost box                            ││
│  │  x1 = max(x1_1, x1_2)  ← Right of the leftmost box                            ││
│  │  y2 = min(y2_1, y2_2)  ← Top of the bottommost box                            ││
│  │  x2 = min(x2_1, x2_2)  ← Left of the rightmost box                            ││
│  └─────────────────────────────────────────────────────────────────────────────────┘│
│                                                                                     │
│  Intersection Area = max(x2 - x1, 0) * max(y2 - y1, 0)                           │
│                                                                                     │
│  If no overlap: x2 - x1 < 0 or y2 - y1 < 0 → Intersection = 0                    │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘

Union = Area(Box1) + Area(Box2) - Intersection                 
(Don't double-count the overlap!) """

def trim_zeros_graph(boxes, name='trim_zeros'):
    """Often boxes are represented with matrices of shape [N, 4] and
    are padded with zeros. This removes zero boxes.

    boxes: [N, 4] matrix of boxes.
    non_zeros: [N] a 1D boolean mask identifying the rows to keep
    """
    non_zeros = tf.cast(tf.reduce_sum(input_tensor=tf.abs(boxes), axis=1), tf.bool)
    boxes = tf.boolean_mask(tensor=boxes, mask=non_zeros, name=name)
    return boxes, non_zeros

def detection_targets_graph(proposals, gt_class_ids, gt_boxes, gt_masks, config):
    """Generates detection targets for one image. Subsamples proposals and
    generates target class IDs, bounding box deltas, and masks for each.

    Inputs:
    proposals: [POST_NMS_ROIS_TRAINING, (y1, x1, y2, x2)] in normalized coordinates. Might
               be zero padded if there are not enough proposals.
    gt_class_ids: [MAX_GT_INSTANCES] int class IDs
    gt_boxes: [MAX_GT_INSTANCES, (y1, x1, y2, x2)] in normalized coordinates.
    gt_masks: [height, width, MAX_GT_INSTANCES] of boolean type.

    Returns: Target ROIs and corresponding class IDs, bounding box shifts,
    and masks.
    rois: [TRAIN_ROIS_PER_IMAGE, (y1, x1, y2, x2)] in normalized coordinates
    class_ids: [TRAIN_ROIS_PER_IMAGE]. Integer class IDs. Zero padded.
    deltas: [TRAIN_ROIS_PER_IMAGE, (dy, dx, log(dh), log(dw))]
    masks: [TRAIN_ROIS_PER_IMAGE, height, width]. Masks cropped to bbox
           boundaries and resized to neural network output size.

    Note: Returned arrays might be zero padded if not enough target ROIs.


    This is the core function inside DetectionTargetLayer that creates training targets for the heads. Let me break it down step by step.

What This Function Does (One Sentence)
Takes 2,000 proposals from RPN and ground truth (boxes, classes, masks), compares them using IoU, selects 200 balanced proposals, 
and creates training targets (class IDs, bbox deltas, masks) for the heads to learn from.
    """
    # Assertions
    asserts = [
        tf.Assert(tf.greater(tf.shape(input=proposals)[0], 0), [proposals],
                  name="roi_assertion"),
    ]
    with tf.control_dependencies(asserts):
        proposals = tf.identity(proposals)

    # Remove zero padding
    #What this does:Ensures there's at least one proposal Removes zero-padded entries from proposals and ground truth
#Why? Different images have different numbers of objects, so arrays are padded with zeros. We need to remove these zeros before processing.
    proposals, _ = trim_zeros_graph(proposals, name="trim_proposals")
    gt_boxes, non_zeros = trim_zeros_graph(gt_boxes, name="trim_gt_boxes")
    gt_class_ids = tf.boolean_mask(tensor=gt_class_ids, mask=non_zeros,
                                   name="trim_gt_class_ids")
    gt_masks = tf.gather(gt_masks, tf.compat.v1.where(non_zeros)[:, 0], axis=2,
                         name="trim_gt_masks")
    
    # Handle COCO crowds
    # A crowd box in COCO is a bounding box around several instances. Exclude
    # them from training. A crowd box is given a negative class ID.
    #What are crowd boxes? In COCO dataset, a crowd box is a bounding box around multiple instances (e.g., a group of people). These are given negative class IDs and are excluded from training.
#Why exclude them? Crowd boxes don't represent a single object, so they can't be used for training.
    crowd_ix = tf.compat.v1.where(gt_class_ids < 0)[:, 0]
    non_crowd_ix = tf.compat.v1.where(gt_class_ids > 0)[:, 0]
    crowd_boxes = tf.gather(gt_boxes, crowd_ix)
    gt_class_ids = tf.gather(gt_class_ids, non_crowd_ix)
    gt_boxes = tf.gather(gt_boxes, non_crowd_ix)
    gt_masks = tf.gather(gt_masks, non_crowd_ix, axis=2)
#     #What this does:

# Computes IoU between all proposals and all GT boxes
# Computes IoU between all proposals and crowd boxes
# Creates a boolean mask for proposals that don't overlap with crowds
# Result: overlaps is a matrix of shape [2000, num_gt] where each cell is the IoU between a proposal and a GT box.
    # Compute overlaps matrix [proposals, gt_boxes]
    overlaps = overlaps_graph(proposals, gt_boxes)

    # Compute overlaps with crowd boxes [proposals, crowd_boxes]
    crowd_overlaps = overlaps_graph(proposals, crowd_boxes)
    crowd_iou_max = tf.reduce_max(input_tensor=crowd_overlaps, axis=1)
    no_crowd_bool = (crowd_iou_max < 0.001)
    # Determine positive and negative ROIs
    roi_iou_max = tf.reduce_max(input_tensor=overlaps, axis=1)
    # 1. Positive ROIs are those with >= 0.5 IoU with a GT box
    positive_roi_bool = (roi_iou_max >= 0.5)
    positive_indices = tf.compat.v1.where(positive_roi_bool)[:, 0]
    # 2. Negative ROIs are those with < 0.5 with every GT box. Skip crowds.
    negative_indices = tf.compat.v1.where(tf.logical_and(roi_iou_max < 0.5, no_crowd_bool))[:, 0]
    # Subsample ROIs. Aim for 33% positive
    # Positive ROIs
    positive_count = int(config.TRAIN_ROIS_PER_IMAGE *
                         config.ROI_POSITIVE_RATIO)
    positive_indices = tf.random.shuffle(positive_indices)[:positive_count]
    positive_count = tf.shape(input=positive_indices)[0]
    # Negative ROIs. Add enough to maintain positive:negative ratio.
    r = 1.0 / config.ROI_POSITIVE_RATIO
    negative_count = tf.cast(r * tf.cast(positive_count, tf.float32), tf.int32) - positive_count
    negative_indices = tf.random.shuffle(negative_indices)[:negative_count]
    # Gather selected ROIs
    positive_rois = tf.gather(proposals, positive_indices)
    negative_rois = tf.gather(proposals, negative_indices)

    # Assign positive ROIs to GT boxes.
    positive_overlaps = tf.gather(overlaps, positive_indices)
    roi_gt_box_assignment = tf.cond(
        pred=tf.greater(tf.shape(input=positive_overlaps)[1], 0),
        true_fn=lambda: tf.argmax(input=positive_overlaps, axis=1),
        false_fn=lambda: tf.cast(tf.constant([]), tf.int64)
    )
    roi_gt_boxes = tf.gather(gt_boxes, roi_gt_box_assignment)
    roi_gt_class_ids = tf.gather(gt_class_ids, roi_gt_box_assignment)

    # Compute bbox refinement for positive ROIs
    deltas = utils.box_refinement_graph(positive_rois, roi_gt_boxes)
    deltas /= config.BBOX_STD_DEV

    # Assign positive ROIs to GT masks
    # Permute masks to [N, height, width, 1]
    transposed_masks = tf.expand_dims(tf.transpose(a=gt_masks, perm=[2, 0, 1]), -1)
    # Pick the right mask for each ROI
    roi_masks = tf.gather(transposed_masks, roi_gt_box_assignment)

    # Compute mask targets
    boxes = positive_rois
    if config.USE_MINI_MASK:
        # Transform ROI coordinates from normalized image space
        # to normalized mini-mask space.
        y1, x1, y2, x2 = tf.split(positive_rois, 4, axis=1)
        gt_y1, gt_x1, gt_y2, gt_x2 = tf.split(roi_gt_boxes, 4, axis=1)
        gt_h = gt_y2 - gt_y1
        gt_w = gt_x2 - gt_x1
        y1 = (y1 - gt_y1) / gt_h
        x1 = (x1 - gt_x1) / gt_w
        y2 = (y2 - gt_y1) / gt_h
        x2 = (x2 - gt_x1) / gt_w
        boxes = tf.concat([y1, x1, y2, x2], 1)
    box_ids = tf.range(0, tf.shape(input=roi_masks)[0])
    masks = tf.image.crop_and_resize(tf.cast(roi_masks, tf.float32), boxes,
                                     box_ids,
                                     config.MASK_SHAPE)
    # Remove the extra dimension from masks.
    masks = tf.squeeze(masks, axis=3)

    # Threshold mask pixels at 0.5 to have GT masks be 0 or 1 to use with
    # binary cross entropy loss.
    masks = tf.round(masks)

    # Append negative ROIs and pad bbox deltas and masks that
    # are not used for negative ROIs with zeros.
    rois = tf.concat([positive_rois, negative_rois], axis=0)
    N = tf.shape(input=negative_rois)[0]
    P = tf.maximum(config.TRAIN_ROIS_PER_IMAGE - tf.shape(input=rois)[0], 0)
    rois = tf.pad(tensor=rois, paddings=[(0, P), (0, 0)])
    roi_gt_boxes = tf.pad(tensor=roi_gt_boxes, paddings=[(0, N + P), (0, 0)])
    roi_gt_class_ids = tf.pad(tensor=roi_gt_class_ids, paddings=[(0, N + P)])
    deltas = tf.pad(tensor=deltas, paddings=[(0, N + P), (0, 0)])
    masks = tf.pad(tensor=masks, paddings=[[0, N + P], (0, 0), (0, 0)])

    return rois, roi_gt_class_ids, deltas, masks

Detection Targets Graph - Complete Flow
INPUT: 2,000 proposals + Ground Truth (boxes, classes, masks)

STEP 1: Remove zero padding from proposals and ground truth.

STEP 2: Handle COCO crowds by excluding negative class IDs (crowd boxes) from training.

STEP 3: Compute IoU between proposals and GT boxes using overlaps_graph().

STEP 4: Classify proposals as Positive (IoU ≥ 0.5), Negative (IoU < 0.5), or Ignored (crowd overlaps).

STEP 5: Subsample to 200 proposals with 33% positive (66 positive, 134 negative).

STEP 6: Generate targets for positive proposals by assigning each to its best GT box, computing bbox deltas, and cropping and resizing GT masks.

STEP 7: Combine positive and negative proposals and pad to exactly 200 proposals, with negative proposals getting class=0, zero deltas, and zero masks.

OUTPUT: rois [200, 4] selected proposals, roi_gt_class_ids [200] class targets (0=background), deltas [200, 4] bbox delta targets, masks [200, 28, 28] mask targets.



In [ ]:
# # This is the Keras Layer wrapper around detection_targets_graph(). It handles batch processing and defines the layer interface for the model.

# # What This Class Does (One Sentence)
# # DetectionTargetLayer is a Keras custom layer that wraps detection_targets_graph() to process a batch of images, applying the target generation function independently to each image in the batch.
# What this does:

# Step	Description
# 1	Extracts the 4 inputs from the layer
# 2	Uses utils.batch_slice() to process each image independently
# 3	For each image, calls detection_targets_graph()
# 4	Returns the combined outputs for the batch
# Why batch_slice? It applies the function to each image in the batch separately, handling multi-GPU training.

class DetectionTargetLayer(KL.Layer):
    """Subsamples proposals and generates target box refinement, class_ids,
    and masks for each.

    Inputs:
    proposals: [batch, N, (y1, x1, y2, x2)] in normalized coordinates. Might
               be zero padded if there are not enough proposals.
    gt_class_ids: [batch, MAX_GT_INSTANCES] Integer class IDs.
    gt_boxes: [batch, MAX_GT_INSTANCES, (y1, x1, y2, x2)] in normalized
              coordinates.
    gt_masks: [batch, height, width, MAX_GT_INSTANCES] of boolean type

    Returns: Target ROIs and corresponding class IDs, bounding box shifts,
    and masks.
    rois: [batch, TRAIN_ROIS_PER_IMAGE, (y1, x1, y2, x2)] in normalized
          coordinates
    target_class_ids: [batch, TRAIN_ROIS_PER_IMAGE]. Integer class IDs.
    target_deltas: [batch, TRAIN_ROIS_PER_IMAGE, (dy, dx, log(dh), log(dw)]
    target_mask: [batch, TRAIN_ROIS_PER_IMAGE, height, width]
                 Masks cropped to bbox boundaries and resized to neural
                 network output size.

    Note: Returned arrays might be zero padded if not enough target ROIs.
    This is the Keras Layer wrapper around detection_targets_graph(). It handles batch processing and defines the layer interface for the model.

What This Class Does (One Sentence)
DetectionTargetLayer is a Keras custom layer that wraps detection_targets_graph() to process a batch of images, applying the target generation function independently to each image in the batch.


    """

    def __init__(self, config, **kwargs):
        super(DetectionTargetLayer, self).__init__(**kwargs)
        self.config = config

    def get_config(self):
        config = super(DetectionTargetLayer, self).get_config()
        config["config"] = self.config.to_dict()
        return config

    def call(self, inputs):
        proposals = inputs[0]
        gt_class_ids = inputs[1]
        gt_boxes = inputs[2]
        gt_masks = inputs[3]

        # Slice the batch and run a graph for each slice
        # TODO: Rename target_bbox to target_deltas for clarity
        names = ["rois", "target_class_ids", "target_bbox", "target_mask"]
        outputs = utils.batch_slice(
            [proposals, gt_class_ids, gt_boxes, gt_masks],
            lambda w, x, y, z: detection_targets_graph(
                w, x, y, z, self.config),
            self.config.IMAGES_PER_GPU, names=names)
        return outputs

    def compute_output_shape(self, input_shape):
        return [
            (None, self.config.TRAIN_ROIS_PER_IMAGE, 4),  # rois
            (None, self.config.TRAIN_ROIS_PER_IMAGE),  # class_ids
            (None, self.config.TRAIN_ROIS_PER_IMAGE, 4),  # deltas
            (None, self.config.TRAIN_ROIS_PER_IMAGE, self.config.MASK_SHAPE[0],
             self.config.MASK_SHAPE[1])  # masks
        ]

    def compute_mask(self, inputs, mask=None):
        return [None, None, None, None]





DetectionTargetLayer Batch Output

BATCH SIZE = 2 (2 images processed together)

rois: [batch, 200, 4]
rois[0]: [200, 4] Proposals selected from Image 1 (200 boxes)
rois[1]: [200, 4] Proposals selected from Image 2 (200 boxes)
Each proposal: [y1, x1, y2, x2] in normalized coordinates [0, 1]

target_class_ids: [batch, 200]
target_class_ids[0]: [200] Class targets for Image 1
target_class_ids[1]: [200] Class targets for Image 2
Values: 0 = Background (negative proposals), 1+ = Object class IDs (positive proposals)

target_bbox: [batch, 200, 4]
target_bbox[0]: [200, 4] BBox delta targets for Image 1
target_bbox[1]: [200, 4] BBox delta targets for Image 2
Each target: [dy, dx, log(dh), log(dw)] (bbox refinements)
Negative proposals get [0, 0, 0, 0]

target_mask: [batch, 200, 28, 28]
target_mask[0]: [200, 28, 28] Mask targets for Image 1
target_mask[1]: [200, 28, 28] Mask targets for Image 2
Each mask: [28, 28] binary mask (0 or 1)
Negative proposals get all zeros



fpn_classifier_graph and build_fpn_mask_graph - Complete Explanation
Where They Fit in the Pipeline
After DetectionTargetLayer outputs rois (200 proposals) and targets, these two functions take the rois and extract features using ROI Align, then predict class labels, refined boxes, and segmentation masks.

fpn_classifier_graph (Classifier + BBox Head)
Input
Input	Shape	Description
rois	[batch, 200, 4]	Selected proposals from DetectionTargetLayer
feature_maps	[P2, P3, P4, P5]	FPN features at different scales
image_meta	[batch, meta_size]	Image metadata
pool_size	7	Output size for ROI Align
num_classes	81 (COCO)	Number of classes
fc_layers_size	1024	Size of FC layers
What It Does (Step by Step)
ROI Align (7x7): Extracts fixed-size features for each proposal from the appropriate FPN level. Output: [batch, 200, 7, 7, 256]

Two FC Layers: Passes features through two fully connected layers (implemented as Conv2D for efficiency). Output: [batch, 200, 1024]

Flatten (Squeeze): Removes spatial dimensions. Output: [batch, 200, 1024]

Classifier Head: Predicts class probabilities using Dense layer + Softmax. Output: [batch, 200, num_classes]

BBox Head: Predicts class-specific bounding box deltas. Output: [batch, 200, num_classes, 4]

Outputs
Output	Shape	Description
mrcnn_class_logits	[batch, 200, num_classes]	Class scores (before softmax)
mrcnn_probs	[batch, 200, num_classes]	Class probabilities (after softmax)
mrcnn_bbox	[batch, 200, num_classes, 4]	Class-specific bbox deltas
Contribution to Final Project
Determines what each object is (class prediction)

Refines the bounding boxes to better fit the object

build_fpn_mask_graph (Mask Head)
Input
Input	Shape	Description
rois	[batch, 200, 4]	Selected proposals from DetectionTargetLayer
feature_maps	[P2, P3, P4, P5]	FPN features at different scales
image_meta	[batch, meta_size]	Image metadata
pool_size	14	Output size for ROI Align (larger for masks)
num_classes	81 (COCO)	Number of classes
What It Does (Step by Step)
ROI Align (14x14): Extracts features for each proposal (larger than classifier for detail). Output: [batch, 200, 14, 14, 256]

4 Conv Layers: Each layer is Conv3x3 (256 filters) + BN + ReLU. Output: [batch, 200, 14, 14, 256]

Deconvolution: Upsamples from 14x14 to 28x28 using Conv2DTranspose. Output: [batch, 200, 28, 28, 256]

Final Conv: 1x1 convolution with sigmoid activation to predict per-class masks. Output: [batch, 200, 28, 28, num_classes]

Output
Output	Shape	Description
mrcnn_mask	[batch, 200, 28, 28, num_classes]	Per-class segmentation masks (sigmoid probabilities)

INPUT TO BOTH HEADS:
rois: [batch, 200, 4]  ← From DetectionTargetLayer
P2, P3, P4, P5: FPN features

fpn_classifier_graph()                          build_fpn_mask_graph()
│                                               │
│  Input: rois + P2-P5                          │  Input: rois + P2-P5
│       │                                       │       │
│       ▼                                       │       ▼
│  ROI Align (7x7)                              │  ROI Align (14x14)
│  [batch, 200, 7, 7, 256]                      │  [batch, 200, 14, 14, 256]
│       │                                       │       │
│       ▼                                       │       ▼
│  2x FC Layers (1024)                          │  4x Conv3x3 (256)
│  [batch, 200, 1024]                           │  [batch, 200, 14, 14, 256]
│       │                                       │       │
│       ▼                                       │       ▼
│  Squeeze (Flatten)                            │  Deconv (2x2, stride=2)
│  [batch, 200, 1024]                           │  [batch, 200, 28, 28, 256]
│       │                                       │       │
│       ▼                                       │       ▼
│  ┌───────────────┬───────────────┐           │  Conv1x1 + Sigmoid
│  │               │               │           │  [batch, 200, 28, 28, num_classes]
│  ▼               ▼               │           │
│  Classifier     BBox             │           │
│  Head            Head            │           │
│  Dense(C)       Dense(C*4)       │           │
│  Softmax        Reshape          │           │
│       │               │           │           │
│       ▼               ▼           │           │
│  class_logits    bbox_deltas      │           │
│  [batch,200,C]   [batch,200,C,4] │           │
│                                   │           │
│  OUTPUTS:                         │           │  OUTPUT:
│  mrcnn_class_logits               │           │  mrcnn_mask
│  mrcnn_probs                      │           │  [batch,200,28,28,C]
│  mrcnn_bbox                       │           │
│                                   │           │
└───────────────────────────────────┴───────────┘

These outputs go to Loss Computation:
- class_logits + target_class_ids → Class Loss
- bbox + target_bbox → BBox Loss
- mask + target_mask → Mask Loss



DetectionTargetLayer creates the final 200 proposals WITHOUT using ROI Align (only box/label manipulation), then fpn_classifier_graph() uses ROI Align (7x7) to find the class of each box and refine it, while build_fpn_mask_graph() uses ROI Align (14x14) to find the mask for each box. 🎯

In [ ]:
############################################################
#  Feature Pyramid Network Heads feature map reltaed
############################################################

def fpn_classifier_graph(rois, feature_maps, image_meta,
                         pool_size, num_classes, train_bn=True,
                         fc_layers_size=1024):
    """Builds the computation graph of the feature pyramid network classifier
    and regressor heads.

    rois: [batch, num_rois, (y1, x1, y2, x2)] Proposal boxes in normalized
          coordinates.
    feature_maps: List of feature maps from different layers of the pyramid,
                  [P2, P3, P4, P5]. Each has a different resolution.
    image_meta: [batch, (meta data)] Image details. See compose_image_meta()
    pool_size: The width of the square feature map generated from ROI Pooling.
    num_classes: number of classes, which determines the depth of the results
    train_bn: Boolean. Train or freeze Batch Norm layers
    fc_layers_size: Size of the 2 FC layers

    Returns:
        logits: [batch, num_rois, NUM_CLASSES] classifier logits (before softmax)
        probs: [batch, num_rois, NUM_CLASSES] classifier probabilities
        bbox_deltas: [batch, num_rois, NUM_CLASSES, (dy, dx, log(dh), log(dw))] Deltas to apply to
                     proposal boxes


                     This function builds the Classifier and BBox Regression Head of Mask R-CNN.
                     It takes ROI features and predicts what class each proposal belongs to and how to refine its bounding box.


                     Takes 200 proposals with their ROI features, passes them through two FC layers, then branches into a 
                     classifier head (predicts class probabilities) and a bbox head (predicts class-specific bounding box refinements).
                     Input Parameters
Parameter	Shape	Description
rois	[batch, 200, 4]	Selected proposals from DetectionTargetLayer
feature_maps	[P2, P3, P4, P5]	FPN feature maps (each [batch, H, W, 256])
image_meta	[batch, meta_size]	Image metadata
pool_size	7 (int)	Size of ROI Align output (7x7)
num_classes	81 (COCO)	Number of classes (including background)
train_bn	True/False	Whether to train BatchNorm layers
fc_layers_size	1024	Size of FC layers
    """
    # ROI Pooling
    # Shape: [batch, num_rois, POOL_SIZE, POOL_SIZE, channels] 
    # What it does: Extracts fixed-size features for each proposal from the appropriate FPN level.
    x = PyramidROIAlign([pool_size, pool_size],
                        name="roi_align_classifier")([rois, image_meta] + feature_maps)
    # Two 1024 FC layers (implemented with Conv2D for consistency)
#     What it does: First fully connected layer (implemented as Conv2D for consistency).
# Before	After
# [batch, 200, 7, 7, 256]	[batch, 200, 1, 1, 1024]
# Why Conv2D instead of Dense? TimeDistributed(Conv2D) is more efficient for processing multiple ROIs simultaneously.
# TimeDistributed means: Apply the same layer independently to each of the 200 proposals.
# padding="valid" ensures the output is 1x1 (since kernel size equals input size)
    x = KL.TimeDistributed(KL.Conv2D(fc_layers_size, (pool_size, pool_size), padding="valid"),
                           name="mrcnn_class_conv1")(x)
    x = KL.TimeDistributed(BatchNorm(), name='mrcnn_class_bn1')(x, training=train_bn)
    x = KL.Activation('relu')(x)
#     What it does: Second fully connected layer.
# Before	After
# [batch, 200, 1, 1, 1024]	[batch, 200, 1, 1, 1024]
# Why 1x1 convolution? It's equivalent to a Dense layer applied to each spatial location.
    x = KL.TimeDistributed(KL.Conv2D(fc_layers_size, (1, 1)),
                           name="mrcnn_class_conv2")(x)
    x = KL.TimeDistributed(BatchNorm(), name='mrcnn_class_bn2')(x, training=train_bn)
    x = KL.Activation('relu')(x)
# What it does: Removes the extra spatial dimensions (1x1).
# Before	After
# [batch, 200, 1, 1, 1024]	[batch, 200, 1024]
# K.squeeze(x, 3) removes dimension 3 (width=1).
# K.squeeze(x, 2) removes dimension 2 (height=1).
# Why? Now we have a flat feature vector per proposal, ready for Dense layers.
    shared = KL.Lambda(lambda x: K.squeeze(K.squeeze(x, 3), 2),
                       name="pool_squeeze")(x)

    # Classifier head

# What it does: Predicts class scores and probabilities.
# Layer	Input	Output
# Dense(num_classes)	[batch, 200, 1024]	[batch, 200, num_classes]
# Softmax	[batch, 200, num_classes]	[batch, 200, num_classes]
# mrcnn_class_logits: Raw scores before softmax (for loss computation).
# mrcnn_probs: Probabilities after softmax (for inference).
# Classes include background (class 0).
    mrcnn_class_logits = KL.TimeDistributed(KL.Dense(num_classes),
                                            name='mrcnn_class_logits')(shared)
    mrcnn_probs = KL.TimeDistributed(KL.Activation("softmax"),
                                     name="mrcnn_class")(mrcnn_class_logits)

    # BBox head
    # [batch, num_rois, NUM_CLASSES * (dy, dx, log(dh), log(dw))]
#     What it does: Predicts class-specific bounding box deltas.

# Layer	Input	Output
# Dense(num_classes * 4)	[batch, 200, 1024]	[batch, 200, num_classes * 4]
# Reshape(-1, num_classes, 4)	[batch, 200, num_classes * 4]	[batch, 200, num_classes, 4]
# Why class-specific? Different classes have different shapes. A person box might need different refinement than a car box.

# Each delta: [dy, dx, log(dh), log(dw)] (how to adjust the proposal box).
    """RPN BBox vs MRCNN BBox - What's the Difference?
Aspect	RPN BBox	MRCNN BBox (This Function)
What it predicts	Deltas for anchors	Deltas for proposals
When	Before ProposalLayer	After DetectionTargetLayer
How many	For ALL 785k anchors	For 200 selected proposals
What it refines	Rough anchor boxes	Already refined proposals
Class-specific?	 NO (just object/no object)	 YES (one delta per class)
The Two BBox Predictions in Mask R-CNN
1. RPN BBox (Early Stage)
text
RPN predicts: "How to adjust EVERY anchor to better fit an object"
- For each of 785k anchors → 4 deltas
- Class-agnostic (same deltas for all objects)
- Output: rpn_bbox [batch, 785k, 4]
2. MRCNN BBox (Late Stage) - THIS FUNCTION!
text
MRCNN BBox predicts: "How to refine the selected proposals for EACH class"
- For each of 200 proposals → num_classes * 4 deltas
- Class-specific (different deltas for person vs car)
- Output: mrcnn_bbox [batch, 200, num_classes, 4]
Why Do We Need Both?
RPN BBox: Coarse Refinement
text
RPN: "This anchor has an object! Adjust it roughly."
Anchor: [100, 50, 200, 150] → RPN Deltas → Proposal: [95, 45, 210, 160]"""
    x = KL.TimeDistributed(KL.Dense(num_classes * 4, activation='linear'),
                           name='mrcnn_bbox_fc')(shared)
    # Reshape to [batch, num_rois, NUM_CLASSES, (dy, dx, log(dh), log(dw))]
    s = K.int_shape(x)
    if s[1] is None:
        mrcnn_bbox = KL.Reshape((-1, num_classes, 4), name="mrcnn_bbox")(x)
    else:
        mrcnn_bbox = KL.Reshape((s[1], num_classes, 4), name="mrcnn_bbox")(x)

    return mrcnn_class_logits, mrcnn_probs, mrcnn_bbox


def build_fpn_mask_graph(rois, feature_maps, image_meta,
                         pool_size, num_classes, train_bn=True):
    """Builds the computation graph of the mask head of Feature Pyramid Network.

    rois: [batch, num_rois, (y1, x1, y2, x2)] Proposal boxes in normalized
          coordinates.
    feature_maps: List of feature maps from different layers of the pyramid,
                  [P2, P3, P4, P5]. Each has a different resolution.
    image_meta: [batch, (meta data)] Image details. See compose_image_meta()
    pool_size: The width of the square feature map generated from ROI Pooling.
    num_classes: number of classes, which determines the depth of the results
    train_bn: Boolean. Train or freeze Batch Norm layers

    Returns: Masks [batch, num_rois, MASK_POOL_SIZE, MASK_POOL_SIZE, NUM_CLASSES]


    What This Function Does (One Sentence)
Takes 200 proposals, extracts 14x14 features via ROI Align, passes them through 4 convolutional layers, 
upsamples to 28x28, and predicts a per-class segmentation mask for each proposal using sigmoid activation.
Input Parameters
Parameter	Shape	Description
rois	[batch, 200, 4]	Selected proposals from DetectionTargetLayer
feature_maps	[P2, P3, P4, P5]	FPN feature maps
image_meta	[batch, meta_size]	Image metadata
pool_size	14	Size of ROI Align output (14x14)
num_classes	81 (COCO)	Number of classes
train_bn	True/False	Whether to train BatchNorm
    """
    # ROI Pooling
    # Shape: [batch, num_rois, MASK_POOL_SIZE, MASK_POOL_SIZE, channels]
    x = PyramidROIAlign([pool_size, pool_size],
                        name="roi_align_mask")([rois, image_meta] + feature_maps)

    # Conv layers
#     What it does: First 3x3 convolution with 256 filters.
# Before	After
# [batch, 200, 14, 14, 256]	[batch, 200, 14, 14, 256]
# padding="same" keeps spatial size 14x14.
# TimeDistributed applies the same conv to each of the 200 proposals independently.
    x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                           name="mrcnn_mask_conv1")(x)
    x = KL.TimeDistributed(BatchNorm(),
                           name='mrcnn_mask_bn1')(x, training=train_bn)
    x = KL.Activation('relu')(x)
# Same as Conv1: 3x3 conv, 256 filters, BN, ReLU.
# Before	After
# [batch, 200, 14, 14, 256]	[batch, 200, 14, 14, 256]
    x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                           name="mrcnn_mask_conv2")(x)
    x = KL.TimeDistributed(BatchNorm(),
                           name='mrcnn_mask_bn2')(x, training=train_bn)
    x = KL.Activation('relu')(x)
# Same pattern: 3x3 conv, 256 filters, BN, ReLU.
# Before	After
# [batch, 200, 14, 14, 256]	[batch, 200, 14, 14, 256]
    x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                           name="mrcnn_mask_conv3")(x)
    x = KL.TimeDistributed(BatchNorm(),
                           name='mrcnn_mask_bn3')(x, training=train_bn)
    x = KL.Activation('relu')(x)
   # Same pattern: 3x3 conv, 256 filters, BN, ReLU.

    x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                           name="mrcnn_mask_conv4")(x)
    x = KL.TimeDistributed(BatchNorm(),
                           name='mrcnn_mask_bn4')(x, training=train_bn)
    x = KL.Activation('relu')(x)
    # What it does: Upsamples from 14x14 to 28x28.
# Before	After
# [batch, 200, 14, 14, 256]	[batch, 200, 28, 28, 256]
# Why upsample? The final mask needs to be 28x28 for better detail.
# Conv2DTranspose is also called "deconvolution" - it learns to upsample features.

    x = KL.TimeDistributed(KL.Conv2DTranspose(256, (2, 2), strides=2, activation="relu"),
                           name="mrcnn_mask_deconv")(x)
    # What it does: Predicts per-class masks using 1x1 convolution + sigmoid.
# Before	After
# [batch, 200, 28, 28, 256]	[batch, 200, 28, 28, num_classes]
# Why 1x1 convolution? It maps 256 channels → num_classes channels (like a Dense layer).
# Why sigmoid? Each pixel is independent binary classification (object vs background).
    x = KL.TimeDistributed(KL.Conv2D(num_classes, (1, 1), strides=1, activation="sigmoid"),
                           name="mrcnn_mask")(x)
    return x
# INPUT:
# rois [batch, 200, 4] + P2-P5 [batch, H, W, 256]

#     ▼
# PyramidROIAlign(14x14)
# [batch, 200, 14, 14, 256]

#     ▼
# Conv1: 3x3, 256, BN, ReLU
# [batch, 200, 14, 14, 256]

#     ▼
# Conv2: 3x3, 256, BN, ReLU
# [batch, 200, 14, 14, 256]

#     ▼
# Conv3: 3x3, 256, BN, ReLU
# [batch, 200, 14, 14, 256]

#     ▼
# Conv4: 3x3, 256, BN, ReLU
# [batch, 200, 14, 14, 256]

#     ▼
# Conv2DTranspose(2x2, stride=2)
# [batch, 200, 28, 28, 256]

#     ▼
# Conv1x1 + Sigmoid
# [batch, 200, 28, 28, num_classes]

# OUTPUT:
# mrcnn_mask [batch, 200, 28, 28, num_classes]




In [ ]:

############################################################
#  Loss Functions
############################################################

def smooth_l1_loss(y_true, y_pred):
    """Implements Smooth-L1 loss.
    y_true and y_pred are typically: [N, 4], but could be any shape.
    """
    diff = K.abs(y_true - y_pred)
    less_than_one = K.cast(K.less(diff, 1.0), "float32")
    loss = (less_than_one * 0.5 * diff**2) + (1 - less_than_one) * (diff - 0.5)
    return loss
"""smooth_l1_loss() - Base Loss Function
What It Does
Smooth L1 loss is a hybrid between L1 and L2 loss. It's less sensitive to outliers than L2 loss.

Formula
text
If |diff| < 1: loss = 0.5 * diff²
If |diff| >= 1: loss = |diff| - 0.5 Why Smooth L1?
Loss Type	Problem	Smooth L1 Solution
L2 Loss	Too sensitive to outliers	Less sensitive
L1 Loss	Not differentiable at 0	Differentiable everywhere
Where It's Used
RPN BBox Loss - for anchor box refinement

MRCNN BBox Loss - for proposal box refinement"""

def rpn_class_loss_graph(rpn_match, rpn_class_logits):
    """RPN anchor classifier loss.

    rpn_match: [batch, anchors, 1]. Anchor match type. 1=positive,
               -1=negative, 0=neutral anchor.
    rpn_class_logits: [batch, anchors, 2]. RPN classifier logits for BG/FG.
    """
    # Squeeze last dim to simplify
    rpn_match = tf.squeeze(rpn_match, -1)
    # Get anchor classes. Convert the -1/+1 match to 0/1 values.
    anchor_class = K.cast(K.equal(rpn_match, 1), tf.int32)
    # Positive and Negative anchors contribute to the loss,
    # but neutral anchors (match value = 0) don't.
    indices = tf.compat.v1.where(K.not_equal(rpn_match, 0))
    # Pick rows that contribute to the loss and filter out the rest.
    rpn_class_logits = tf.gather_nd(rpn_class_logits, indices)
    anchor_class = tf.gather_nd(anchor_class, indices)
    # Cross entropy loss
    loss = K.sparse_categorical_crossentropy(target=anchor_class,
                                             output=rpn_class_logits,
                                             from_logits=True)
    loss = K.switch(tf.size(input=loss) > 0, K.mean(loss), tf.constant(0.0))
    return loss
"""RPN Classification Loss
What It Does
Teaches RPN which anchors contain objects (FG) and which are background (BG).

Inputs
Input	Shape	Description
rpn_match	[batch, anchors, 1]	1=positive, -1=negative, 0=neutral
rpn_class_logits	[batch, anchors, 2]	BG/FG logits from RPN. and Anchor Classification:
┌─────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                                                     │
│  Anchor 1: IoU=0.85 → rpn_match=1 (positive) → Should predict FG (class=1)                       │
│  Anchor 2: IoU=0.15 → rpn_match=-1 (negative) → Should predict BG (class=0)                      │
│  Anchor 3: IoU=0.50 → rpn_match=0 (neutral) → IGNORED (no contribution)                          │
│                                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────────────────────┘"""
def batch_pack_graph(x, counts, num_rows):
    """Picks different number of values from each row
    in x depending on the values in counts.
    """
    outputs = []
    for i in range(num_rows):
        outputs.append(x[i, :counts[i]])
    return tf.concat(outputs, axis=0)
def rpn_bbox_loss_graph(config, target_bbox, rpn_match, rpn_bbox):
    """Return the RPN bounding box loss graph.

    config: the model config object.
    target_bbox: [batch, max positive anchors, (dy, dx, log(dh), log(dw))].
        Uses 0 padding to fill in unsed bbox deltas.
    rpn_match: [batch, anchors, 1]. Anchor match type. 1=positive,
               -1=negative, 0=neutral anchor.
    rpn_bbox: [batch, anchors, (dy, dx, log(dh), log(dw))]
    """
    # Positive anchors contribute to the loss, but negative and
    # neutral anchors (match value of 0 or -1) don't.
    rpn_match = K.squeeze(rpn_match, -1)
    indices = tf.compat.v1.where(K.equal(rpn_match, 1))

    # Pick bbox deltas that contribute to the loss
    rpn_bbox = tf.gather_nd(rpn_bbox, indices)

    # Trim target bounding box deltas to the same length as rpn_bbox.
    batch_counts = K.sum(K.cast(K.equal(rpn_match, 1), tf.int32), axis=1)
    target_bbox = batch_pack_graph(target_bbox, batch_counts,
                                   config.IMAGES_PER_GPU)

    loss = smooth_l1_loss(target_bbox, rpn_bbox)

    loss = K.switch(tf.size(input=loss) > 0, K.mean(loss), tf.constant(0.0))
    return loss
"""RPN BBox Loss
What It Does
Teaches RPN to refine anchor boxes to better fit objects.

Inputs
Input	Shape	Description
target_bbox	[batch, max_pos, 4]	Target deltas for positive anchors
rpn_match	[batch, anchors, 1]	1=positive, -1=negative, 0=neutral
rpn_bbox	[batch, anchors, 4]	Predicted deltas from RPN. Visual
text
Positive Anchor 1: target=[0.1, -0.05, 0.2, 0.15], pred=[0.08, -0.04, 0.18, 0.12]
→ Smooth L1 loss: small (good refinement)

Positive Anchor 2: target=[-0.2, 0.3, 0.1, -0.1], pred=[0.5, -0.2, 0.8, 0.6]
→ Smooth L1 loss: large (bad refinement)"""

def mrcnn_class_loss_graph(target_class_ids, pred_class_logits,
                           active_class_ids):
    """Loss for the classifier head of Mask RCNN.

    target_class_ids: [batch, num_rois]. Integer class IDs. Uses zero
        padding to fill in the array.
    pred_class_logits: [batch, num_rois, num_classes]
    active_class_ids: [batch, num_classes]. Has a value of 1 for
        classes that are in the dataset of the image, and 0
        for classes that are not in the dataset.
    """
    # During model building, Keras calls this function with
    # target_class_ids of type float32. Unclear why. Cast it
    # to int to get around it.
    target_class_ids = tf.cast(target_class_ids, 'int64')

    # Find predictions of classes that are not in the dataset.
    pred_class_ids = tf.argmax(input=pred_class_logits, axis=2)
    # TODO: Update this line to work with batch > 1. Right now it assumes all
    #       images in a batch have the same active_class_ids
    pred_active = tf.gather(active_class_ids[0], pred_class_ids)

    # Loss
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=target_class_ids, logits=pred_class_logits)

    # Erase losses of predictions of classes that are not in the active
    # classes of the image.
    loss = loss * pred_active

    # Computer loss mean. Use only predictions that contribute
    # to the loss to get a correct mean.
    loss = tf.reduce_sum(input_tensor=loss) / tf.reduce_sum(input_tensor=pred_active)
    return loss

""" mrcnn_class_loss_graph() - MRCNN Classification Loss
What It Does
Teaches the classifier head to predict the correct class for each proposal.

Inputs
Input	Shape	Description
target_class_ids	[batch, 200]	Ground truth class IDs
pred_class_logits	[batch, 200, num_classes]	Predicted class logits
active_class_ids	[batch, num_classes]	Which classes are in dataset
Visual
text
Proposal 1: target_class=1 (person), pred=[0.1, 0.85, 0.05, ...] → Low loss
Proposal 2: target_class=3 (car), pred=[0.8, 0.1, 0.05, ...] → High loss (wrong)
Proposal 3: target_class=0 (background), pred=[0.9, 0.05, 0.05, ...] → Low loss"""
def mrcnn_bbox_loss_graph(target_bbox, target_class_ids, pred_bbox):
    """Loss for Mask R-CNN bounding box refinement.

    target_bbox: [batch, num_rois, (dy, dx, log(dh), log(dw))]
    target_class_ids: [batch, num_rois]. Integer class IDs.
    pred_bbox: [batch, num_rois, num_classes, (dy, dx, log(dh), log(dw))]
    """
    # Reshape to merge batch and roi dimensions for simplicity.
    target_class_ids = K.reshape(target_class_ids, (-1,))
    target_bbox = K.reshape(target_bbox, (-1, 4))
    pred_bbox = K.reshape(pred_bbox, (-1, K.int_shape(pred_bbox)[2], 4))

    # Only positive ROIs contribute to the loss. And only
    # the right class_id of each ROI. Get their indices.
    positive_roi_ix = tf.compat.v1.where(target_class_ids > 0)[:, 0]
    positive_roi_class_ids = tf.cast(
        tf.gather(target_class_ids, positive_roi_ix), tf.int64)
    indices = tf.stack([positive_roi_ix, positive_roi_class_ids], axis=1)

    # Gather the deltas (predicted and true) that contribute to loss
    target_bbox = tf.gather(target_bbox, positive_roi_ix)
    pred_bbox = tf.gather_nd(pred_bbox, indices)

    # Smooth-L1 Loss
    loss = K.switch(tf.size(input=target_bbox) > 0,
                    smooth_l1_loss(y_true=target_bbox, y_pred=pred_bbox),
                    tf.constant(0.0))
    loss = K.mean(loss)
    return loss
"""MRCNN BBox Loss
What It Does
Teaches the BBox head to refine proposals correctly for each class.

Inputs
Input	Shape	Description
target_bbox	[batch, 200, 4]	Target deltas from GT
target_class_ids	[batch, 200]	Ground truth class IDs
pred_bbox	[batch, 200, num_classes, 4]	Predicted class-specific deltas
Visual
text
Proposal 1: target_class=1 (person), target_bbox=[0.1, -0.05, 0.2, 0.15]
           pred_bbox for class=1: [0.08, -0.04, 0.18, 0.12] → Good
           pred_bbox for class=2: [0.5, 0.3, 0.6, 0.4] → IGNORED (wrong class)

Proposal 2: target_class=0 (background) → IGNORED (no bbox loss)"""

def mrcnn_mask_loss_graph(target_masks, target_class_ids, pred_masks):
    """Mask binary cross-entropy loss for the masks head.

    target_masks: [batch, num_rois, height, width].
        A float32 tensor of values 0 or 1. Uses zero padding to fill array.
    target_class_ids: [batch, num_rois]. Integer class IDs. Zero padded.
    pred_masks: [batch, proposals, height, width, num_classes] float32 tensor
                with values from 0 to 1.
    """
    # Reshape for simplicity. Merge first two dimensions into one.
    target_class_ids = K.reshape(target_class_ids, (-1,))
    mask_shape = tf.shape(input=target_masks)
    target_masks = K.reshape(target_masks, (-1, mask_shape[2], mask_shape[3]))
    pred_shape = tf.shape(input=pred_masks)
    pred_masks = K.reshape(pred_masks,
                           (-1, pred_shape[2], pred_shape[3], pred_shape[4]))
    # Permute predicted masks to [N, num_classes, height, width]
    pred_masks = tf.transpose(a=pred_masks, perm=[0, 3, 1, 2])

    # Only positive ROIs contribute to the loss. And only
    # the class specific mask of each ROI.
    positive_ix = tf.compat.v1.where(target_class_ids > 0)[:, 0]
    positive_class_ids = tf.cast(
        tf.gather(target_class_ids, positive_ix), tf.int64)
    indices = tf.stack([positive_ix, positive_class_ids], axis=1)

    # Gather the masks (predicted and true) that contribute to loss
    y_true = tf.gather(target_masks, positive_ix)
    y_pred = tf.gather_nd(pred_masks, indices)

    # Compute binary cross entropy. If no positive ROIs, then return 0.
    # shape: [batch, roi, num_classes]
    loss = K.switch(tf.size(input=y_true) > 0,
                    K.binary_crossentropy(target=y_true, output=y_pred),
                    tf.constant(0.0))
    loss = K.mean(loss)
    return loss
"""MRCNN Mask Loss
What It Does
Teaches the mask head to predict accurate binary masks for each object.

Inputs
Input	Shape	Description
target_masks	[batch, 200, 28, 28]	Ground truth masks
target_class_ids	[batch, 200]	Ground truth class IDs
pred_masks	[batch, 200, 28, 28, num_classes]	Predicted masks per class
Visual
text
For one positive proposal (person):
┌─────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  Target Mask (GT)                    Predicted Mask (person)                                       │
│  ┌─────────────────────────┐        ┌─────────────────────────┐                                    │
│  │  ██████░░░░░░░░░░░░░░░░ │        │  ██████░░░░░░░░░░░░░░░░ │                                    │
│  │  ██████░░░░░░░░░░░░░░░░ │        │  ██████░░░░░░░░░░░░░░░░ │                                    │
│  │  ░░░░░░░████████░░░░░░ │        │  ░░░░░░░████████░░░░░░ │                                    │
│  │  ░░░░░░░████████░░░░░░ │        │  ░░░░░░░████████░░░░░░ │                                    │
│  │  ░░░░░░░░░░░░░░░░░░░░░ │        │  ░░░░░░░░░░░░░░░░░░░░░ │                                    │
│  └─────────────────────────┘        └─────────────────────────┘                                    │
│                                                                                                     │
│  Binary Cross Entropy Loss: Compare pixel by pixel                                                 │
│                                                                                                     │
│  Other class masks (car, dog, etc.) are IGNORED                                                    │
└─────────────────────────────────────────────────────────────────────────────────────────────────────┘"""

Summary: All Losses at a Glance
Loss	Function	Type	Only Positive	What It Teaches
RPN Class	rpn_class_loss_graph	Cross Entropy	No (uses all anchors)	Which anchors have objects
RPN BBox	rpn_bbox_loss_graph	Smooth L1	Yes (only positive anchors)	Refine anchor boxes
MRCNN Class	mrcnn_class_loss_graph	Cross Entropy	No (all proposals)	Which class is in proposal
MRCNN BBox	mrcnn_bbox_loss_graph	Smooth L1	Yes (only positive ROIs)	Refine boxes per class
MRCNN Mask	mrcnn_mask_loss_graph	Binary CE	Yes (only positive ROIs)	Predict accurate masks

Complete Breakdown: DetectionLayer
This is the final layer of Mask R-CNN that converts the head predictions into final detections during inference.

What This Layer Does (One Sentence)
Takes class probabilities and bbox deltas from the heads, refines the boxes, filters low-confidence and background detections, applies class-wise NMS, and outputs final detections with boxes, class IDs, and scores.

Inputs
Input	Shape	Description	Source
rois	[batch, proposal_count, 4]	Proposal boxes	RPN (via ProposalLayer)
mrcnn_class	[batch, proposal_count, num_classes]	Class probabilities	fpn_classifier_graph()
mrcnn_bbox	[batch, proposal_count, num_classes, 4]	Class-specific bbox deltas	fpn_classifier_graph()
image_meta	[batch, meta_size]	Image metadata	Input to model
Note: In inference, proposal_count = 1000 (POST_NMS_ROIS_INFERENCE).



so basically its same as detction lyaer we built before just that thi sthis one is for when the model is trained fully and we do prediction using test images


The Key Difference
Aspect	DetectionTargetLayer	DetectionLayer
When used	Training only	Inference only
Input	rpn_rois + Ground Truth (boxes, classes, masks)	rpn_rois + Predictions (class_probs, bbox_deltas)
Purpose	Prepare training targets for the heads	Generate final detections
Needs GT?	 YES	 NO
Output	rois (200) + targets (class_ids, bbox, masks)	Final detections [boxes, class_ids, scores]


Component	Training	Inference
ResNet Backbone	✅Used	✅ Used
FPN	✅ Used	✅ Used
RPN	✅ Used	✅ Used
ProposalLayer	✅ Used	✅ Used
DetectionTargetLayer	✅ USED	❌ NOT USED
Heads (Classifier + Mask)	✅ Used	✅ Used
DetectionLayer	❌ NOT USED	✅ USED
Loss Functions	✅ USED	❌ NOT USED


In [ ]:

############################################################
#  Detection Layer
############################################################
"""Takes class probabilities and bbox deltas from the heads, refines the boxes, filters background and low-confidence detections, applies class-wise NMS,
 and outputs final detections [batch, max_instances, (y1,x1,y2,x2,class_id,score)].
 
 STEP 1: Get best class for each proposal (argmax over class probabilities)
STEP 2: Get class-specific bbox deltas for the predicted class
STEP 3: Refine boxes by applying deltas to proposals
STEP 4: Clip boxes to image boundaries
STEP 5: Filter out background (class=0) and low confidence detections
STEP 6: Apply class-wise Non-Maximum Suppression (NMS) to remove duplicates
STEP 7: Keep top detections and pad to fixed size"""
def refine_detections_graph(rois, probs, deltas, window, config):
    """Refine classified proposals and filter overlaps and return final
    detections.

    Inputs:
        rois: [N, (y1, x1, y2, x2)] in normalized coordinates
        probs: [N, num_classes]. Class probabilities.
        deltas: [N, num_classes, (dy, dx, log(dh), log(dw))]. Class-specific
                bounding box deltas.
        window: (y1, x1, y2, x2) in normalized coordinates. The part of the image
            that contains the image excluding the padding.

    Returns detections shaped: [num_detections, (y1, x1, y2, x2, class_id, score)] where
        coordinates are normalized.
    """
    # Class IDs per ROI
    class_ids = tf.argmax(input=probs, axis=1, output_type=tf.int32)
    # Class probability of the top class of each ROI
    indices = tf.stack([tf.range(probs.shape[0]), class_ids], axis=1)
    class_scores = tf.gather_nd(probs, indices)
    # Class-specific bounding box deltas
    deltas_specific = tf.gather_nd(deltas, indices)
    # Apply bounding box deltas
    # Shape: [boxes, (y1, x1, y2, x2)] in normalized coordinates
    refined_rois = apply_box_deltas_graph(
        rois, deltas_specific * config.BBOX_STD_DEV)
    # Clip boxes to image window
    refined_rois = clip_boxes_graph(refined_rois, window)

    # TODO: Filter out boxes with zero area

    # Filter out background boxes
    keep = tf.compat.v1.where(class_ids > 0)[:, 0]
    # Filter out low confidence boxes
    if config.DETECTION_MIN_CONFIDENCE:
        conf_keep = tf.compat.v1.where(class_scores >= config.DETECTION_MIN_CONFIDENCE)[:, 0]
        keep = tf.sets.intersection(tf.expand_dims(keep, 0),
                                        tf.expand_dims(conf_keep, 0))
        keep = tf.sparse.to_dense(keep)[0]

    # Apply per-class NMS
    # 1. Prepare variables
    pre_nms_class_ids = tf.gather(class_ids, keep)
    pre_nms_scores = tf.gather(class_scores, keep)
    pre_nms_rois = tf.gather(refined_rois,   keep)
    unique_pre_nms_class_ids = tf.unique(pre_nms_class_ids)[0]

    def nms_keep_map(class_id):
        """Apply Non-Maximum Suppression on ROIs of the given class."""
        # Indices of ROIs of the given class
        ixs = tf.compat.v1.where(tf.equal(pre_nms_class_ids, class_id))[:, 0]
        # Apply NMS
        class_keep = tf.image.non_max_suppression(
                tf.gather(pre_nms_rois, ixs),
                tf.gather(pre_nms_scores, ixs),
                max_output_size=config.DETECTION_MAX_INSTANCES,
                iou_threshold=config.DETECTION_NMS_THRESHOLD)
        # Map indices
        class_keep = tf.gather(keep, tf.gather(ixs, class_keep))
        # Pad with -1 so returned tensors have the same shape
        gap = config.DETECTION_MAX_INSTANCES - tf.shape(input=class_keep)[0]
        class_keep = tf.pad(tensor=class_keep, paddings=[(0, gap)],
                            mode='CONSTANT', constant_values=-1)
        # Set shape so map_fn() can infer result shape
        class_keep.set_shape([config.DETECTION_MAX_INSTANCES])
        return class_keep

    # 2. Map over class IDs
    nms_keep = tf.map_fn(nms_keep_map, unique_pre_nms_class_ids,
                         dtype=tf.int64)
    # 3. Merge results into one list, and remove -1 padding
    nms_keep = tf.reshape(nms_keep, [-1])
    nms_keep = tf.gather(nms_keep, tf.compat.v1.where(nms_keep > -1)[:, 0])
    # 4. Compute intersection between keep and nms_keep
    keep = tf.sets.intersection(tf.expand_dims(keep, 0),
                                    tf.expand_dims(nms_keep, 0))
    keep = tf.sparse.to_dense(keep)[0]
    # Keep top detections
    roi_count = config.DETECTION_MAX_INSTANCES
    class_scores_keep = tf.gather(class_scores, keep)
    num_keep = tf.minimum(tf.shape(input=class_scores_keep)[0], roi_count)
    top_ids = tf.nn.top_k(class_scores_keep, k=num_keep, sorted=True)[1]
    keep = tf.gather(keep, top_ids)

    # Arrange output as [N, (y1, x1, y2, x2, class_id, score)]
    # Coordinates are normalized.
    detections = tf.concat([
        tf.gather(refined_rois, keep),
        tf.dtypes.cast(tf.gather(class_ids, keep), tf.float32)[..., tf.newaxis],
        tf.gather(class_scores, keep)[..., tf.newaxis]
        ], axis=1)

    # Pad with zeros if detections < DETECTION_MAX_INSTANCES
    gap = config.DETECTION_MAX_INSTANCES - tf.shape(input=detections)[0]
    detections = tf.pad(tensor=detections, paddings=[(0, gap), (0, 0)], mode="CONSTANT")
    return detections


class DetectionLayer(KL.Layer):
    """Takes classified proposal boxes and their bounding box deltas and
    returns the final detection boxes.

    Returns:
    [batch, num_detections, (y1, x1, y2, x2, class_id, class_score)] where
    coordinates are normalized.
    """

    def __init__(self, config=None, **kwargs):
        super(DetectionLayer, self).__init__(**kwargs)
        self.config = config

    def get_config(self):
        config = super(DetectionLayer, self).get_config()
        config["config"] = self.config.to_dict()
        return config

    def call(self, inputs):
        rois = inputs[0]
        mrcnn_class = inputs[1]
        mrcnn_bbox = inputs[2]
        image_meta = inputs[3]

        # Get windows of images in normalized coordinates. Windows are the area
        # in the image that excludes the padding.
        # Use the shape of the first image in the batch to normalize the window
        # because we know that all images get resized to the same size.
        m = parse_image_meta_graph(image_meta)
        image_shape = m['image_shape'][0]
        window = norm_boxes_graph(m['window'], image_shape[:2])

        # Run detection refinement graph on each item in the batch
        detections_batch = utils.batch_slice(
            [rois, mrcnn_class, mrcnn_bbox, window],
            lambda x, y, w, z: refine_detections_graph(x, y, w, z, self.config),
            self.config.IMAGES_PER_GPU)

        # Reshape output
        # [batch, num_detections, (y1, x1, y2, x2, class_id, class_score)] in
        # normalized coordinates
        return tf.reshape(
            detections_batch,
            [self.config.BATCH_SIZE, self.config.DETECTION_MAX_INSTANCES, 6])

    def compute_output_shape(self, input_shape):
        return (None, self.config.DETECTION_MAX_INSTANCES, 6)


In [ ]:
############################################################
#  Data Generator
############################################################
def mold_image(images, config):
    """Expects an RGB image (or array of images) and subtracts
    the mean pixel and converts it to float. Expects image
    colors in RGB order.
    """
    return images.astype(np.float32) - config.MEAN_PIXEL

def load_image_gt(dataset, config, image_id, augmentation=None):
    """Load and return ground truth data for an image (image, mask, bounding boxes).

    augmentation: Optional. An imgaug (https://github.com/aleju/imgaug) augmentation.
        For example, passing imgaug.augmenters.Fliplr(0.5) flips images
        right/left 50% of the time.

    Returns:
    image: [height, width, 3]
    shape: the original shape of the image before resizing and cropping.
    class_ids: [instance_count] Integer class IDs
    bbox: [instance_count, (y1, x1, y2, x2)]
    mask: [height, width, instance_count]. The height and width are those
        of the image unless use_mini_mask is True, in which case they are
        defined in MINI_MASK_SHAPE.
    """
    # Load image and mask
    image = dataset.load_image(image_id)
    mask, class_ids = dataset.load_mask(image_id)
    original_shape = image.shape
    image, window, scale, padding, crop = utils.resize_image(
        image,
        min_dim=config.IMAGE_MIN_DIM,
        min_scale=config.IMAGE_MIN_SCALE,
        max_dim=config.IMAGE_MAX_DIM,
        mode=config.IMAGE_RESIZE_MODE)
    mask = utils.resize_mask(mask, scale, padding, crop)

    # Augmentation
    # This requires the imgaug lib (https://github.com/aleju/imgaug)
    if augmentation:
        import imgaug

        # Augmenters that are safe to apply to masks
        # Some, such as Affine, have settings that make them unsafe, so always
        # test your augmentation on masks
        MASK_AUGMENTERS = ["Sequential", "SomeOf", "OneOf", "Sometimes",
                           "Fliplr", "Flipud", "CropAndPad",
                           "Affine", "PiecewiseAffine"]

        def hook(images, augmenter, parents, default):
            """Determines which augmenters to apply to masks."""
            return augmenter.__class__.__name__ in MASK_AUGMENTERS

        # Store shapes before augmentation to compare
        image_shape = image.shape
        # print("image_shape is {}".format(image_shape))
        mask_shape = mask.shape
        # Make augmenters deterministic to apply similarly to images and masks
        det = augmentation.to_deterministic()
        image = det.augment_image(image)
        # Change mask to np.uint8 because imgaug doesn't support np.bool
        mask = det.augment_image(mask.astype(np.uint8),
                                 hooks=imgaug.HooksImages(activator=hook))
        # Verify that shapes didn't change
        assert image.shape == image_shape, "Augmentation shouldn't change image size"
        assert mask.shape == mask_shape, "Augmentation shouldn't change mask size"
        # Change mask back to bool
        mask = mask.astype(np.bool)

    # Note that some boxes might be all zeros if the corresponding mask got cropped out.
    # and here is to filter them out
    _idx = np.sum(mask, axis=(0, 1)) > 0
    mask = mask[:, :, _idx]
    class_ids = class_ids[_idx]
    # Bounding boxes. Note that some boxes might be all zeros
    # if the corresponding mask got cropped out.
    # bbox: [num_instances, (y1, x1, y2, x2)]
    bbox = utils.extract_bboxes(mask)

    # Active classes
    # Different datasets have different classes, so track the
    # classes supported in the dataset of this image.
    active_class_ids = np.zeros([dataset.num_classes], dtype=np.int32)
    source_class_ids = dataset.source_class_ids[dataset.image_info[image_id]["source"]]
    active_class_ids[source_class_ids] = 1

    # Resize masks to smaller size to reduce memory usage
    if config.USE_MINI_MASK:
        mask = utils.minimize_mask(bbox, mask, config.MINI_MASK_SHAPE)

    # Image meta data
    image_meta = compose_image_meta(image_id, original_shape, image.shape,
                                    window, scale, active_class_ids)

    return image, image_meta, class_ids, bbox, mask


def build_detection_targets(rpn_rois, gt_class_ids, gt_boxes, gt_masks, config):
    """Generate targets for training Stage 2 classifier and mask heads.
    This is not used in normal training. It's useful for debugging or to train
    the Mask RCNN heads without using the RPN head.

    Inputs:
    rpn_rois: [N, (y1, x1, y2, x2)] proposal boxes.
    gt_class_ids: [instance count] Integer class IDs
    gt_boxes: [instance count, (y1, x1, y2, x2)]
    gt_masks: [height, width, instance count] Ground truth masks. Can be full
              size or mini-masks.

    Returns:
    rois: [TRAIN_ROIS_PER_IMAGE, (y1, x1, y2, x2)]
    class_ids: [TRAIN_ROIS_PER_IMAGE]. Integer class IDs.
    bboxes: [TRAIN_ROIS_PER_IMAGE, NUM_CLASSES, (y, x, log(h), log(w))]. Class-specific
            bbox refinements.
    masks: [TRAIN_ROIS_PER_IMAGE, height, width, NUM_CLASSES). Class specific masks cropped
           to bbox boundaries and resized to neural network output size.
    """
    assert rpn_rois.shape[0] > 0
    assert gt_class_ids.dtype == np.int32, "Expected int but got {}".format(
        gt_class_ids.dtype)
    assert gt_boxes.dtype == np.int32, "Expected int but got {}".format(
        gt_boxes.dtype)
    assert gt_masks.dtype == np.bool_, "Expected bool but got {}".format(
        gt_masks.dtype)

    # It's common to add GT Boxes to ROIs but we don't do that here because
    # according to XinLei Chen's paper, it doesn't help.

    # Trim empty padding in gt_boxes and gt_masks parts
    instance_ids = np.where(gt_class_ids > 0)[0]
    assert instance_ids.shape[0] > 0, "Image must contain instances."
    gt_class_ids = gt_class_ids[instance_ids]
    gt_boxes = gt_boxes[instance_ids]
    gt_masks = gt_masks[:, :, instance_ids]

    # Compute areas of ROIs and ground truth boxes.
    rpn_roi_area = (rpn_rois[:, 2] - rpn_rois[:, 0]) * \
        (rpn_rois[:, 3] - rpn_rois[:, 1])
    gt_box_area = (gt_boxes[:, 2] - gt_boxes[:, 0]) * \
        (gt_boxes[:, 3] - gt_boxes[:, 1])

    # Compute overlaps [rpn_rois, gt_boxes]
    overlaps = np.zeros((rpn_rois.shape[0], gt_boxes.shape[0]))
    for i in range(overlaps.shape[1]):
        gt = gt_boxes[i]
        overlaps[:, i] = utils.compute_iou(
            gt, rpn_rois, gt_box_area[i], rpn_roi_area)

    # Assign ROIs to GT boxes
    rpn_roi_iou_argmax = np.argmax(overlaps, axis=1)
    rpn_roi_iou_max = overlaps[np.arange(
        overlaps.shape[0]), rpn_roi_iou_argmax]
    # GT box assigned to each ROI
    rpn_roi_gt_boxes = gt_boxes[rpn_roi_iou_argmax]
    rpn_roi_gt_class_ids = gt_class_ids[rpn_roi_iou_argmax]

    # Positive ROIs are those with >= 0.5 IoU with a GT box.
    fg_ids = np.where(rpn_roi_iou_max > 0.5)[0]

    # Negative ROIs are those with max IoU 0.1-0.5 (hard example mining)
    # TODO: To hard example mine or not to hard example mine, that's the question
    # bg_ids = np.where((rpn_roi_iou_max >= 0.1) & (rpn_roi_iou_max < 0.5))[0]
    bg_ids = np.where(rpn_roi_iou_max < 0.5)[0]

    # Subsample ROIs. Aim for 33% foreground.
    # FG
    fg_roi_count = int(config.TRAIN_ROIS_PER_IMAGE * config.ROI_POSITIVE_RATIO)
    if fg_ids.shape[0] > fg_roi_count:
        keep_fg_ids = np.random.choice(fg_ids, fg_roi_count, replace=False)
    else:
        keep_fg_ids = fg_ids
    # BG
    remaining = config.TRAIN_ROIS_PER_IMAGE - keep_fg_ids.shape[0]
    if bg_ids.shape[0] > remaining:
        keep_bg_ids = np.random.choice(bg_ids, remaining, replace=False)
    else:
        keep_bg_ids = bg_ids
    # Combine indices of ROIs to keep
    keep = np.concatenate([keep_fg_ids, keep_bg_ids])
    # Need more?
    remaining = config.TRAIN_ROIS_PER_IMAGE - keep.shape[0]
    if remaining > 0:
        # Looks like we don't have enough samples to maintain the desired
        # balance. Reduce requirements and fill in the rest. This is
        # likely different from the Mask RCNN paper.

        # There is a small chance we have neither fg nor bg samples.
        if keep.shape[0] == 0:
            # Pick bg regions with easier IoU threshold
            bg_ids = np.where(rpn_roi_iou_max < 0.5)[0]
            assert bg_ids.shape[0] >= remaining
            keep_bg_ids = np.random.choice(bg_ids, remaining, replace=False)
            assert keep_bg_ids.shape[0] == remaining
            keep = np.concatenate([keep, keep_bg_ids])
        else:
            # Fill the rest with repeated bg rois.
            keep_extra_ids = np.random.choice(
                keep_bg_ids, remaining, replace=True)
            keep = np.concatenate([keep, keep_extra_ids])
    assert keep.shape[0] == config.TRAIN_ROIS_PER_IMAGE, \
        "keep doesn't match ROI batch size {}, {}".format(
            keep.shape[0], config.TRAIN_ROIS_PER_IMAGE)

    # Reset the gt boxes assigned to BG ROIs.
    rpn_roi_gt_boxes[keep_bg_ids, :] = 0
    rpn_roi_gt_class_ids[keep_bg_ids] = 0

    # For each kept ROI, assign a class_id, and for FG ROIs also add bbox refinement.
    rois = rpn_rois[keep]
    roi_gt_boxes = rpn_roi_gt_boxes[keep]
    roi_gt_class_ids = rpn_roi_gt_class_ids[keep]
    roi_gt_assignment = rpn_roi_iou_argmax[keep]

    # Class-aware bbox deltas. [y, x, log(h), log(w)]
    bboxes = np.zeros((config.TRAIN_ROIS_PER_IMAGE,
                       config.NUM_CLASSES, 4), dtype=np.float32)
    pos_ids = np.where(roi_gt_class_ids > 0)[0]
    bboxes[pos_ids, roi_gt_class_ids[pos_ids]] = utils.box_refinement(
        rois[pos_ids], roi_gt_boxes[pos_ids, :4])
    # Normalize bbox refinements
    bboxes /= config.BBOX_STD_DEV

    # Generate class-specific target masks
    masks = np.zeros((config.TRAIN_ROIS_PER_IMAGE, config.MASK_SHAPE[0], config.MASK_SHAPE[1], config.NUM_CLASSES),
                     dtype=np.float32)
    for i in pos_ids:
        class_id = roi_gt_class_ids[i]
        assert class_id > 0, "class id must be greater than 0"
        gt_id = roi_gt_assignment[i]
        class_mask = gt_masks[:, :, gt_id]

        if config.USE_MINI_MASK:
            # Create a mask placeholder, the size of the image
            placeholder = np.zeros(config.IMAGE_SHAPE[:2], dtype=bool)
            # GT box
            gt_y1, gt_x1, gt_y2, gt_x2 = gt_boxes[gt_id]
            gt_w = gt_x2 - gt_x1
            gt_h = gt_y2 - gt_y1
            # Resize mini mask to size of GT box
            placeholder[gt_y1:gt_y2, gt_x1:gt_x2] = \
                np.round(utils.resize(class_mask, (gt_h, gt_w))).astype(bool)
            # Place the mini batch in the placeholder
            class_mask = placeholder

        # Pick part of the mask and resize it
        y1, x1, y2, x2 = rois[i].astype(np.int32)
        m = class_mask[y1:y2, x1:x2]
        mask = utils.resize(m, config.MASK_SHAPE)
        masks[i, :, :, class_id] = mask

    return rois, roi_gt_class_ids, bboxes, masks


def build_rpn_targets(image_shape, anchors, gt_class_ids, gt_boxes, config):
    """Given the anchors and GT boxes, compute overlaps and identify positive
    anchors and deltas to refine them to match their corresponding GT boxes.

    anchors: [num_anchors, (y1, x1, y2, x2)]
    gt_class_ids: [num_gt_boxes] Integer class IDs.
    gt_boxes: [num_gt_boxes, (y1, x1, y2, x2)]

    Returns:
    rpn_match: [N] (int32) matches between anchors and GT boxes.
               1 = positive anchor, -1 = negative anchor, 0 = neutral
    rpn_bbox: [N, (dy, dx, log(dh), log(dw))] Anchor bbox deltas.
    """
    # RPN Match: 1 = positive anchor, -1 = negative anchor, 0 = neutral
    rpn_match = np.zeros([anchors.shape[0]], dtype=np.int32)
    # RPN bounding boxes: [max anchors per image, (dy, dx, log(dh), log(dw))]
    rpn_bbox = np.zeros((config.RPN_TRAIN_ANCHORS_PER_IMAGE, 4))

    # Handle COCO crowds
    # A crowd box in COCO is a bounding box around several instances. Exclude
    # them from training. A crowd box is given a negative class ID.
    crowd_ix = np.where(gt_class_ids < 0)[0]
    if crowd_ix.shape[0] > 0:
        # Filter out crowds from ground truth class IDs and boxes
        non_crowd_ix = np.where(gt_class_ids > 0)[0]
        crowd_boxes = gt_boxes[crowd_ix]
        gt_class_ids = gt_class_ids[non_crowd_ix]
        gt_boxes = gt_boxes[non_crowd_ix]
        # Compute overlaps with crowd boxes [anchors, crowds]
        crowd_overlaps = utils.compute_overlaps(anchors, crowd_boxes)
        crowd_iou_max = np.amax(crowd_overlaps, axis=1)
        no_crowd_bool = (crowd_iou_max < 0.001)
    else:
        # All anchors don't intersect a crowd
        no_crowd_bool = np.ones([anchors.shape[0]], dtype=bool)

    # Compute overlaps [num_anchors, num_gt_boxes]
    overlaps = utils.compute_overlaps(anchors, gt_boxes)

    # Match anchors to GT Boxes
    # If an anchor overlaps a GT box with IoU >= 0.7 then it's positive.
    # If an anchor overlaps a GT box with IoU < 0.3 then it's negative.
    # Neutral anchors are those that don't match the conditions above,
    # and they don't influence the loss function.
    # However, don't keep any GT box unmatched (rare, but happens). Instead,
    # match it to the closest anchor (even if its max IoU is < 0.3).
    #
    # 1. Set negative anchors first. They get overwritten below if a GT box is
    # matched to them. Skip boxes in crowd areas.
    anchor_iou_argmax = np.argmax(overlaps, axis=1)
    anchor_iou_max = overlaps[np.arange(overlaps.shape[0]), anchor_iou_argmax]
    rpn_match[(anchor_iou_max < 0.3) & (no_crowd_bool)] = -1
    # 2. Set an anchor for each GT box (regardless of IoU value).
    # If multiple anchors have the same IoU match all of them
    gt_iou_argmax = np.argwhere(overlaps == np.max(overlaps, axis=0))[:,0]
    rpn_match[gt_iou_argmax] = 1
    # 3. Set anchors with high overlap as positive.
    rpn_match[anchor_iou_max >= 0.7] = 1

    # Subsample to balance positive and negative anchors
    # Don't let positives be more than half the anchors
    ids = np.where(rpn_match == 1)[0]
    extra = len(ids) - (config.RPN_TRAIN_ANCHORS_PER_IMAGE // 2)
    if extra > 0:
        # Reset the extra ones to neutral
        ids = np.random.choice(ids, extra, replace=False)
        rpn_match[ids] = 0
    # Same for negative proposals
    ids = np.where(rpn_match == -1)[0]
    extra = len(ids) - (config.RPN_TRAIN_ANCHORS_PER_IMAGE -
                        np.sum(rpn_match == 1))
    if extra > 0:
        # Rest the extra ones to neutral
        ids = np.random.choice(ids, extra, replace=False)
        rpn_match[ids] = 0

    # For positive anchors, compute shift and scale needed to transform them
    # to match the corresponding GT boxes.
    ids = np.where(rpn_match == 1)[0]
    ix = 0  # index into rpn_bbox
    # TODO: use box_refinement() rather than duplicating the code here
    for i, a in zip(ids, anchors[ids]):
        # Closest gt box (it might have IoU < 0.7)
        gt = gt_boxes[anchor_iou_argmax[i]]

        # Convert coordinates to center plus width/height.
        # GT Box
        gt_h = gt[2] - gt[0]
        gt_w = gt[3] - gt[1]
        gt_center_y = gt[0] + 0.5 * gt_h
        gt_center_x = gt[1] + 0.5 * gt_w
        # Anchor
        a_h = a[2] - a[0]
        a_w = a[3] - a[1]
        a_center_y = a[0] + 0.5 * a_h
        a_center_x = a[1] + 0.5 * a_w

        # Compute the bbox refinement that the RPN should predict.
        rpn_bbox[ix] = [
            (gt_center_y - a_center_y) / a_h,
            (gt_center_x - a_center_x) / a_w,
            np.log(gt_h / a_h),
            np.log(gt_w / a_w),
        ]
        # Normalize
        rpn_bbox[ix] /= config.RPN_BBOX_STD_DEV
        ix += 1

    return rpn_match, rpn_bbox


def generate_random_rois(image_shape, count, gt_class_ids, gt_boxes):
    """Generates ROI proposals similar to what a region proposal network
    would generate.

    image_shape: [Height, Width, Depth]
    count: Number of ROIs to generate
    gt_class_ids: [N] Integer ground truth class IDs
    gt_boxes: [N, (y1, x1, y2, x2)] Ground truth boxes in pixels.

    Returns: [count, (y1, x1, y2, x2)] ROI boxes in pixels.
    """
    # placeholder
    rois = np.zeros((count, 4), dtype=np.int32)

    # Generate random ROIs around GT boxes (90% of count)
    rois_per_box = int(0.9 * count / gt_boxes.shape[0])
    for i in range(gt_boxes.shape[0]):
        gt_y1, gt_x1, gt_y2, gt_x2 = gt_boxes[i]
        h = gt_y2 - gt_y1
        w = gt_x2 - gt_x1
        # random boundaries
        r_y1 = max(gt_y1 - h, 0)
        r_y2 = min(gt_y2 + h, image_shape[0])
        r_x1 = max(gt_x1 - w, 0)
        r_x2 = min(gt_x2 + w, image_shape[1])

        # To avoid generating boxes with zero area, we generate double what
        # we need and filter out the extra. If we get fewer valid boxes
        # than we need, we loop and try again.
        while True:
            y1y2 = np.random.randint(r_y1, r_y2, (rois_per_box * 2, 2))
            x1x2 = np.random.randint(r_x1, r_x2, (rois_per_box * 2, 2))
            # Filter out zero area boxes
            threshold = 1
            y1y2 = y1y2[np.abs(y1y2[:, 0] - y1y2[:, 1]) >=
                        threshold][:rois_per_box]
            x1x2 = x1x2[np.abs(x1x2[:, 0] - x1x2[:, 1]) >=
                        threshold][:rois_per_box]
            if y1y2.shape[0] == rois_per_box and x1x2.shape[0] == rois_per_box:
                break

        # Sort on axis 1 to ensure x1 <= x2 and y1 <= y2 and then reshape
        # into x1, y1, x2, y2 order
        x1, x2 = np.split(np.sort(x1x2, axis=1), 2, axis=1)
        y1, y2 = np.split(np.sort(y1y2, axis=1), 2, axis=1)
        box_rois = np.hstack([y1, x1, y2, x2])
        rois[rois_per_box * i:rois_per_box * (i + 1)] = box_rois

    # Generate random ROIs anywhere in the image (10% of count)
    remaining_count = count - (rois_per_box * gt_boxes.shape[0])
    # To avoid generating boxes with zero area, we generate double what
    # we need and filter out the extra. If we get fewer valid boxes
    # than we need, we loop and try again.
    while True:
        y1y2 = np.random.randint(0, image_shape[0], (remaining_count * 2, 2))
        x1x2 = np.random.randint(0, image_shape[1], (remaining_count * 2, 2))
        # Filter out zero area boxes
        threshold = 1
        y1y2 = y1y2[np.abs(y1y2[:, 0] - y1y2[:, 1]) >=
                    threshold][:remaining_count]
        x1x2 = x1x2[np.abs(x1x2[:, 0] - x1x2[:, 1]) >=
                    threshold][:remaining_count]
        if y1y2.shape[0] == remaining_count and x1x2.shape[0] == remaining_count:
            break

    # Sort on axis 1 to ensure x1 <= x2 and y1 <= y2 and then reshape
    # into x1, y1, x2, y2 order
    x1, x2 = np.split(np.sort(x1x2, axis=1), 2, axis=1)
    y1, y2 = np.split(np.sort(y1y2, axis=1), 2, axis=1)
    global_rois = np.hstack([y1, x1, y2, x2])
    rois[-remaining_count:] = global_rois
    return rois


class DataGenerator(KU.Sequence):
    """An iterable that returns images and corresponding target class ids,
        bounding box deltas, and masks. It inherits from keras.utils.Sequence to avoid data redundancy
        when multiprocessing=True.

        dataset: The Dataset object to pick data from
        config: The model config object
        shuffle: If True, shuffles the samples before every epoch
        augmentation: Optional. An imgaug (https://github.com/aleju/imgaug) augmentation.
            For example, passing imgaug.augmenters.Fliplr(0.5) flips images
            right/left 50% of the time.
        random_rois: If > 0 then generate proposals to be used to train the
                     network classifier and mask heads. Useful if training
                     the Mask RCNN part without the RPN.
        detection_targets: If True, generate detection targets (class IDs, bbox
            deltas, and masks). Typically for debugging or visualizations because
            in trainig detection targets are generated by DetectionTargetLayer.

        Returns a Python iterable. Upon calling __getitem__() on it, the
        iterable returns two lists, inputs and outputs. The contents
        of the lists differ depending on the received arguments:
        inputs list:
        - images: [batch, H, W, C]
        - image_meta: [batch, (meta data)] Image details. See compose_image_meta()
        - rpn_match: [batch, N] Integer (1=positive anchor, -1=negative, 0=neutral)
        - rpn_bbox: [batch, N, (dy, dx, log(dh), log(dw))] Anchor bbox deltas.
        - gt_class_ids: [batch, MAX_GT_INSTANCES] Integer class IDs
        - gt_boxes: [batch, MAX_GT_INSTANCES, (y1, x1, y2, x2)]
        - gt_masks: [batch, height, width, MAX_GT_INSTANCES]. The height and width
                    are those of the image unless use_mini_mask is True, in which
                    case they are defined in MINI_MASK_SHAPE.

        outputs list: Usually empty in regular training. But if detection_targets
            is True then the outputs list contains target class_ids, bbox deltas,
            and masks.
        """

    def __init__(self, dataset, config, shuffle=True, augmentation=None,
                 random_rois=0, detection_targets=False):

        self.image_ids = np.copy(dataset.image_ids)
        self.dataset = dataset
        self.config = config

        # Anchors
        # [anchor_count, (y1, x1, y2, x2)]
        self.backbone_shapes = compute_backbone_shapes(config, config.IMAGE_SHAPE)
        self.anchors = utils.generate_pyramid_anchors(config.RPN_ANCHOR_SCALES,
                                                      config.RPN_ANCHOR_RATIOS,
                                                      self.backbone_shapes,
                                                      config.BACKBONE_STRIDES,
                                                      config.RPN_ANCHOR_STRIDE)

        self.shuffle = shuffle
        self.augmentation = augmentation
        self.random_rois = random_rois
        self.batch_size = self.config.BATCH_SIZE
        self.detection_targets = detection_targets

    def __len__(self):
        return int(np.ceil(len(self.image_ids) / float(self.batch_size)))

    def __getitem__(self, idx):
        b = 0
        image_index = -1
        while b < self.batch_size:
            # Increment index to pick next image. Shuffle if at the start of an epoch.
            image_index = (image_index + 1) % len(self.image_ids)

            if self.shuffle and image_index == 0:
                np.random.shuffle(self.image_ids)

            # Get GT bounding boxes and masks for image.
            image_id = self.image_ids[image_index]
            image, image_meta, gt_class_ids, gt_boxes, gt_masks = \
                load_image_gt(self.dataset, self.config, image_id,
                              augmentation=self.augmentation)

            # Skip images that have no instances. This can happen in cases
            # where we train on a subset of classes and the image doesn't
            # have any of the classes we care about.
            if not np.any(gt_class_ids > 0):
                continue

            # RPN Targets
            rpn_match, rpn_bbox = build_rpn_targets(image.shape, self.anchors,
                                                    gt_class_ids, gt_boxes, self.config)

            # Mask R-CNN Targets
            if self.random_rois:
                rpn_rois = generate_random_rois(
                    image.shape, self.random_rois, gt_class_ids, gt_boxes)
                if self.detection_targets:
                    rois, mrcnn_class_ids, mrcnn_bbox, mrcnn_mask = \
                        build_detection_targets(
                            rpn_rois, gt_class_ids, gt_boxes, gt_masks, self.config)

            # Init batch arrays
            if b == 0:
                batch_image_meta = np.zeros(
                    (self.batch_size,) + image_meta.shape, dtype=image_meta.dtype)
                batch_rpn_match = np.zeros(
                    [self.batch_size, self.anchors.shape[0], 1], dtype=rpn_match.dtype)
                batch_rpn_bbox = np.zeros(
                    [self.batch_size, self.config.RPN_TRAIN_ANCHORS_PER_IMAGE, 4], dtype=rpn_bbox.dtype)
                batch_images = np.zeros(
                    (self.batch_size,) + image.shape, dtype=np.float32)
                batch_gt_class_ids = np.zeros(
                    (self.batch_size, self.config.MAX_GT_INSTANCES), dtype=np.int32)
                batch_gt_boxes = np.zeros(
                    (self.batch_size, self.config.MAX_GT_INSTANCES, 4), dtype=np.int32)
                batch_gt_masks = np.zeros(
                    (self.batch_size, gt_masks.shape[0], gt_masks.shape[1],
                     self.config.MAX_GT_INSTANCES), dtype=gt_masks.dtype)
                if self.random_rois:
                    batch_rpn_rois = np.zeros(
                        (self.batch_size, rpn_rois.shape[0], 4), dtype=rpn_rois.dtype)
                    if self.detection_targets:
                        batch_rois = np.zeros(
                            (self.batch_size,) + rois.shape, dtype=rois.dtype)
                        batch_mrcnn_class_ids = np.zeros(
                            (self.batch_size,) + mrcnn_class_ids.shape, dtype=mrcnn_class_ids.dtype)
                        batch_mrcnn_bbox = np.zeros(
                            (self.batch_size,) + mrcnn_bbox.shape, dtype=mrcnn_bbox.dtype)
                        batch_mrcnn_mask = np.zeros(
                            (self.batch_size,) + mrcnn_mask.shape, dtype=mrcnn_mask.dtype)

            # If more instances than fits in the array, sub-sample from them.
            if gt_boxes.shape[0] > self.config.MAX_GT_INSTANCES:
                ids = np.random.choice(
                    np.arange(gt_boxes.shape[0]), self.config.MAX_GT_INSTANCES, replace=False)
                gt_class_ids = gt_class_ids[ids]
                gt_boxes = gt_boxes[ids]
                gt_masks = gt_masks[:, :, ids]

            # Add to batch
            batch_image_meta[b] = image_meta
            batch_rpn_match[b] = rpn_match[:, np.newaxis]
            batch_rpn_bbox[b] = rpn_bbox
            batch_images[b] = mold_image(image.astype(np.float32), self.config)
            batch_gt_class_ids[b, :gt_class_ids.shape[0]] = gt_class_ids
            batch_gt_boxes[b, :gt_boxes.shape[0]] = gt_boxes
            batch_gt_masks[b, :, :, :gt_masks.shape[-1]] = gt_masks
            if self.random_rois:
                batch_rpn_rois[b] = rpn_rois
                if self.detection_targets:
                    batch_rois[b] = rois
                    batch_mrcnn_class_ids[b] = mrcnn_class_ids
                    batch_mrcnn_bbox[b] = mrcnn_bbox
                    batch_mrcnn_mask[b] = mrcnn_mask
            b += 1

        inputs = [batch_images, batch_image_meta, batch_rpn_match, batch_rpn_bbox,
                  batch_gt_class_ids, batch_gt_boxes, batch_gt_masks]
        outputs = []

        if self.random_rois:
            inputs.extend([batch_rpn_rois])
            if self.detection_targets:
                inputs.extend([batch_rois])
                # Keras requires that output and targets have the same number of dimensions
                batch_mrcnn_class_ids = np.expand_dims(
                    batch_mrcnn_class_ids, -1)
                outputs.extend(
                    [batch_mrcnn_class_ids, batch_mrcnn_bbox, batch_mrcnn_mask])

        return inputs, outputs

Data Generator - Complete Explanation
This code handles loading, preprocessing, and batching of training data for Mask R-CNN. It's the data pipeline that feeds images and ground truth into the model during training.

What This Code Does (One Sentence)
Loads images and their annotations (masks, classes, boxes), resizes them, applies augmentations, generates RPN training targets, and creates batches of data ready for model training.

The 4 Main Components
Component	Purpose
load_image_gt()	Loads one image + its ground truth from dataset
build_rpn_targets()	Generates RPN training targets (positive/negative anchors)
build_detection_targets()	Generates detection head targets (debugging only)
DataGenerator	Keras Sequence that creates batches of data

In [ ]:
############################################################
#  MaskRCNN Class
############################################################

class MaskRCNN(object):
    """Encapsulates the Mask RCNN model functionality.

    The actual Keras model is in the keras_model property.
    """

    def __init__(self, mode, config, model_dir):
        """
        mode: Either "training" or "inference"
        config: A Sub-class of the Config class
        model_dir: Directory to save training logs and trained weights
        """
        assert mode in ['training', 'inference']
        self.mode = mode
        self.config = config
        self.model_dir = model_dir
        self.set_log_dir()
        self.keras_model = self.build(mode=mode, config=config)

    def build(self, mode, config):
        """Build Mask R-CNN architecture.
            input_shape: The shape of the input image.
            mode: Either "training" or "inference". The inputs and
                outputs of the model differ accordingly.
        """
        assert mode in ['training', 'inference']

        # Image size must be dividable by 2 multiple times
        h, w = config.IMAGE_SHAPE[:2]
        if h / 2**6 != int(h / 2**6) or w / 2**6 != int(w / 2**6):
            raise Exception("Image size must be dividable by 2 at least 6 times "
                            "to avoid fractions when downscaling and upscaling."
                            "For example, use 256, 320, 384, 448, 512, ... etc. ")

        # Inputs
        input_image = KL.Input(
            shape=[None, None, config.IMAGE_SHAPE[2]], name="input_image")
        input_image_meta = KL.Input(shape=[config.IMAGE_META_SIZE],
                                    name="input_image_meta")
        if mode == "training":
            # RPN GT
            input_rpn_match = KL.Input(
                shape=[None, 1], name="input_rpn_match", dtype=tf.int32)
            input_rpn_bbox = KL.Input(
                shape=[None, 4], name="input_rpn_bbox", dtype=tf.float32)

            # Detection GT (class IDs, bounding boxes, and masks)
            # 1. GT Class IDs (zero padded)
            input_gt_class_ids = KL.Input(
                shape=[None], name="input_gt_class_ids", dtype=tf.int32)
            # 2. GT Boxes in pixels (zero padded)
            # [batch, MAX_GT_INSTANCES, (y1, x1, y2, x2)] in image coordinates
            input_gt_boxes = KL.Input(
                shape=[None, 4], name="input_gt_boxes", dtype=tf.float32)
            # Normalize coordinates
            gt_boxes = KL.Lambda(lambda x: norm_boxes_graph(
                x, K.shape(input_image)[1:3]))(input_gt_boxes)
            # 3. GT Masks (zero padded)
            # [batch, height, width, MAX_GT_INSTANCES]
            if config.USE_MINI_MASK:
                input_gt_masks = KL.Input(
                    shape=[config.MINI_MASK_SHAPE[0],
                           config.MINI_MASK_SHAPE[1], None],
                    name="input_gt_masks", dtype=bool)
            else:
                input_gt_masks = KL.Input(
                    shape=[config.IMAGE_SHAPE[0], config.IMAGE_SHAPE[1], None],
                    name="input_gt_masks", dtype=bool)
        elif mode == "inference":
            # Anchors in normalized coordinates
            input_anchors = KL.Input(shape=[None, 4], name="input_anchors")

        # Build the shared convolutional layers.
        # Bottom-up Layers
        # Returns a list of the last layers of each stage, 5 in total.
        # Don't create the thead (stage 5), so we pick the 4th item in the list.
        if callable(config.BACKBONE):
            _, C2, C3, C4, C5 = config.BACKBONE(input_image, stage5=True,
                                                train_bn=config.TRAIN_BN)
        else:
            _, C2, C3, C4, C5 = resnet_graph(input_image, config.BACKBONE,
                                             stage5=True, train_bn=config.TRAIN_BN)
        # Top-down Layers
        # TODO: add assert to varify feature map sizes match what's in config
        P5 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c5p5')(C5)
        P4 = KL.Add(name="fpn_p4add")([
            KL.UpSampling2D(size=(2, 2), name="fpn_p5upsampled")(P5),
            KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c4p4')(C4)])
        P3 = KL.Add(name="fpn_p3add")([
            KL.UpSampling2D(size=(2, 2), name="fpn_p4upsampled")(P4),
            KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c3p3')(C3)])
        P2 = KL.Add(name="fpn_p2add")([
            KL.UpSampling2D(size=(2, 2), name="fpn_p3upsampled")(P3),
            KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (1, 1), name='fpn_c2p2')(C2)])
        # Attach 3x3 conv to all P layers to get the final feature maps.
        P2 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p2")(P2)
        P3 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p3")(P3)
        P4 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p4")(P4)
        P5 = KL.Conv2D(config.TOP_DOWN_PYRAMID_SIZE, (3, 3), padding="SAME", name="fpn_p5")(P5)
        # P6 is used for the 5th anchor scale in RPN. Generated by
        # subsampling from P5 with stride of 2.
        P6 = KL.MaxPooling2D(pool_size=(1, 1), strides=2, name="fpn_p6")(P5)

        # Note that P6 is used in RPN, but not in the classifier heads.
        rpn_feature_maps = [P2, P3, P4, P5, P6]
        mrcnn_feature_maps = [P2, P3, P4, P5]

        # Anchors
        if mode == "training":
            anchors = self.get_anchors(config.IMAGE_SHAPE)
            # Duplicate across the batch dimension because Keras requires it
            # TODO: can this be optimized to avoid duplicating the anchors?
            anchors = np.broadcast_to(anchors, (config.BATCH_SIZE,) + anchors.shape)
            # A hack to get around Keras's bad support for constants
            # This class returns a constant layer
            class ConstLayer(tf.keras.layers.Layer):
                def __init__(self, x, name=None):
                    super(ConstLayer, self).__init__(name=name)
                    self.x = tf.Variable(x)

                def call(self, input):
                    return self.x

            anchors = ConstLayer(anchors, name="anchors")(input_image)
        else:
            anchors = input_anchors

        # RPN Model
        rpn = build_rpn_model(config.RPN_ANCHOR_STRIDE,
                              len(config.RPN_ANCHOR_RATIOS), config.TOP_DOWN_PYRAMID_SIZE)
        # Loop through pyramid layers
        layer_outputs = []  # list of lists
        for p in rpn_feature_maps:
            layer_outputs.append(rpn([p]))
        # Concatenate layer outputs
        # Convert from list of lists of level outputs to list of lists
        # of outputs across levels.
        # e.g. [[a1, b1, c1], [a2, b2, c2]] => [[a1, a2], [b1, b2], [c1, c2]]
        output_names = ["rpn_class_logits", "rpn_class", "rpn_bbox"]
        outputs = list(zip(*layer_outputs))
        outputs = [KL.Concatenate(axis=1, name=n)(list(o))
                   for o, n in zip(outputs, output_names)]

        rpn_class_logits, rpn_class, rpn_bbox = outputs

        # Generate proposals
        # Proposals are [batch, N, (y1, x1, y2, x2)] in normalized coordinates
        # and zero padded.
        proposal_count = config.POST_NMS_ROIS_TRAINING if mode == "training"\
            else config.POST_NMS_ROIS_INFERENCE
        rpn_rois = ProposalLayer(
            proposal_count=proposal_count,
            nms_threshold=config.RPN_NMS_THRESHOLD,
            name="ROI",
            config=config)([rpn_class, rpn_bbox, anchors])

        if mode == "training":
            # Class ID mask to mark class IDs supported by the dataset the image
            # came from.
            active_class_ids = KL.Lambda(
                lambda x: parse_image_meta_graph(x)["active_class_ids"]
                )(input_image_meta)

            if not config.USE_RPN_ROIS:
                # Ignore predicted ROIs and use ROIs provided as an input.
                input_rois = KL.Input(shape=[config.POST_NMS_ROIS_TRAINING, 4],
                                      name="input_roi", dtype=np.int32)
                # Normalize coordinates
                target_rois = KL.Lambda(lambda x: norm_boxes_graph(
                    x, K.shape(input_image)[1:3]))(input_rois)
            else:
                target_rois = rpn_rois

            # Generate detection targets
            # Subsamples proposals and generates target outputs for training
            # Note that proposal class IDs, gt_boxes, and gt_masks are zero
            # padded. Equally, returned rois and targets are zero padded.
            rois, target_class_ids, target_bbox, target_mask =\
                DetectionTargetLayer(config, name="proposal_targets")([
                    target_rois, input_gt_class_ids, gt_boxes, input_gt_masks])

            # Network Heads
            # TODO: verify that this handles zero padded ROIs
            mrcnn_class_logits, mrcnn_class, mrcnn_bbox =\
                fpn_classifier_graph(rois, mrcnn_feature_maps, input_image_meta,
                                     config.POOL_SIZE, config.NUM_CLASSES,
                                     train_bn=config.TRAIN_BN,
                                     fc_layers_size=config.FPN_CLASSIF_FC_LAYERS_SIZE)

            mrcnn_mask = build_fpn_mask_graph(rois, mrcnn_feature_maps,
                                              input_image_meta,
                                              config.MASK_POOL_SIZE,
                                              config.NUM_CLASSES,
                                              train_bn=config.TRAIN_BN)

            # TODO: clean up (use tf.identify if necessary)
            output_rois = KL.Lambda(lambda x: x * 1, name="output_rois")(rois)

            # Losses
            rpn_class_loss = KL.Lambda(lambda x: rpn_class_loss_graph(*x), name="rpn_class_loss")(
                [input_rpn_match, rpn_class_logits])
            rpn_bbox_loss = KL.Lambda(lambda x: rpn_bbox_loss_graph(config, *x), name="rpn_bbox_loss")(
                [input_rpn_bbox, input_rpn_match, rpn_bbox])
            class_loss = KL.Lambda(lambda x: mrcnn_class_loss_graph(*x), name="mrcnn_class_loss")(
                [target_class_ids, mrcnn_class_logits, active_class_ids])
            bbox_loss = KL.Lambda(lambda x: mrcnn_bbox_loss_graph(*x), name="mrcnn_bbox_loss")(
                [target_bbox, target_class_ids, mrcnn_bbox])
            mask_loss = KL.Lambda(lambda x: mrcnn_mask_loss_graph(*x), name="mrcnn_mask_loss")(
                [target_mask, target_class_ids, mrcnn_mask])

            # Model
            inputs = [input_image, input_image_meta,
                      input_rpn_match, input_rpn_bbox, input_gt_class_ids, input_gt_boxes, input_gt_masks]
            if not config.USE_RPN_ROIS:
                inputs.append(input_rois)
            outputs = [rpn_class_logits, rpn_class, rpn_bbox,
                       mrcnn_class_logits, mrcnn_class, mrcnn_bbox, mrcnn_mask,
                       rpn_rois, output_rois,
                       rpn_class_loss, rpn_bbox_loss, class_loss, bbox_loss, mask_loss]
            model = KM.Model(inputs, outputs, name='mask_rcnn')
        else:
            # Network Heads
            # Proposal classifier and BBox regressor heads
            mrcnn_class_logits, mrcnn_class, mrcnn_bbox =\
                fpn_classifier_graph(rpn_rois, mrcnn_feature_maps, input_image_meta,
                                     config.POOL_SIZE, config.NUM_CLASSES,
                                     train_bn=config.TRAIN_BN,
                                     fc_layers_size=config.FPN_CLASSIF_FC_LAYERS_SIZE)

            # Detections
            # output is [batch, num_detections, (y1, x1, y2, x2, class_id, score)] in
            # normalized coordinates
            detections = DetectionLayer(config, name="mrcnn_detection")(
                [rpn_rois, mrcnn_class, mrcnn_bbox, input_image_meta])

            # Create masks for detections
            detection_boxes = KL.Lambda(lambda x: x[..., :4])(detections)
            mrcnn_mask = build_fpn_mask_graph(detection_boxes, mrcnn_feature_maps,
                                              input_image_meta,
                                              config.MASK_POOL_SIZE,
                                              config.NUM_CLASSES,
                                              train_bn=config.TRAIN_BN)

            model = KM.Model([input_image, input_image_meta, input_anchors],
                             [detections, mrcnn_class, mrcnn_bbox,
                                 mrcnn_mask, rpn_rois, rpn_class, rpn_bbox],
                             name='mask_rcnn')

        # Add multi-GPU support.
        if config.GPU_COUNT > 1:
            model = ParallelModel(model, config.GPU_COUNT)

        return model

    def find_last(self):
        """Finds the last checkpoint file of the last trained model in the
        model directory.
        Returns:
            The path of the last checkpoint file
        """
        # Get directory names. Each directory corresponds to a model
        dir_names = next(os.walk(self.model_dir))[1]
        key = self.config.NAME.lower()
        dir_names = filter(lambda f: f.startswith(key), dir_names)
        dir_names = sorted(dir_names)
        if not dir_names:
            import errno
            raise FileNotFoundError(
                errno.ENOENT,
                "Could not find model directory under {}".format(self.model_dir))
        # Pick last directory
        dir_name = os.path.join(self.model_dir, dir_names[-1])
        # Find the last checkpoint
        checkpoints = next(os.walk(dir_name))[2]
        checkpoints = filter(lambda f: f.startswith("mask_rcnn"), checkpoints)
        checkpoints = sorted(checkpoints)
        if not checkpoints:
            import errno
            raise FileNotFoundError(
                errno.ENOENT, "Could not find weight files in {}".format(dir_name))
        checkpoint = os.path.join(dir_name, checkpoints[-1])
        return checkpoint

    def load_weights(self, filepath, by_name=False, exclude=None):
        """Modified version of the corresponding Keras function with
        the addition of multi-GPU support and the ability to exclude
        some layers from loading.
        exclude: list of layer names to exclude
        """
        import h5py
        from tensorflow.python.keras.saving import hdf5_format
        # try:
        #     from keras.engine import saving
        # except ImportError:
        #     # Keras before 2.2 used the 'topology' namespace.
        #     from keras.engine import topology as saving

        if exclude:
            by_name = True

        if h5py is None:
            raise ImportError('`load_weights` requires h5py.')
        with h5py.File(filepath, mode='r') as f:
            if 'layer_names' not in f.attrs and 'model_weights' in f:
                f = f['model_weights']

            # In multi-GPU training, we wrap the model. Get layers
            # of the inner model because they have the weights.
            keras_model = self.keras_model
            layers = keras_model.inner_model.layers if hasattr(keras_model, "inner_model")\
                else keras_model.layers

            # Exclude some layers
            if exclude:
                layers = filter(lambda l: l.name not in exclude, layers)

            if by_name:
                hdf5_format.load_weights_from_hdf5_group_by_name(f, layers)
            else:
                hdf5_format.load_weights_from_hdf5_group(f, layers)

        # Update the log directory
        self.set_log_dir(filepath)

    def get_imagenet_weights(self):
        """Downloads ImageNet trained weights from Keras.
        Returns path to weights file.
        """
        from keras.utils.data_utils import get_file
        TF_WEIGHTS_PATH_NO_TOP = 'https://github.com/fchollet/deep-learning-models/'\
                                 'releases/download/v0.2/'\
                                 'resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5'
        weights_path = get_file('resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5',
                                TF_WEIGHTS_PATH_NO_TOP,
                                cache_subdir='models',
                                md5_hash='a268eb855778b3df3c7506639542a6af')
        return weights_path

    def compile(self, learning_rate, momentum):
        """Gets the model ready for training. Adds losses, regularization, and
        metrics. Then calls the Keras compile() function.
        """
        # Optimizer object
        optimizer = keras.optimizers.legacy.SGD(
            lr=learning_rate, momentum=momentum,
            clipnorm=self.config.GRADIENT_CLIP_NORM)
        # Add Losses
        loss_names = [
            "rpn_class_loss",  "rpn_bbox_loss",
            "mrcnn_class_loss", "mrcnn_bbox_loss", "mrcnn_mask_loss"]
        for name in loss_names:
            layer = self.keras_model.get_layer(name)
            if layer.output in self.keras_model.losses:
                continue
            loss = (
                tf.reduce_mean(input_tensor=layer.output, keepdims=True)
                * self.config.LOSS_WEIGHTS.get(name, 1.))
            self.keras_model.add_loss(loss)

        # Add L2 Regularization
        # Skip gamma and beta weights of batch normalization layers.
        reg_losses = [
            keras.regularizers.l2(self.config.WEIGHT_DECAY)(w) / tf.cast(tf.size(input=w), tf.float32)
            for w in self.keras_model.trainable_weights
            if 'gamma' not in w.name and 'beta' not in w.name]
        self.keras_model.add_loss(tf.add_n(reg_losses))

        # Compile
        self.keras_model.compile(
            optimizer=optimizer,
            loss=[None] * len(self.keras_model.outputs))

        # Add metrics for losses
        for name in loss_names:
            if name in self.keras_model.metrics_names:
                continue
            layer = self.keras_model.get_layer(name)
            self.keras_model.metrics_names.append(name)
            loss = (
                tf.reduce_mean(input_tensor=layer.output, keepdims=True)
                * self.config.LOSS_WEIGHTS.get(name, 1.))
            self.keras_model.add_metric(loss, name=name, aggregation='mean')

    def set_trainable(self, layer_regex, keras_model=None, indent=0, verbose=1):
        """Sets model layers as trainable if their names match
        the given regular expression.
        """
        # Print message on the first call (but not on recursive calls)
        if verbose > 0 and keras_model is None:
            log("Selecting layers to train")

        keras_model = keras_model or self.keras_model

        # In multi-GPU training, we wrap the model. Get layers
        # of the inner model because they have the weights.
        layers = keras_model.inner_model.layers if hasattr(keras_model, "inner_model")\
            else keras_model.layers

        for layer in layers:
            # Is the layer a model?
            if layer.__class__.__name__ == 'Model':
                print("In model: ", layer.name)
                self.set_trainable(
                    layer_regex, keras_model=layer, indent=indent + 4)
                continue

            if not layer.weights:
                continue
            # Is it trainable?
            trainable = bool(re.fullmatch(layer_regex, layer.name))
            # Update layer. If layer is a container, update inner layer.
            if layer.__class__.__name__ == 'TimeDistributed':
                layer.layer.trainable = trainable
            else:
                layer.trainable = trainable
            # Print trainable layer names
            if trainable and verbose > 0:
                log("{}{:20}   ({})".format(" " * indent, layer.name,
                                            layer.__class__.__name__))

    def set_log_dir(self, model_path=None):
        """Sets the model log directory and epoch counter.

        model_path: If None, or a format different from what this code uses
            then set a new log directory and start epochs from 0. Otherwise,
            extract the log directory and the epoch counter from the file
            name.
        """
        # Set date and epoch counter as if starting a new model
        self.epoch = 0
        now = datetime.datetime.now()

        # If we have a model path with date and epochs use them
        if model_path:
            # Continue from we left of. Get epoch and date from the file name
            # A sample model path might look like:
            # \path\to\logs\coco20171029T2315\mask_rcnn_coco_0001.h5 (Windows)
            # /path/to/logs/coco20171029T2315/mask_rcnn_coco_0001.h5 (Linux)
            regex = r".*[/\\][\w-]+(\d{4})(\d{2})(\d{2})T(\d{2})(\d{2})[/\\]mask\_rcnn\_[\w-]+(\d{4})\.h5"
            # Use string for regex since we might want to use pathlib.Path as model_path
            m = re.match(regex, str(model_path))
            if m:
                now = datetime.datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                                        int(m.group(4)), int(m.group(5)))
                # Epoch number in file is 1-based, and in Keras code it's 0-based.
                # So, adjust for that then increment by one to start from the next epoch
                self.epoch = int(m.group(6)) - 1 + 1
                print('Re-starting from epoch %d' % self.epoch)

        # Directory for training logs
        self.log_dir = os.path.join(self.model_dir, "{}{:%Y%m%dT%H%M}".format(
            self.config.NAME.lower(), now))

        # Path to save after each epoch. Include placeholders that get filled by Keras.
        self.checkpoint_path = os.path.join(self.log_dir, "mask_rcnn_{}_*epoch*.h5".format(
            self.config.NAME.lower()))
        self.checkpoint_path = self.checkpoint_path.replace(
            "*epoch*", "{epoch:04d}")

    def train(self, train_dataset, val_dataset, learning_rate, epochs, layers,
              augmentation=None, custom_callbacks=None, no_augmentation_sources=None):
        """Train the model.
        train_dataset, val_dataset: Training and validation Dataset objects.
        learning_rate: The learning rate to train with
        epochs: Number of training epochs. Note that previous training epochs
                are considered to be done alreay, so this actually determines
                the epochs to train in total rather than in this particaular
                call.
        layers: Allows selecting wich layers to train. It can be:
            - A regular expression to match layer names to train
            - One of these predefined values:
              heads: The RPN, classifier and mask heads of the network
              all: All the layers
              3+: Train Resnet stage 3 and up
              4+: Train Resnet stage 4 and up
              5+: Train Resnet stage 5 and up
        augmentation: Optional. An imgaug (https://github.com/aleju/imgaug)
            augmentation. For example, passing imgaug.augmenters.Fliplr(0.5)
            flips images right/left 50% of the time. You can pass complex
            augmentations as well. This augmentation applies 50% of the
            time, and when it does it flips images right/left half the time
            and adds a Gaussian blur with a random sigma in range 0 to 5.

                augmentation = imgaug.augmenters.Sometimes(0.5, [
                    imgaug.augmenters.Fliplr(0.5),
                    imgaug.augmenters.GaussianBlur(sigma=(0.0, 5.0))
                ])
	    custom_callbacks: Optional. Add custom callbacks to be called
	        with the keras fit_generator method. Must be list of type keras.callbacks.
        no_augmentation_sources: Optional. List of sources to exclude for
            augmentation. A source is string that identifies a dataset and is
            defined in the Dataset class.
        """
        assert self.mode == "training", "Create model in training mode."

        # Pre-defined layer regular expressions
        layer_regex = {
            # all layers but the backbone
            "heads": r"(mrcnn\_.*)|(rpn\_.*)|(fpn\_.*)",
            # From a specific Resnet stage and up
            "3+": r"(res3.*)|(bn3.*)|(res4.*)|(bn4.*)|(res5.*)|(bn5.*)|(mrcnn\_.*)|(rpn\_.*)|(fpn\_.*)",
            "4+": r"(res4.*)|(bn4.*)|(res5.*)|(bn5.*)|(mrcnn\_.*)|(rpn\_.*)|(fpn\_.*)",
            "5+": r"(res5.*)|(bn5.*)|(mrcnn\_.*)|(rpn\_.*)|(fpn\_.*)",
            # All layers
            "all": ".*",
        }
        if layers in layer_regex.keys():
            layers = layer_regex[layers]

        # Data generators
        train_generator = DataGenerator(train_dataset, self.config, shuffle=True,
                                         augmentation=augmentation)
        val_generator = DataGenerator(val_dataset, self.config, shuffle=True)

        # Create log_dir if it does not exist
        if not os.path.exists(self.log_dir):
            os.makedirs(self.log_dir)

        # Callbacks
        callbacks = [
            keras.callbacks.TensorBoard(log_dir=self.log_dir,
                                        histogram_freq=0, write_graph=True, write_images=False),
            keras.callbacks.ModelCheckpoint(self.checkpoint_path,
                                            verbose=0, save_weights_only=True),
        ]

        # Add custom callbacks to the list
        if custom_callbacks:
            callbacks += custom_callbacks

        # Train
        log("\nStarting at epoch {}. LR={}\n".format(self.epoch, learning_rate))
        log("Checkpoint Path: {}".format(self.checkpoint_path))
        self.set_trainable(layers)
        self.compile(learning_rate, self.config.LEARNING_MOMENTUM)

        # Work-around for Windows: Keras fails on Windows when using
        # multiprocessing workers. See discussion here:
        # https://github.com/matterport/Mask_RCNN/issues/13#issuecomment-353124009
        if os.name == 'nt':
            workers = 0
        else:
            workers = multiprocessing.cpu_count()

        self.keras_model.fit(
            train_generator,
            initial_epoch=self.epoch,
            epochs=epochs,
            steps_per_epoch=self.config.STEPS_PER_EPOCH,
            callbacks=callbacks,
            validation_data=val_generator,
            validation_steps=self.config.VALIDATION_STEPS,
            max_queue_size=100,
            workers=0,
            use_multiprocessing=False,
        )
        self.epoch = max(self.epoch, epochs)

    def mold_inputs(self, images):
        """Takes a list of images and modifies them to the format expected
        as an input to the neural network.
        images: List of image matrices [height,width,depth]. Images can have
            different sizes.

        Returns 3 Numpy matrices:
        molded_images: [N, h, w, 3]. Images resized and normalized.
        image_metas: [N, length of meta data]. Details about each image.
        windows: [N, (y1, x1, y2, x2)]. The portion of the image that has the
            original image (padding excluded).
        """
        molded_images = []
        image_metas = []
        windows = []
        for image in images:
            # Resize image
            # TODO: move resizing to mold_image()
            molded_image, window, scale, padding, crop = utils.resize_image(
                image,
                min_dim=self.config.IMAGE_MIN_DIM,
                min_scale=self.config.IMAGE_MIN_SCALE,
                max_dim=self.config.IMAGE_MAX_DIM,
                mode=self.config.IMAGE_RESIZE_MODE)
            molded_image = mold_image(molded_image, self.config)
            # Build image_meta
            image_meta = compose_image_meta(
                0, image.shape, molded_image.shape, window, scale,
                np.zeros([self.config.NUM_CLASSES], dtype=np.int32))
            # Append
            molded_images.append(molded_image)
            windows.append(window)
            image_metas.append(image_meta)
        # Pack into arrays
        molded_images = np.stack(molded_images)
        image_metas = np.stack(image_metas)
        windows = np.stack(windows)
        return molded_images, image_metas, windows

    def unmold_detections(self, detections, mrcnn_mask, original_image_shape,
                          image_shape, window):
        """Reformats the detections of one image from the format of the neural
        network output to a format suitable for use in the rest of the
        application.

        detections: [N, (y1, x1, y2, x2, class_id, score)] in normalized coordinates
        mrcnn_mask: [N, height, width, num_classes]
        original_image_shape: [H, W, C] Original image shape before resizing
        image_shape: [H, W, C] Shape of the image after resizing and padding
        window: [y1, x1, y2, x2] Pixel coordinates of box in the image where the real
                image is excluding the padding.

        Returns:
        boxes: [N, (y1, x1, y2, x2)] Bounding boxes in pixels
        class_ids: [N] Integer class IDs for each bounding box
        scores: [N] Float probability scores of the class_id
        masks: [height, width, num_instances] Instance masks
        """
        # How many detections do we have?
        # Detections array is padded with zeros. Find the first class_id == 0.
        zero_ix = np.where(detections[:, 4] == 0)[0]
        N = zero_ix[0] if zero_ix.shape[0] > 0 else detections.shape[0]

        # Extract boxes, class_ids, scores, and class-specific masks
        boxes = detections[:N, :4]
        class_ids = detections[:N, 4].astype(np.int32)
        scores = detections[:N, 5]
        masks = mrcnn_mask[np.arange(N), :, :, class_ids]

        # Translate normalized coordinates in the resized image to pixel
        # coordinates in the original image before resizing
        window = utils.norm_boxes(window, image_shape[:2])
        wy1, wx1, wy2, wx2 = window
        shift = np.array([wy1, wx1, wy1, wx1])
        wh = wy2 - wy1  # window height
        ww = wx2 - wx1  # window width
        scale = np.array([wh, ww, wh, ww])
        # Convert boxes to normalized coordinates on the window
        boxes = np.divide(boxes - shift, scale)
        # Convert boxes to pixel coordinates on the original image
        boxes = utils.denorm_boxes(boxes, original_image_shape[:2])

        # Filter out detections with zero area. Happens in early training when
        # network weights are still random
        exclude_ix = np.where(
            (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) <= 0)[0]
        if exclude_ix.shape[0] > 0:
            boxes = np.delete(boxes, exclude_ix, axis=0)
            class_ids = np.delete(class_ids, exclude_ix, axis=0)
            scores = np.delete(scores, exclude_ix, axis=0)
            masks = np.delete(masks, exclude_ix, axis=0)
            N = class_ids.shape[0]

        # Resize masks to original image size and set boundary threshold.
        full_masks = []
        for i in range(N):
            # Convert neural network mask to full size mask
            full_mask = utils.unmold_mask(masks[i], boxes[i], original_image_shape)
            full_masks.append(full_mask)
        full_masks = np.stack(full_masks, axis=-1)\
            if full_masks else np.empty(original_image_shape[:2] + (0,))

        return boxes, class_ids, scores, full_masks

    def detect(self, images, verbose=0):
        """Runs the detection pipeline.

        images: List of images, potentially of different sizes.

        Returns a list of dicts, one dict per image. The dict contains:
        rois: [N, (y1, x1, y2, x2)] detection bounding boxes
        class_ids: [N] int class IDs
        scores: [N] float probability scores for the class IDs
        masks: [H, W, N] instance binary masks
        """
        assert self.mode == "inference", "Create model in inference mode."
        assert len(
            images) == self.config.BATCH_SIZE, "len(images) must be equal to BATCH_SIZE"

        if verbose:
            log("Processing {} images".format(len(images)))
            for image in images:
                log("image", image)

        # Mold inputs to format expected by the neural network
        molded_images, image_metas, windows = self.mold_inputs(images)

        # Validate image sizes
        # All images in a batch MUST be of the same size
        image_shape = molded_images[0].shape
        for g in molded_images[1:]:
            assert g.shape == image_shape,\
                "After resizing, all images must have the same size. Check IMAGE_RESIZE_MODE and image sizes."

        # Anchors
        anchors = self.get_anchors(image_shape)
        # Duplicate across the batch dimension because Keras requires it
        # TODO: can this be optimized to avoid duplicating the anchors?
        anchors = np.broadcast_to(anchors, (self.config.BATCH_SIZE,) + anchors.shape)

        if verbose:
            log("molded_images", molded_images)
            log("image_metas", image_metas)
            log("anchors", anchors)
        # Run object detection
        detections, _, _, mrcnn_mask, _, _, _ =\
            self.keras_model.predict([molded_images, image_metas, anchors], verbose=0)
        # Process detections
        results = []
        for i, image in enumerate(images):
            final_rois, final_class_ids, final_scores, final_masks =\
                self.unmold_detections(detections[i], mrcnn_mask[i],
                                       image.shape, molded_images[i].shape,
                                       windows[i])
            results.append({
                "rois": final_rois,
                "class_ids": final_class_ids,
                "scores": final_scores,
                "masks": final_masks,
            })
        return results

    def detect_molded(self, molded_images, image_metas, verbose=0):
        """Runs the detection pipeline, but expect inputs that are
        molded already. Used mostly for debugging and inspecting
        the model.

        molded_images: List of images loaded using load_image_gt()
        image_metas: image meta data, also returned by load_image_gt()

        Returns a list of dicts, one dict per image. The dict contains:
        rois: [N, (y1, x1, y2, x2)] detection bounding boxes
        class_ids: [N] int class IDs
        scores: [N] float probability scores for the class IDs
        masks: [H, W, N] instance binary masks
        """
        assert self.mode == "inference", "Create model in inference mode."
        assert len(molded_images) == self.config.BATCH_SIZE,\
            "Number of images must be equal to BATCH_SIZE"

        if verbose:
            log("Processing {} images".format(len(molded_images)))
            for image in molded_images:
                log("image", image)

        # Validate image sizes
        # All images in a batch MUST be of the same size
        image_shape = molded_images[0].shape
        for g in molded_images[1:]:
            assert g.shape == image_shape, "Images must have the same size"

        # Anchors
        anchors = self.get_anchors(image_shape)
        # Duplicate across the batch dimension because Keras requires it
        # TODO: can this be optimized to avoid duplicating the anchors?
        anchors = np.broadcast_to(anchors, (self.config.BATCH_SIZE,) + anchors.shape)

        if verbose:
            log("molded_images", molded_images)
            log("image_metas", image_metas)
            log("anchors", anchors)
        # Run object detection
        detections, _, _, mrcnn_mask, _, _, _ =\
            self.keras_model.predict([molded_images, image_metas, anchors], verbose=0)
        # Process detections
        results = []
        for i, image in enumerate(molded_images):
            window = [0, 0, image.shape[0], image.shape[1]]
            final_rois, final_class_ids, final_scores, final_masks =\
                self.unmold_detections(detections[i], mrcnn_mask[i],
                                       image.shape, molded_images[i].shape,
                                       window)
            results.append({
                "rois": final_rois,
                "class_ids": final_class_ids,
                "scores": final_scores,
                "masks": final_masks,
            })
        return results

    def get_anchors(self, image_shape):
        """Returns anchor pyramid for the given image size."""
        backbone_shapes = compute_backbone_shapes(self.config, image_shape)
        # Cache anchors and reuse if image shape is the same
        if not hasattr(self, "_anchor_cache"):
            self._anchor_cache = {}
        if not tuple(image_shape) in self._anchor_cache:
            # Generate Anchors
            a = utils.generate_pyramid_anchors(
                self.config.RPN_ANCHOR_SCALES,
                self.config.RPN_ANCHOR_RATIOS,
                backbone_shapes,
                self.config.BACKBONE_STRIDES,
                self.config.RPN_ANCHOR_STRIDE)
            # Keep a copy of the latest anchors in pixel coordinates because
            # it's used in inspect_model notebooks.
            # TODO: Remove this after the notebook are refactored to not use it
            self.anchors = a
            # Normalize coordinates
            self._anchor_cache[tuple(image_shape)] = utils.norm_boxes(a, image_shape[:2])
        return self._anchor_cache[tuple(image_shape)]

    def ancestor(self, tensor, name, checked=None):
        """Finds the ancestor of a TF tensor in the computation graph.
        tensor: TensorFlow symbolic tensor.
        name: Name of ancestor tensor to find
        checked: For internal use. A list of tensors that were already
                 searched to avoid loops in traversing the graph.
        """
        checked = checked if checked is not None else []
        # Put a limit on how deep we go to avoid very long loops
        if len(checked) > 500:
            return None
        # Convert name to a regex and allow matching a number prefix
        # because Keras adds them automatically
        if isinstance(name, str):
            name = re.compile(name.replace("/", r"(\_\d+)*/"))

        parents = tensor.op.inputs
        for p in parents:
            if p in checked:
                continue
            if bool(re.fullmatch(name, p.name)):
                return p
            checked.append(p)
            a = self.ancestor(p, name, checked)
            if a is not None:
                return a
        return None

    def find_trainable_layer(self, layer):
        """If a layer is encapsulated by another layer, this function
        digs through the encapsulation and returns the layer that holds
        the weights.
        """
        if layer.__class__.__name__ == 'TimeDistributed':
            return self.find_trainable_layer(layer.layer)
        return layer

    def get_trainable_layers(self):
        """Returns a list of layers that have weights."""
        layers = []
        # Loop through all layers
        for l in self.keras_model.layers:
            # If layer is a wrapper, find inner trainable layer
            l = self.find_trainable_layer(l)
            # Include layer if it has weights
            if l.get_weights():
                layers.append(l)
        return layers

    def run_graph(self, images, outputs, image_metas=None):
        """Runs a sub-set of the computation graph that computes the given
        outputs.

        image_metas: If provided, the images are assumed to be already
            molded (i.e. resized, padded, and normalized)

        outputs: List of tuples (name, tensor) to compute. The tensors are
            symbolic TensorFlow tensors and the names are for easy tracking.

        Returns an ordered dict of results. Keys are the names received in the
        input and values are Numpy arrays.
        """
        model = self.keras_model

        # Organize desired outputs into an ordered dict
        outputs = OrderedDict(outputs)
        for o in outputs.values():
            assert o is not None

        # Build a Keras function to run parts of the computation graph
        inputs = model.inputs
        # if model.uses_learning_phase and not isinstance(K.learning_phase(), int):
        #     inputs += [K.learning_phase()]
        kf = K.function(model.inputs, list(outputs.values()))

        # Prepare inputs
        if image_metas is None:
            molded_images, image_metas, _ = self.mold_inputs(images)
        else:
            molded_images = images
        image_shape = molded_images[0].shape
        # Anchors
        anchors = self.get_anchors(image_shape)
        # Duplicate across the batch dimension because Keras requires it
        # TODO: can this be optimized to avoid duplicating the anchors?
        anchors = np.broadcast_to(anchors, (self.config.BATCH_SIZE,) + anchors.shape)
        model_in = [molded_images, image_metas, anchors]

        # Run inference
        # if model.uses_learning_phase and not isinstance(K.learning_phase(), int):
        #     model_in.append(0.)
        outputs_np = kf(model_in)

        # Pack the generated Numpy arrays into a a dict and log the results.
        outputs_np = OrderedDict([(k, v)
                                  for k, v in zip(outputs.keys(), outputs_np)])
        for k, v in outputs_np.items():
            log(k, v)
        return outputs_np



┌─────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                    MASK R-CNN COMPLETE ARCHITECTURE                                                 │
│                    (With Function Mappings)                                                         │
├─────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                                     │
│  INPUT                                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  Image [batch, H, W, 3]                                                                        ││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 1: RESNET BACKBONE (resnet_graph())                                                      ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  Conv1 + MaxPool                                                                           │││
│  │  │  conv_block() + identity_block() × N (Stage 2-5)                                          │││
│  │  │                                                                                             │││
│  │  │  Output: C2 [256x256x256], C3 [128x128x512], C4 [64x64x1024], C5 [32x32x2048]             │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────┘││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 2: FPN CONSTRUCTION (In build() method)                                                  ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  C5 → Conv1x1 → P5 [32x32x256]                                                             │││
│  │  │  C4 → Conv1x1 + Upsample(P5) → P4 [64x64x256]                                              │││
│  │  │  C3 → Conv1x1 + Upsample(P4) → P3 [128x128x256]                                            │││
│  │  │  C2 → Conv1x1 + Upsample(P3) → P2 [256x256x256]                                            │││
│  │  │  P5 → MaxPool → P6 [16x16x256]                                                             │││
│  │  │                                                                                             │││
│  │  │  rpn_feature_maps = [P2, P3, P4, P5, P6]                                                   │││
│  │  │  mrcnn_feature_maps = [P2, P3, P4, P5]                                                     │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────┘││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 3: RPN (rpn_graph() + build_rpn_model())                                                ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  For each P2-P6:                                                                           │││
│  │  │  ┌─────────────────────────────────────────────────────────────────────────────────────────┐│││
│  │  │  │  Shared Conv3x3(512)                                                                    ││││
│  │  │  │  ├── Classification Head: Conv1x1(2A) → rpn_class_logits [batch, HWA, 2]              ││││
│  │  │  │  └── BBox Head: Conv1x1(4A) → rpn_bbox [batch, HWA, 4]                                ││││
│  │  │  └─────────────────────────────────────────────────────────────────────────────────────────┘│││
│  │  │                                                                                             │││
│  │  │  Concatenate across all levels:                                                            │││
│  │  │  rpn_class_logits [batch, total_anchors, 2]                                               │││
│  │  │  rpn_class [batch, total_anchors, 2]                                                       │││
│  │  │  rpn_bbox [batch, total_anchors, 4]                                                        │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────┘││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    ▼                                                               │
│  ┌─────────────────────────────────────────────────────────────────────────────────────────────────┐│
│  │  STEP 4: PROPOSAL LAYER (ProposalLayer)                                                        ││
│  │  ┌─────────────────────────────────────────────────────────────────────────────────────────────┐││
│  │  │  Inputs: rpn_class, rpn_bbox, anchors                                                      │││
│  │  │  1. Select top 6000 anchors by FG score                                                    │││
│  │  │  2. apply_box_deltas_graph() → refine boxes                                                │││
│  │  │  3. clip_boxes_graph() → clip to image boundaries                                         │││
│  │  │  4. NMS → select top proposals                                                             │││
│  │  │                                                                                             │││
│  │  │  Output: rpn_rois [batch, 2000/1000, (y1,x1,y2,x2)]                                       │││
│  │  └─────────────────────────────────────────────────────────────────────────────────────────────┘││
│  └─────────────────────────────────────────────────────────────────────────────────────────────────┘│
│                                    │                                                               │
│                                    │                                                               │
│              ┌─────────────────────┴─────────────────────┐                                       │
│              │                                           │                                       │
│              ▼                                           ▼                                       │
│  ┌─────────────────────────────────┐  ┌──────────────────────────────────────────────────────────┐│
│  │  TRAINING PATH                   │  │  INFERENCE PATH                                          ││
│  │  ┌─────────────────────────────┐│  │  ┌────────────────────────────────────────────────────┐ ││
│  │  │  STEP 5A: DETECTION TARGET  ││  │  │  STEP 5B: HEADS (Direct from rpn_rois)             │ ││
│  │  │  LAYER                       ││  │  │                                                    │ ││
│  │  │  (DetectionTargetLayer)     ││  │  │  rpn_rois [1000] → Heads                          │ ││
│  │  │                              ││  │  └────────────────────────────────────────────────────┘ ││
│  │  │  Inputs: rpn_rois + GT      ││  │                         │                               ││
│  │  │  1. Compute IoU             ││  │                         ▼                               ││
│  │  │  2. Classify proposals      ││  │  ┌────────────────────────────────────────────────────┐ ││
│  │  │  3. Subsample to 200        ││  │  │  STEP 9B: DETECTION LAYER                          │ ││
│  │  │  4. Generate targets        ││  │  │  (DetectionLayer)                                  │ ││
│  │  │                              ││  │  │                                                    │ ││
│  │  │  Outputs:                    ││  │  │  Inputs: rpn_rois + mrcnn_class + mrcnn_bbox     │ ││
│  │  │  rois [200],                 ││  │  │  1. Get best class (argmax)                      │ ││
│  │  │  target_class_ids [200],    ││  │  │  2. Get class-specific deltas                    │ ││
│  │  │  target_bbox [200,4],       ││  │  │  3. Refine boxes                                 │ ││
│  │  │  target_mask [200,28,28]   ││  │  │  4. Filter background + low confidence            │ ││
│  │  └─────────────────────────────┘│  │  │  5. Class-wise NMS                                │ ││
│  │              │                   │  │  │  6. Pad to max_instances                         │ ││
│  │              ▼                   │  │  │                                                    │ ││
│  │  ┌─────────────────────────────┐│  │  │  Output: detections [max, 6]                     │ ││
│  │  │  STEP 6: ROI ALIGN          ││  │  └────────────────────────────────────────────────────┘ ││
│  │  │  (PyramidROIAlign)          ││  │                         │                               ││
│  │  │                              ││  │                         ▼                               ││
│  │  │  rois + P2-P5 → ROI features││  │  ┌────────────────────────────────────────────────────┐ ││
│  │  │  - 7x7 for classifier       ││  │  │  STEP 10B: MASK HEAD                               │ ││
│  │  │  - 14x14 for mask           ││  │  │  (build_fpn_mask_graph())                         │ ││
│  │  └─────────────────────────────┘│  │  │                                                    │ ││
│  │              │                   │  │  │  detection_boxes + P2-P5 → masks                 │ ││
│  │              ▼                   │  │  │  Output: mrcnn_mask [max, 28, 28, C]             │ ││
│  │  ┌─────────────────────────────┐│  │  └────────────────────────────────────────────────────┘ ││
│  │  │  STEP 7: CLASSIFIER HEAD    ││  │                         │                               ││
│  │  │  (fpn_classifier_graph())   ││  │                         ▼                               ││
│  │  │                              ││  │  ┌────────────────────────────────────────────────────┐ ││
│  │  │  7x7 ROI features →          ││  │  │  FINAL OUTPUT (Inference)                        │ ││
│  │  │  2x FC Layers →              ││  │  │  detections + mrcnn_mask                         │ ││
│  │  │  ├── Classifier Head         ││  │  └────────────────────────────────────────────────────┘ ││
│  │  │  └── BBox Head               ││  │                                                       ││
│  │  │                              ││  │                                                       ││
│  │  │  Outputs:                    ││  │                                                       ││
│  │  │  mrcnn_class_logits [C]     ││  │                                                       ││
│  │  │  mrcnn_class [C]            ││  │                                                       ││
│  │  │  mrcnn_bbox [C,4]           ││  │                                                       ││
│  │  └─────────────────────────────┘│  │                                                       ││
│  │              │                   │  │                                                       ││
│  │              ▼                   │  │                                                       ││
│  │  ┌─────────────────────────────┐│  │                                                       ││
│  │  │  STEP 8: MASK HEAD          ││  │                                                       ││
│  │  │  (build_fpn_mask_graph())   ││  │                                                       ││
│  │  │                              ││  │                                                       ││
│  │  │  14x14 ROI features →        ││  │                                                       ││
│  │  │  4x Conv3x3 →               ││  │                                                       ││
│  │  │  Deconv(28x28) →            ││  │                                                       ││
│  │  │  Conv1x1 + Sigmoid           ││  │                                                       ││
│  │  │                              ││  │                                                       ││
│  │  │  Output: mrcnn_mask          ││  │                                                       ││
│  │  │  [200, 28, 28, C]           ││  │                                                       ││
│  │  └─────────────────────────────┘│  │                                                       ││
│  │              │                   │  │                                                       ││
│  │              ▼                   │  │                                                       ││
│  │  ┌─────────────────────────────┐│  │                                                       ││
│  │  │  STEP 9A: LOSS COMPUTATION ││  │                                                       ││
│  │  │  (Loss Functions)           ││  │                                                       ││
│  │  │                              ││  │                                                       ││
│  │  │  rpn_class_loss             ││  │                                                       ││
│  │  │  rpn_bbox_loss              ││  │                                                       ││
│  │  │  mrcnn_class_loss           ││  │                                                       ││
│  │  │  mrcnn_bbox_loss            ││  │                                                       ││
│  │  │  mrcnn_mask_loss            ││  │                                                       ││
│  │  └─────────────────────────────┘│  │                                                       ││
│  └─────────────────────────────────┘  └──────────────────────────────────────────────────────────┘│
│                                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────────────────────┘

MaskRCNN.__init__()
    │
    └──► build()
            │
            ├──► resnet_graph()
            │       ├──► conv_block()
            │       └──► identity_block()
            │
            ├──► FPN Layers (Conv2D, Add, Upsample)
            │
            ├──► build_rpn_model()
            │       └──► rpn_graph()
            │               ├──► KL.Conv2D (Shared)
            │               ├──► KL.Conv2D (Classification)
            │               └──► KL.Conv2D (BBox)
            │
            ├──► ProposalLayer()
            │       ├──► apply_box_deltas_graph()
            │       └──► clip_boxes_graph()
            │
            ├──► DetectionTargetLayer() [TRAINING]
            │       └──► detection_targets_graph()
            │               ├──► overlaps_graph()
            │               ├──► trim_zeros_graph()
            │               └──► utils.box_refinement_graph()
            │
            ├──► fpn_classifier_graph()
            │       ├──► PyramidROIAlign(7x7)
            │       ├──► KL.TimeDistributed(Conv2D)
            │       ├──► KL.Lambda(squeeze)
            │       ├──► KL.Dense(Classifier)
            │       └──► KL.Dense(BBox)
            │
            └──► build_fpn_mask_graph()
                    ├──► PyramidROIAlign(14x14)
                    ├──► KL.TimeDistributed(Conv2D) × 4
                    ├──► KL.Conv2DTranspose
                    └──► KL.Conv2D(num_classes, sigmoid)

MaskRCNN.compile()
    │
    ├──► keras.optimizers.legacy.SGD()
    ├──► self.keras_model.add_loss() × 5
    └──► self.keras_model.compile()

MaskRCNN.train()
    │
    ├──► DataGenerator()
    │       ├──► load_image_gt()
    │       └──► build_rpn_targets()
    │
    ├──► set_trainable()
    ├──► compile()
    └──► self.keras_model.fit()

MaskRCNN.detect()
    │
    ├──► mold_inputs()
    │       ├──► utils.resize_image()
    │       ├──► mold_image()
    │       └──► compose_image_meta()
    │
    ├──► get_anchors()
    │       └──► utils.generate_pyramid_anchors()
    │
    ├──► self.keras_model.predict()
    │       └──► DetectionLayer()
    │               └──► refine_detections_graph()
    │                       ├──► apply_box_deltas_graph()
    │                       └──► clip_boxes_graph()
    │
    └──► unmold_detections()
            ├──► utils.norm_boxes()
            ├──► utils.denorm_boxes()
            └──► utils.unmold_mask()